Load csvs. 1 csv- one conversation. each row, one utterance. some consecutive speaker rows.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path.cwd()
WIRED_DIR = PROJECT_ROOT / "data" / "wired"

csv_paths = sorted(WIRED_DIR.glob("wired_*/*.csv"))

print(f"Found {len(csv_paths)} CSV files")
csv_paths[:10]

Found 105 CSV files


[PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_12.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_13.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_14.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_15.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_16.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_12.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_13.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_14.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_15.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_

In [2]:
def parse_wired_path(path):
    """
    Example:
    data/wired/wired_astro/wired_astro_12.csv
    """
    folder_name = path.parent.name
    file_stem = path.stem

    video_id = re.sub(r"^wired_", "", folder_name)

    match = re.search(r"_(\d+)$", file_stem)
    file_number = int(match.group(1)) if match else None

    return {
        "dataset": "wired",
        "video_id": video_id,
        "file_number": file_number,
        "conversation_id": file_stem,
        "source_file": str(path.relative_to(PROJECT_ROOT))
    }

In [3]:
raw_frames = []

for path in csv_paths:
    df = pd.read_csv(path)
    metadata = parse_wired_path(path)

    # Preserve original within-file row order
    df["raw_row_id"] = np.arange(len(df))

    for column, value in metadata.items():
        df[column] = value

    raw_frames.append(df)

wired_raw = pd.concat(
    raw_frames,
    ignore_index=True,
    sort=False
)

wired_raw.shape

(6384, 10)

In [4]:
for path in csv_paths:
    try:
        df = pd.read_csv(path)
        print("OK:", path.name)
    except pd.errors.ParserError as error:
        print("ERROR:", path)
        print(error)

OK: wired_astro_12.csv
OK: wired_astro_13.csv
OK: wired_astro_14.csv
OK: wired_astro_15.csv
OK: wired_astro_16.csv
OK: wired_blockchain_12.csv
OK: wired_blockchain_13.csv
OK: wired_blockchain_14.csv
OK: wired_blockchain_15.csv
OK: wired_blockchain_16.csv
OK: wired_crispr_12.csv
OK: wired_crispr_13.csv
OK: wired_crispr_14.csv
OK: wired_crispr_15.csv
OK: wired_crispr_16.csv
OK: wired_dimension_12.csv
OK: wired_dimension_13.csv
OK: wired_dimension_14.csv
OK: wired_dimension_15.csv
OK: wired_dimension_16.csv
OK: wired_fractal_12.csv
OK: wired_fractal_13.csv
OK: wired_fractal_14.csv
OK: wired_fractal_15.csv
OK: wired_fractal_16.csv
OK: wired_gravity_12.csv
OK: wired_gravity_13.csv
OK: wired_gravity_14.csv
OK: wired_gravity_15.csv
OK: wired_gravity_16.csv
OK: wired_hacking_12.csv
OK: wired_hacking_13.csv
OK: wired_hacking_14.csv
OK: wired_hacking_15.csv
OK: wired_hacking_16.csv
OK: wired_infinity_12.csv
OK: wired_infinity_13.csv
OK: wired_infinity_14.csv
OK: wired_infinity_15.csv
OK: wired_i

In [ ]:
schema_rows = []

for path in csv_paths:
    df = pd.read_csv(path)

    schema_rows.append({
        "source_file": path.name,
        "folder": path.parent.name,
        "n_rows": len(df),
        "columns": tuple(df.columns)
    })

schema_df = pd.DataFrame(schema_rows)

schema_df.head()

In [ ]:
def parse_wired_path(path):
    """
    Example:
    data/wired/wired_astro/wired_astro_12.csv
    """
    folder_name = path.parent.name
    file_stem = path.stem

    video_id = re.sub(r"^wired_", "", folder_name)

    match = re.search(r"_(\d+)$", file_stem)
    file_number = int(match.group(1)) if match else None

    return {
        "dataset": "wired",
        "video_id": video_id,
        "file_number": file_number,
        "conversation_id": file_stem,
        "source_file": str(path.relative_to(PROJECT_ROOT))
    }

In [ ]:
raw_frames = []

for path in csv_paths:
    df = pd.read_csv(path)
    metadata = parse_wired_path(path)

    # Preserve original within-file row order
    df["raw_row_id"] = np.arange(len(df))

    for column, value in metadata.items():
        df[column] = value

    raw_frames.append(df)

wired_raw = pd.concat(
    raw_frames,
    ignore_index=True,
    sort=False
)

wired_raw.shape

table with all utterances, all conversations

In [ ]:
wired_raw.head()

In [ ]:
#rename columns, minimally clean text
wired_raw = wired_raw.rename(columns={
    "Speaker": "speaker_raw",
    "Utterance": "text"
})
wired_raw["text"] = (
    wired_raw["text"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

wired_raw = wired_raw[
    wired_raw["text"].notna() &
    wired_raw["text"].ne("")
].copy()

In [ ]:
level_map = {
    12: (0, "child"),
    13: (1, "teenager"),
    14: (2, "undergraduate"),
    15: (3, "graduate"),
    16: (4, "expert")
}

In [ ]:
wired_raw["level"] = wired_raw["file_number"].map(
    lambda x: level_map[x][0]
)

wired_raw["level_label"] = wired_raw["file_number"].map(
    lambda x: level_map[x][1]
)

In [ ]:
#order levels
level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert"
]

wired_raw["level_label"] = pd.Categorical(
    wired_raw["level_label"],
    categories=level_order,
    ordered=True
)

In [ ]:
(
    wired_raw[
        ["video_id", "conversation_id", "speaker_raw"]
    ]
    .drop_duplicates()
    .sort_values(["video_id", "conversation_id", "speaker_raw"])
)

In [ ]:
'''
speaker_mapping = {
    #astro
     "Speaker 1 (Janna Levin)": "A",
     "Speaker 2 (child)": "B",
     "Speaker 3 (teen)": "B",
    "Speaker 4 (college student)": "B",
    "Speaker 5 (grad student)": "B",
    "Speaker 6 (expert)": "B",
    #blockchain
    "Speaker 1 (Bettina Warburg)": "A",
    "Speaker 2 (Child)": "B",
    "Speaker 3 (Teen - Ian)": "B",
    "Speaker 4 (College Student)": "B",
    "Speaker 5 (Grad Student)": "B",
    "Speaker 6 (Expert)": "B",
    #crispr
    "Speaker 1 (Neville Sanjana)": "A",
    "Speaker 2 (Tegan - Child)": "B",
    "Speaker 3 (Bella - Teenager)": "B",
    "Speaker 4 (Christopher - Student)": "B",
    "Speaker 5 (Lauren Schiff)": "B",
    "Speaker 6 (Matthew Canver)": "B",
    #dimension
    "Speaker 1 (Sean Carroll PhD)": "A",
    #fractal
    "Speaker 1 (Keenan Crane Phd)": "A",
    #gravity
    "Speaker 1 (Janna Levin)": "A",
    "Speaker 2 (Bonét Sofía Kanayet)": "B",
    "Speaker 3 (Maria Teresa Furtado)": "B",
    "Speaker 4 (Lisa Chan)": "B",
    "Speaker 5 (Will Gyory)": "B",
    "Speaker 6 (Matthew Kleban)": "B",
    #hacking
    "Speaker 1 (Samy Kamkar)": "A",
    #infinity
    "Speaker 1 (Emily Riehl)": "A",
    #internet
    "Speaker 1 (Jim Kurose)": "A",
    #laser
    "Speaker 1 (Donna Strickland)": "A",
    "Speaker 2 (Harmoni - Child)": "B",
    "Speaker 3 (Eli Kaplan - Teenager)": "B",
    "Speaker 4 (Caitlin - Student)": "B",
    "Speaker 5 (Aditya - Grad Student)": "B",
    "Speaker 6 (Mike Campbell)": "B",
    #machine
    "Speaker 1 (Hilary Mason)": "A",
    #memory
    "Speaker 1 (Daphna Shohamy)": "A",
    #moravec
    "Speaker 1 (Chelsea Finn)": "A",
    #nano
    "Speaker 1 (Dr. George S. Tulevski)": "A",
    #neuro
    "Speaker 1 (Dr. Bobby Kasthuri)": "A",
    "Speaker 2 (child - Daniel)": "B",
    "Speaker 3 (Jabez Griggs)": "B",
    "Speaker 4 (Elena Dowling)": "B",
    "Speaker 5 (Mala Ananth)": "B",
    "Speaker 6 (Russell Hanson)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",


}
'''
is_expert = wired_raw["speaker_raw"].str.contains(
    r"\bSpeaker\s*1\b",
    case=False,
    na=False
)

wired_raw["speaker"] = np.where(is_expert, "A", "B")
wired_raw["speaker_role"] = np.where(
    is_expert,
    "expert",
    "partner"
)




wired_raw["speaker_role"] = wired_raw["speaker"].map({
    "A": "expert",
    "B": "partner"
})

In [ ]:
display(
    wired_raw.loc[
        wired_raw["raw_row_id"] == 4458
    ].T
)

collapse consecutive rows from same speaker-> one row is one full conversational turn not an utterance.

In [ ]:
def collapse_consecutive_speaker_rows(group):
    group = group.sort_values("raw_row_id").copy()

    # A new turn starts whenever the speaker changes
    group["turn_block"] = (
        group["speaker"].ne(group["speaker"].shift())
    ).cumsum()

    collapsed = (
        group.groupby("turn_block", sort=False)
        .agg(
            dataset=("dataset", "first"),
            video_id=("video_id", "first"),
            conversation_id=("conversation_id", "first"),
            file_number=("file_number", "first"),
            level=("level", "first"),
            level_label=("level_label", "first"),
            speaker=("speaker", "first"),
            speaker_raw=("speaker_raw", "first"),
            speaker_role=("speaker_role", "first"),
            text=("text", " ".join),
            source_file=("source_file", "first"),
            raw_row_start=("raw_row_id", "min"),
            raw_row_end=("raw_row_id", "max")
        )
        .reset_index(drop=True)
    )

    collapsed["turn_id"] = np.arange(len(collapsed))
    collapsed["n_words"] = collapsed["text"].str.split().str.len()

    n_turns = len(collapsed)
    #NORMALIZE TIME
    collapsed["normalized_time"] = (
        collapsed["turn_id"] / max(n_turns - 1, 1)
    )

    return collapsed

In [ ]:
turns = (
    wired_raw
    .groupby("conversation_id", group_keys=False, observed=True)
    .apply(collapse_consecutive_speaker_rows)
    .reset_index(drop=True)
)

In [ ]:
turns

In [ ]:
turns = turns[
    [
        "dataset",
        "video_id",
        "conversation_id",
        "file_number",
        "level",
        "level_label",
        "turn_id",
        "normalized_time",
        "speaker",
        "speaker_raw",
        "speaker_role",
        "text",
        "n_words",
        "raw_row_start",
        "raw_row_end",
        "source_file"
    ]
].sort_values(
    ["video_id", "level", "turn_id"]
).reset_index(drop=True)

In [ ]:
#validation checks
# Every conversation should have exactly two standardized speakers
speaker_counts = (
    turns.groupby("conversation_id")["speaker"]
    .nunique()
)

assert speaker_counts.eq(2).all(), (
    "Some conversations do not have exactly two speakers:\n"
    f"{speaker_counts[speaker_counts.ne(2)]}"
)

In [ ]:
# Conversation-level overview
conversation_summary = (
    turns.groupby(
        [
            "video_id",
            "conversation_id",
            "level",
            "level_label"
        ],
        observed=True
    )
    .agg(
        n_turns=("turn_id", "size"),
        total_words=("n_words", "sum"),
        speakers=("speaker", "nunique")
    )
    .reset_index()
)

conversation_summary.head(10)

wraps up basic cleaning and organization

# embedding

segments table

each row of the

In [ ]:
turns["n_words"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

In [ ]:
(
    turns
    .nlargest(10, "n_words")
    [
        [
            "conversation_id",
            "turn_id",
            "speaker",
            "speaker_role",
            "n_words",
            "text",
        ]
    ]
)

some of the turns are super long so segment so max word count of each segment is 150 words

In [ ]:
import re

MAX_SEGMENT_WORDS = 150


def count_words(text):
    """Count whitespace-separated words."""
    return len(str(text).split())


def normalize_whitespace(text):
    """Remove repeated whitespace without changing the words."""
    return re.sub(r"\s+", " ", str(text)).strip()


def split_oversized_unit(text, max_words=MAX_SEGMENT_WORDS):
    """
    Split a single sentence or text unit that exceeds the maximum length.
    This is the fallback for exceptionally long sentences.
    """
    words = text.split()

    return [
        " ".join(words[start:start + max_words])
        for start in range(0, len(words), max_words)
    ]


def split_turn_text(text, max_words=MAX_SEGMENT_WORDS):
    """
    Split one turn into non-overlapping, sentence-preserving segments.
    """
    text = normalize_whitespace(text)

    if not text:
        return []

    # Most turns should pass through unchanged.
    if count_words(text) <= max_words:
        return [text]

    # Provisional sentence splitting based on terminal punctuation.
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    # Guarantee that no individual unit exceeds the maximum.
    units = []

    for sentence in sentences:
        if count_words(sentence) <= max_words:
            units.append(sentence)
        else:
            units.extend(
                split_oversized_unit(sentence, max_words=max_words)
            )

    # Greedily combine consecutive sentences without exceeding the maximum.
    segments = []
    current_units = []
    current_words = 0

    for unit in units:
        unit_words = count_words(unit)

        if current_units and current_words + unit_words > max_words:
            segments.append(" ".join(current_units))
            current_units = [unit]
            current_words = unit_words
        else:
            current_units.append(unit)
            current_words += unit_words

    if current_units:
        segments.append(" ".join(current_units))

    return segments

In [ ]:
example_text = turns.loc[turns["n_words"].idxmax(), "text"]
example_segments = split_turn_text(example_text)

print("Original words:", count_words(example_text))
print("Number of segments:", len(example_segments))
print("Segment lengths:", [count_words(x) for x in example_segments])

for i, segment in enumerate(example_segments, start=1):
    print(f"\n--- Segment {i} ---")
    print(segment)

construct segments table

In [ ]:
segment_records = []

# Preserve the existing order of the turns table.
ordered_turns = turns.reset_index(drop=True).copy()
ordered_turns["_turn_order"] = range(len(ordered_turns))

for _, turn in ordered_turns.iterrows():
    turn_segments = split_turn_text(
        turn["text"],
        max_words=MAX_SEGMENT_WORDS,
    )

    n_segments = len(turn_segments)

    for segment_number, segment_text in enumerate(
        turn_segments,
        start=1,
    ):
        record = {
            # Conversation metadata
            "dataset": turn["dataset"],
            "video_id": turn["video_id"],
            "conversation_id": turn["conversation_id"],
            "file_number": turn["file_number"],
            "level": turn["level"],
            "level_label": turn["level_label"],

            # Turn and sequence metadata
            "turn_id": turn["turn_id"],
            "segment_in_turn": segment_number,
            "n_segments_in_turn": n_segments,
            "_turn_order": turn["_turn_order"],
            "normalized_time": turn["normalized_time"],

            # Speaker metadata
            "speaker": turn["speaker"],
            "speaker_raw": turn["speaker_raw"],
            "speaker_role": turn["speaker_role"],

            # Segment content
            "text": segment_text,
            "n_words": count_words(segment_text),

            # Original-turn information
            "turn_n_words": turn["n_words"],
            "is_split_turn": n_segments > 1,

            # Provenance
            "raw_row_start": turn["raw_row_start"],
            "raw_row_end": turn["raw_row_end"],
            "source_file": turn["source_file"],
        }

        segment_records.append(record)

segments = pd.DataFrame(segment_records)

In [ ]:
segments

In [ ]:
segments["segment_id"] = (
    segments["dataset"].astype(str)
    + "::"
    + segments["conversation_id"].astype(str)
    + "::turn_"
    + segments["turn_id"].astype(str)
    + "::segment_"
    + segments["segment_in_turn"].astype(str).str.zfill(2)
)

In [ ]:
segment_columns = [
    "segment_id",
    "dataset",
    "video_id",
    "conversation_id",
    "file_number",
    "level",
    "level_label",
    "turn_id",
    "segment_in_turn",
    "n_segments_in_turn",
    "normalized_time",
    "speaker",
    "speaker_raw",
    "speaker_role",
    "text",
    "n_words",
    "turn_n_words",
    "is_split_turn",
    "raw_row_start",
    "raw_row_end",
    "source_file",
    "_turn_order",
]

segments = segments[segment_columns]

In [ ]:
segment_number_check = (
    segments
    .groupby("_turn_order")["segment_in_turn"]
    .apply(lambda x: list(x) == list(range(1, len(x) + 1)))
)

assert segment_number_check.all()

In [ ]:
segments["n_words"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

## embedding

make one normalized 382 dim vector per segments row

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import sentence_transformers
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model:", MODEL_NAME)
print("Embedding dimensions:", model.get_sentence_embedding_dimension())
print("Maximum model tokens:", model.max_seq_length)
print("SentenceTransformers version:", sentence_transformers.__version__)

In [ ]:
segments = segments.reset_index(drop=True).copy()

assert segments["segment_id"].is_unique
assert segments["text"].notna().all()
assert segments["text"].str.strip().ne("").all()

segments["embedding_idx"] = np.arange(len(segments))

segment_texts = segments["text"].astype(str).tolist()

print("Segments to embed:", len(segment_texts))

segments[
    ["embedding_idx", "segment_id", "speaker", "n_words", "text"]
].head()

In [ ]:
segments["n_model_tokens"] = [
    len(
        model.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )
    for text in segment_texts
]

segments["would_truncate"] = (
    segments["n_model_tokens"] > model.max_seq_length
)

print(
    "Maximum token count:",
    segments["n_model_tokens"].max()
)

print(
    "Segments exceeding model limit:",
    segments["would_truncate"].sum()
)

In [ ]:
#generate embeddings
E_segments = model.encode(
    segment_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

E_segments = E_segments.astype(np.float32)

In [ ]:
#validate embedding matrix
expected_dimension = model.get_sentence_embedding_dimension()

assert E_segments.shape == (
    len(segments),
    expected_dimension,
)

assert np.isfinite(E_segments).all()

embedding_norms = np.linalg.norm(E_segments, axis=1)

print("Embedding matrix shape:", E_segments.shape)
print("Minimum norm:", embedding_norms.min())
print("Maximum norm:", embedding_norms.max())
print("Mean norm:", embedding_norms.mean())

assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-5,
)

In [ ]:
#save output
OUTPUT_DIR = Path("horizons2_embeddings")
OUTPUT_DIR.mkdir(exist_ok=True)

np.save(
    OUTPUT_DIR / "E_segments_all_MiniLM_L6_v2.npy",
    E_segments,
    allow_pickle=False,
)

segments.to_csv(
    OUTPUT_DIR / "segments_with_embedding_index.csv",
    index=False,
)

embedding_config = {
    "model_name": MODEL_NAME,
    "sentence_transformers_version": sentence_transformers.__version__,
    "embedding_dimension": int(expected_dimension),
    "normalized": True,
    "number_of_embeddings": int(len(E_segments)),
    "maximum_model_tokens": int(model.max_seq_length),
    "matrix_file": "E_segments_all_MiniLM_L6_v2.npy",
    "index_column": "embedding_idx",
    "identifier_column": "segment_id",
}

with open(
    OUTPUT_DIR / "embedding_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        embedding_config,
        file,
        indent=2,
    )

## utterances table -> embed utterances

In [ ]:
import re
import numpy as np
import pandas as pd


def count_words(text):
    return len(
        re.findall(
            r"\b[\w'-]+\b",
            str(text),
        )
    )


def create_utterances_table(wired_raw):
    """
    Create one canonical row per original transcript utterance.

    The function does not concatenate adjacent utterances.
    """

    utterances = (
        wired_raw
        .copy()
        .reset_index(drop=False)
        .rename(columns={"index": "_original_dataframe_row"})
    )

    # Accommodate either the original CSV column names or the
    # standardized names already present in wired_raw.
    rename_map = {}

    aliases = {
        "Sequence": "sequence",
        "Utterance": "text",
        "Speaker": "speaker_raw",
        "Notes": "notes",
    }

    for old_name, new_name in aliases.items():
        if (
            old_name in utterances.columns
            and new_name not in utterances.columns
        ):
            rename_map[old_name] = new_name

    utterances = utterances.rename(columns=rename_map)

    # If canonical speaker has not already been created, use
    # the original speaker label.
    if "speaker" not in utterances.columns:
        utterances["speaker"] = utterances["speaker_raw"]

    # Stable source order.
    if "sequence" not in utterances.columns:
        utterances["sequence"] = (
            utterances
            .groupby(
                ["dataset", "conversation_id"],
                sort=False,
            )
            .cumcount()
            .add(1)
        )

    utterances["_sequence_numeric"] = pd.to_numeric(
        utterances["sequence"],
        errors="coerce",
    )

    utterances["_sequence_numeric"] = (
        utterances["_sequence_numeric"]
        .fillna(utterances["_original_dataframe_row"])
    )

    conversation_keys = [
        "dataset",
        "conversation_id",
    ]

    sort_columns = [
        *conversation_keys,
        "_sequence_numeric",
        "_original_dataframe_row",
    ]

    utterances = (
        utterances
        .sort_values(
            sort_columns,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # Clean text without altering its linguistic contents.
    utterances["text"] = (
        utterances["text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    if utterances["text"].eq("").any():
        empty_rows = utterances.loc[
            utterances["text"].eq(""),
            [
                *conversation_keys,
                "sequence",
            ],
        ]

        raise ValueError(
            "Empty utterance text found:\n"
            + empty_rows.to_string(index=False)
        )

    # Reconstruct turn membership without concatenating anything.
    # A new turn begins whenever the speaker changes.
    speaker_changed = (
        utterances
        .groupby(
            conversation_keys,
            sort=False,
        )["speaker"]
        .transform(
            lambda speakers:
                speakers.ne(speakers.shift())
        )
    )

    utterances["turn_id"] = (
        speaker_changed
        .astype(int)
        .groupby(
            [
                utterances[column]
                for column in conversation_keys
            ]
        )
        .cumsum()
    )

    # Position of the utterance inside its speaker turn.
    utterances["utterance_in_turn"] = (
        utterances
        .groupby(
            [
                *conversation_keys,
                "turn_id",
            ],
            sort=False,
        )
        .cumcount()
        .add(1)
    )

    # Position within the whole conversation.
    utterances["utterance_number"] = (
        utterances
        .groupby(
            conversation_keys,
            sort=False,
        )
        .cumcount()
        .add(1)
    )

    utterances["utterance_id"] = (
        utterances["dataset"].astype(str)
        + "::"
        + utterances["conversation_id"].astype(str)
        + "::utterance_"
        + utterances["utterance_number"]
            .astype(str)
            .str.zfill(3)
    )

    utterances["n_words"] = (
        utterances["text"]
        .map(count_words)
        .astype(int)
    )

    # Keep all utterances in the canonical table. This flag only
    # determines the primary geometry subset later.
    utterances["include_geometry_primary"] = (
        utterances["n_words"] >= 5
    )

    # This index will correspond exactly to rows of E_utterances.
    utterances["embedding_idx"] = np.arange(
        len(utterances),
        dtype=int,
    )

    preferred_columns = [
        "dataset",
        "video_id",
        "conversation_id",
        "file_number",
        "level",
        "level_label",
        "utterance_id",
        "utterance_number",
        "turn_id",
        "utterance_in_turn",
        "sequence",
        "normalized_time",
        "speaker",
        "speaker_raw",
        "speaker_role",
        "text",
        "n_words",
        "include_geometry_primary",
        "embedding_idx",
        "notes",
        "source_file",
        "_original_dataframe_row",
    ]

    # Retain only columns that actually exist in this dataset.
    output_columns = [
        column
        for column in preferred_columns
        if column in utterances.columns
    ]

    return utterances[output_columns].copy()


utterances = create_utterances_table(wired_raw)

utterances.head()

In [ ]:
from sentence_transformers import SentenceTransformer


EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())
print("Maximum sequence length:", embedding_model.max_seq_length)

In [ ]:
# Prepare and validate utterance texts
utterance_texts = utterances["text"].astype(str).tolist()

utterances["n_model_tokens"] = [
    len(
        embedding_model.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )
    for text in utterance_texts
]

utterances["exceeds_model_limit"] = (
    utterances["n_model_tokens"]
    > embedding_model.max_seq_length
)

if utterances["exceeds_model_limit"].any():
    raise ValueError(
        "At least one utterance exceeds the model's token limit."
    )

# Generate one normalized embedding per utterance
E_utterances = embedding_model.encode(
    utterance_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

E_utterances = np.asarray(E_utterances, dtype=np.float32)

assert E_utterances.shape[0] == len(utterances)
assert np.isfinite(E_utterances).all()
assert np.allclose(
    np.linalg.norm(E_utterances, axis=1),
    1.0,
    atol=1e-5,
)

print("E_utterances shape:", E_utterances.shape)

In [ ]:
geometry_utterances = utterances.copy()

print("All utterances:", len(utterances))
print(
    "Primary geometry utterances:",
    len(geometry_utterances),
)
print(
    "Excluded short utterances:",
    len(utterances) - len(geometry_utterances),
)

## analysis - look at distribution/sturcture of information

## conversation level geometry

In [ ]:
import numpy as np
import pandas as pd


def _sample_sd(values):
    """Sample standard deviation, or NaN if only one value exists."""
    values = np.asarray(values)

    if values.size < 2:
        return np.nan

    return float(np.std(values, ddof=1))


def calculate_conversation_geometry(embeddings, n_words=None, n_turns=None, k=5):
    """
    Calculate the complete geometric metric set for one conversation.

    Parameters
    ----------
    embeddings : array-like, shape (n_utterances, embedding_dimension)
        One embedding per utterance.

    n_words : int, optional
        Total number of words in the conversation.

    n_turns : int, optional
        Total number of conversational turns.

    k : int, default=5
        Neighbor rank used for the k-nearest-neighbor statistic.

    Returns
    -------
    scalar_metrics : dict
        Conversation-level scalar metrics suitable for a DataFrame.

    spectrum : dict
        Eigenvalue arrays retained separately for spectral plots.
    """

    X = np.asarray(embeddings, dtype=np.float64)

    if X.ndim != 2:
        raise ValueError(
            "embeddings must have shape "
            "(n_utterances, embedding_dimension)"
        )

    if not np.isfinite(X).all():
        raise ValueError("Embeddings contain NaN or infinite values.")

    n, d = X.shape

    if n < 2:
        raise ValueError(
            "At least two utterances are required."
        )

    if not isinstance(k, (int, np.integer)) or k < 1:
        raise ValueError("k must be a positive integer.")

    # ------------------------------------------------------------
    # 1. Normalize embeddings
    # ------------------------------------------------------------

    norms = np.linalg.norm(X, axis=1, keepdims=True)

    if np.any(norms == 0):
        raise ValueError("At least one embedding has zero norm.")

    X_unit = X / norms

    # ------------------------------------------------------------
    # 2. Pairwise cosine similarity and distance
    # ------------------------------------------------------------

    cosine_similarity_matrix = np.clip(
        X_unit @ X_unit.T,
        -1.0,
        1.0,
    )

    cosine_distance_matrix = 1.0 - cosine_similarity_matrix

    # Set the diagonal to exactly zero for neighbor calculations.
    np.fill_diagonal(cosine_distance_matrix, 0.0)

    upper_triangle = np.triu_indices(n, k=1)

    pairwise_similarities = cosine_similarity_matrix[
        upper_triangle
    ]

    pairwise_distances = cosine_distance_matrix[
        upper_triangle
    ]

    n_pairs = len(pairwise_distances)

    sim_q10, sim_median, sim_q90 = np.quantile(
        pairwise_similarities,
        [0.10, 0.50, 0.90],
    )

    dist_q10, dist_median, dist_q90 = np.quantile(
        pairwise_distances,
        [0.10, 0.50, 0.90],
    )

    # ------------------------------------------------------------
    # 3. Centroid concentration
    # ------------------------------------------------------------

    centroid = X_unit.mean(axis=0)

    centroid_resultant_length = float(
        np.linalg.norm(centroid)
    )

    centroid_dispersion_cosine = float(
        1.0 - centroid_resultant_length
    )

    # ------------------------------------------------------------
    # 4. Radial structure
    # ------------------------------------------------------------

    centroid_tolerance = (
        100 * np.finfo(np.float64).eps
    )

    if centroid_resultant_length > centroid_tolerance:
        centroid_direction = (
            centroid / centroid_resultant_length
        )

        radial_distances = np.clip(
            1.0 - X_unit @ centroid_direction,
            0.0,
            2.0,
        )

        radial_distance_mean = float(
            np.mean(radial_distances)
        )

        radial_distance_median = float(
            np.median(radial_distances)
        )

        radial_distance_q90 = float(
            np.quantile(radial_distances, 0.90)
        )

    else:
        # A zero centroid has no defined direction.
        radial_distances = np.full(n, np.nan)
        radial_distance_mean = np.nan
        radial_distance_median = np.nan
        radial_distance_q90 = np.nan

    # ------------------------------------------------------------
    # 5. Local-neighborhood structure
    # ------------------------------------------------------------

    neighbor_distance_matrix = (
        cosine_distance_matrix.copy()
    )

    # Prevent each utterance from selecting itself.
    np.fill_diagonal(
        neighbor_distance_matrix,
        np.inf,
    )

    nearest_neighbor_distances = np.min(
        neighbor_distance_matrix,
        axis=1,
    )

    k_used = min(k, n - 1)

    kth_neighbor_distances = np.partition(
        neighbor_distance_matrix,
        kth=k_used - 1,
        axis=1,
    )[:, k_used - 1]

    # ------------------------------------------------------------
    # 6. Centered covariance spectrum
    # ------------------------------------------------------------

    X_centered = X_unit - centroid

    singular_values = np.linalg.svd(
        X_centered,
        compute_uv=False,
        full_matrices=False,
    )

    # Covariance eigenvalues:
    # lambda_j = s_j^2 / (n - 1)
    eigenvalues = (
        singular_values**2 / (n - 1)
    )

    total_variance = float(
        eigenvalues.sum()
    )

    maximum_available_rank = min(
        n - 1,
        d,
    )

    leading_singular_value = (
        float(singular_values[0])
        if singular_values.size
        else 0.0
    )

    sv_tolerance = (
        max(n, d)
        * np.finfo(np.float64).eps
        * leading_singular_value
    )

    active = singular_values > sv_tolerance

    # ------------------------------------------------------------
    # 7. Spectral metrics
    # ------------------------------------------------------------

    if np.isclose(
        total_variance,
        0.0,
        atol=1e-15,
        rtol=0.0,
    ):
        numerical_rank = 0
        normalized_eigenvalues = np.array([])

        spectral_entropy = np.nan
        spectral_entropy_normalized = np.nan
        spectral_effective_rank = np.nan
        participation_ratio = np.nan
        pc1_variance_share = np.nan

    else:
        numerical_rank = int(active.sum())

        active_eigenvalues = eigenvalues[active]

        normalized_eigenvalues = (
            active_eigenvalues
            / active_eigenvalues.sum()
        )

        # H = -sum(p_j log p_j)
        spectral_entropy = float(
            -np.sum(
                normalized_eigenvalues
                * np.log(normalized_eigenvalues)
            )
        )

        # Effective rank = exp(H)
        spectral_effective_rank = float(
            np.exp(spectral_entropy)
        )

        # PR = 1 / sum(p_j^2)
        participation_ratio = float(
            1.0
            / np.sum(normalized_eigenvalues**2)
        )

        # Proportion of variance explained by PC1
        pc1_variance_share = float(
            eigenvalues[0] / total_variance
        )

        if maximum_available_rank > 1:
            spectral_entropy_normalized = float(
                spectral_entropy
                / np.log(maximum_available_rank)
            )
        else:
            spectral_entropy_normalized = np.nan

    # ------------------------------------------------------------
    # 8. Collect scalar results
    # ------------------------------------------------------------

    scalar_metrics = {
        # Bookkeeping
        "n_utterances": int(n),
        "n_pairs": int(n_pairs),
        "n_words": (
            int(n_words)
            if n_words is not None
            else np.nan
        ),
        "n_turns": (
            int(n_turns)
            if n_turns is not None
            else np.nan
        ),
        "embedding_dimension": int(d),
        "k_used": int(k_used),

        # Pairwise cosine similarity
        "pairwise_cosine_similarity_mean": float(
            np.mean(pairwise_similarities)
        ),
        "pairwise_cosine_similarity_sd": _sample_sd(
            pairwise_similarities
        ),
        "pairwise_cosine_similarity_q10": float(
            sim_q10
        ),
        "pairwise_cosine_similarity_median": float(
            sim_median
        ),
        "pairwise_cosine_similarity_q90": float(
            sim_q90
        ),

        # Centroid concentration
        "centroid_resultant_length": (
            centroid_resultant_length
        ),
        "centroid_dispersion_cosine": (
            centroid_dispersion_cosine
        ),

        # Radial structure
        # The mean is retained mainly as an identity check.
        "radial_distance_mean_check": (
            radial_distance_mean
        ),
        "radial_distance_median": (
            radial_distance_median
        ),
        "radial_distance_q90": (
            radial_distance_q90
        ),

        # Pairwise cosine distance
        "pairwise_distance_mean": float(
            np.mean(pairwise_distances)
        ),
        "pairwise_distance_sd": _sample_sd(
            pairwise_distances
        ),
        "pairwise_distance_q10": float(
            dist_q10
        ),
        "pairwise_distance_median": float(
            dist_median
        ),
        "pairwise_distance_q90": float(
            dist_q90
        ),
        "pairwise_semantic_span": float(
            dist_q90 - dist_q10
        ),

        # Local-neighborhood structure
        "nearest_neighbor_distance_mean": float(
            np.mean(nearest_neighbor_distances)
        ),
        "nearest_neighbor_distance_median": float(
            np.median(nearest_neighbor_distances)
        ),
        "knn_distance_median": float(
            np.median(kth_neighbor_distances)
        ),

        # Covariance-spectrum structure
        "maximum_available_rank": int(
            maximum_available_rank
        ),
        "numerical_rank": int(
            numerical_rank
        ),
        "total_variance": (
            total_variance
        ),
        "spectral_entropy": (
            spectral_entropy
        ),
        "spectral_entropy_normalized": (
            spectral_entropy_normalized
        ),
        "spectral_effective_rank": (
            spectral_effective_rank
        ),
        "participation_ratio": (
            participation_ratio
        ),
        "pc1_variance_share": (
            pc1_variance_share
        ),
    }

    # Retain arrays separately so they do not clutter the DataFrame.
    spectrum = {
        "covariance_eigenvalues": eigenvalues,
        "normalized_eigenvalues": normalized_eigenvalues,
    }

    # ------------------------------------------------------------
    # 9. Mathematical consistency checks
    # ------------------------------------------------------------

    np.testing.assert_allclose(
        scalar_metrics["pairwise_distance_mean"],
        total_variance,
        atol=1e-10,
    )

    np.testing.assert_allclose(
        scalar_metrics[
            "pairwise_cosine_similarity_mean"
        ],
        1.0
        - scalar_metrics["pairwise_distance_mean"],
        atol=1e-10,
    )

    np.testing.assert_allclose(
        scalar_metrics["pairwise_distance_mean"],
        n / (n - 1)
        * (1.0 - centroid_resultant_length**2),
        atol=1e-10,
    )

    if np.isfinite(radial_distance_mean):
        np.testing.assert_allclose(
            radial_distance_mean,
            centroid_dispersion_cosine,
            atol=1e-10,
        )

    return scalar_metrics, spectrum


# ================================================================
# Calculate metrics for every conversation
# ================================================================

geometry_records = []
conversation_spectra = {}

group_columns = [
    "dataset",
    "conversation_id",
]

for group_key, group in geometry_utterances.groupby(
    group_columns,
    sort=False,
    observed=True,
):
    dataset, conversation_id = group_key

    # embedding_idx maps each utterance row to E_utterances.
    embedding_indices = group[
        "embedding_idx"
    ].to_numpy(dtype=int)

    conversation_embeddings = E_utterances[
        embedding_indices
    ]

    scalar_metrics, spectrum = (
        calculate_conversation_geometry(
            conversation_embeddings,
            n_words=group["n_words"].sum(),
            n_turns=group["turn_id"].nunique(),
            k=5,
        )
    )

    row = {
        "dataset": dataset,
        "conversation_id": conversation_id,
    }

    # Preserve useful conversation metadata where available.
    for column in [
        "video_id",
        "file_number",
        "level",
        "level_label",
    ]:
        if column in group.columns:
            row[column] = group[column].iloc[0]

    row.update(scalar_metrics)
    geometry_records.append(row)

    conversation_spectra[
        (dataset, conversation_id)
    ] = spectrum


conversation_geometry = pd.DataFrame(
    geometry_records
)

sort_columns = [
    column
    for column in [
        "level",
        "dataset",
        "conversation_id",
    ]
    if column in conversation_geometry.columns
]

conversation_geometry = (
    conversation_geometry
    .sort_values(sort_columns)
    .reset_index(drop=True)
)

print(
    "Conversations analyzed:",
    len(conversation_geometry),
)

print(
    "Geometry table shape:",
    conversation_geometry.shape,
)

display(conversation_geometry)

In [ ]:
conversation_spectra[
    (dataset, conversation_id)
]["normalized_eigenvalues"]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert",
]

def plot_metric_distribution(df, metric, ylabel=None):
    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="level_label",
        y=metric,
        order=level_order,
        showfliers=False,
        color="lightgray",
    )

    sns.stripplot(
        data=df,
        x="level_label",
        y=metric,
        order=level_order,
        color="black",
        alpha=0.6,
        size=4,
        jitter=0.18,
    )

    plt.xlabel("Partner expertise → smaller expertise gap")
    plt.ylabel(ylabel or metric)
    plt.title(ylabel or metric)
    plt.tight_layout()
    plt.show()

In [ ]:
conversation_geometry


In [ ]:
plot_metric_distribution(
    conversation_geometry,
    "total_variance"
)

In [ ]:
conversation_geometry

In [ ]:
def plot_metric_trajectories(df, metric, ylabel=None):
    fig, ax = plt.subplots(figsize=(8, 5))

    # One thin trajectory per topic
    sns.lineplot(
        data=df,
        x="level",
        y=metric,
        units="video_id",
        estimator=None,
        color="gray",
        alpha=0.3,
        linewidth=1,
        marker="o",
        markersize=3,
        ax=ax,
    )

    # Overall mean and 95% bootstrap CI
    sns.lineplot(
        data=df,
        x="level",
        y=metric,
        estimator="mean",
        errorbar=("ci", 95),
        color="black",
        linewidth=3,
        marker="o",
        markersize=7,
        ax=ax,
    )

    ax.set_xticks(range(5))
    ax.set_xticklabels(level_order)
    ax.set_xlabel("Partner expertise → smaller expertise gap")
    ax.set_ylabel(ylabel or metric)
    ax.set_title(ylabel or metric)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric_trajectories(
    conversation_geometry,
    "nearest_neighbor_distance_mean",
    "Mean nearest-neighbor distance",
)

In [ ]:
def plot_child_expert_difference(df, metric, ylabel=None):
    endpoints = (
        df[df["level_label"].isin(["child", "expert"])]
        .pivot(
            index="video_id",
            columns="level_label",
            values=metric,
        )
        .dropna()
    )

    endpoints["difference"] = (
        endpoints["expert"] - endpoints["child"]
    )

    endpoints = endpoints.sort_values("difference")

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(
        endpoints["difference"],
        endpoints.index,
        color="black",
        s=35,
    )

    ax.axvline(0, color="gray", linestyle="--")

    ax.axvline(
        endpoints["difference"].mean(),
        color="red",
        linewidth=2,
        label=(
            "Mean difference = "
            f"{endpoints['difference'].mean():.3f}"
        ),
    )

    ax.set_xlabel(
        f"Expert − child difference\n"
        f"({ylabel or metric})"
    )
    ax.set_ylabel("Topic")
    ax.set_title("Smallest gap versus largest gap")
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_sample_size_diagnostic(df, metric, ylabel=None):
    plt.figure(figsize=(7, 5))

    sns.scatterplot(
        data=df,
        x="n_utterances",
        y=metric,
        hue="level_label",
        hue_order=level_order,
        alpha=0.8,
    )

    sns.regplot(
        data=df,
        x="n_utterances",
        y=metric,
        scatter=False,
        color="black",
        line_kws={"linestyle": "--"},
    )

    plt.xlabel("Number of utterances")
    plt.ylabel(ylabel or metric)
    plt.title(f"{ylabel or metric} versus conversation size")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_sample_size_diagnostic(
    conversation_geometry,
    "participation_ratio",
    "Participation ratio",
)

plot_sample_size_diagnostic(
    conversation_geometry,
    "spectral_effective_rank",
    "Spectral effective rank",
)

In [ ]:
plot_metric_distribution(
    conversation_geometry,
    "nearest_neighbor_distance_mean",
    "Mean nearest-neighbor distance",
)

plot_metric_distribution(
    conversation_geometry,
    "spectral_entropy_normalized",
    "Normalized spectral entropy",
)

plot_metric_distribution(
    conversation_geometry,
    "pc1_variance_share",
    "PC1 variance share",
)

In [ ]:
conversation_geometry

In [ ]:
plot_metric_distribution(
    conversation_geometry,
    "pairwise_cosine_similarity_mean"
)

# dynamic metrics

In [ ]:
import re
import numpy as np
import pandas as pd


def _sample_sd(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan

    return float(np.std(values, ddof=1))


def _linear_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 2:
        return np.nan

    x = x[valid]
    y = y[valid]

    if np.isclose(np.ptp(x), 0):
        return np.nan

    return float(np.polyfit(x, y, deg=1)[0])


def _mean_or_nan(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.mean(values))


def standardize_speaker_role(value):
    """
    Convert existing speaker labels to 'expert' or 'learner'.

    This recognizes labels such as:
        Speaker A
        Speaker B
        Speaker 1
        expert
        child
        teenager
        college student
        graduate student
    """
    if pd.isna(value):
        return None

    label = str(value).strip().lower()

    if (
        label == "expert"
        or "speaker a" in label
        or re.search(r"\bspeaker\s*1\b", label)
    ):
        return "expert"

    learner_terms = [
        "learner",
        "speaker b",
        "child",
        "teen",
        "student",
        "undergraduate",
        "graduate",
        "college",
        "novice",
        "partner",
    ]

    if any(term in label for term in learner_terms):
        return "learner"

    return None

In [ ]:
#trajectory calculations
def _calculate_order_metrics(X):
    """
    Calculate metrics that depend directly on utterance order.
    X must already contain unit-normalized embeddings.
    """
    n = len(X)

    # ------------------------------------------------------------
    # Consecutive semantic movement
    # ------------------------------------------------------------

    adjacent_similarities = np.sum(
        X[:-1] * X[1:],
        axis=1,
    )

    adjacent_similarities = np.clip(
        adjacent_similarities,
        -1.0,
        1.0,
    )

    step_cosine_distances = (
        1.0 - adjacent_similarities
    )

    # Euclidean chord distances on the unit sphere.
    # Needed for valid path-efficiency calculations.
    step_chord_distances = np.linalg.norm(
        X[1:] - X[:-1],
        axis=1,
    )

    # Each step is placed at the midpoint between two utterances.
    step_times = (
        np.arange(n - 1) + 0.5
    ) / (n - 1)

    step_distance_slope = _linear_slope(
        step_times,
        step_cosine_distances,
    )

    early_steps = (
        step_cosine_distances[
            step_times <= 1 / 3
        ]
    )

    late_steps = (
        step_cosine_distances[
            step_times >= 2 / 3
        ]
    )

    early_step_mean = _mean_or_nan(early_steps)
    late_step_mean = _mean_or_nan(late_steps)

    # ------------------------------------------------------------
    # Path length, net displacement, and efficiency
    # ------------------------------------------------------------

    total_path_length_chord = float(
        np.sum(step_chord_distances)
    )

    net_displacement_chord = float(
        np.linalg.norm(X[-1] - X[0])
    )

    net_displacement_cosine = float(
        np.clip(
            1.0 - np.dot(X[0], X[-1]),
            0.0,
            2.0,
        )
    )

    if total_path_length_chord > 1e-15:
        trajectory_efficiency_chord = float(
            net_displacement_chord
            / total_path_length_chord
        )
    else:
        trajectory_efficiency_chord = np.nan

    # ------------------------------------------------------------
    # Directional persistence
    # ------------------------------------------------------------

    movement_vectors = X[1:] - X[:-1]

    movement_norms = np.linalg.norm(
        movement_vectors,
        axis=1,
    )

    valid_movements = movement_norms > 1e-15

    movement_directions = np.full_like(
        movement_vectors,
        np.nan,
    )

    movement_directions[valid_movements] = (
        movement_vectors[valid_movements]
        / movement_norms[valid_movements, None]
    )

    persistence_values = []

    for index in range(len(movement_directions) - 1):
        first = movement_directions[index]
        second = movement_directions[index + 1]

        if np.isfinite(first).all() and np.isfinite(second).all():
            persistence_values.append(
                np.clip(
                    np.dot(first, second),
                    -1.0,
                    1.0,
                )
            )

    persistence_values = np.asarray(
        persistence_values,
        dtype=float,
    )

    directional_persistence_mean = _mean_or_nan(
        persistence_values
    )

    if len(persistence_values):
        directional_reversal_rate = float(
            np.mean(persistence_values < 0)
        )
    else:
        directional_reversal_rate = np.nan

    # ------------------------------------------------------------
    # Change in movement magnitude
    # ------------------------------------------------------------

    step_size_changes = np.diff(
        step_cosine_distances
    )

    mean_absolute_step_change = _mean_or_nan(
        np.abs(step_size_changes)
    )

    # ------------------------------------------------------------
    # Distance from the beginning
    # ------------------------------------------------------------

    utterance_times = np.linspace(0, 1, n)

    distance_from_start = np.clip(
        1.0 - X @ X[0],
        0.0,
        2.0,
    )

    distance_from_start_slope = _linear_slope(
        utterance_times,
        distance_from_start,
    )

    distance_from_start_auc = float(
        np.trapz(
            distance_from_start,
            utterance_times,
        )
    )

    maximum_index = int(
        np.argmax(distance_from_start)
    )

    return {
        "step_cosine_distances":
            step_cosine_distances,

        "step_chord_distances":
            step_chord_distances,

        "step_times":
            step_times,

        "step_distance_slope":
            step_distance_slope,

        "step_distance_early_mean":
            early_step_mean,

        "step_distance_late_mean":
            late_step_mean,

        "step_distance_late_minus_early":
            late_step_mean - early_step_mean,

        "total_path_length_chord":
            total_path_length_chord,

        "net_displacement_chord":
            net_displacement_chord,

        "net_displacement_cosine":
            net_displacement_cosine,

        "trajectory_efficiency_chord":
            trajectory_efficiency_chord,

        "directional_persistence_values":
            persistence_values,

        "directional_persistence_mean":
            directional_persistence_mean,

        "directional_reversal_rate":
            directional_reversal_rate,

        "step_size_changes":
            step_size_changes,

        "mean_absolute_step_change":
            mean_absolute_step_change,

        "utterance_times":
            utterance_times,

        "distance_from_start":
            distance_from_start,

        "distance_from_start_slope":
            distance_from_start_slope,

        "distance_from_start_auc":
            distance_from_start_auc,

        "distance_from_start_max":
            float(distance_from_start[maximum_index]),

        "distance_from_start_max_time":
            float(utterance_times[maximum_index]),
    }

In [ ]:
#conversation-level dynamic function
#adds comparison against all pair geometry, recurrence, speaker transitions, and shuffle-order baselines
def calculate_conversation_dynamics(
    embeddings,
    speaker_roles=None,
    recurrence_min_lag=4,
    recurrence_max_lag=10,
    n_permutations=200,
    random_state=2026,
):
    """
    Calculate Stage 1 trajectory metrics for one conversation.

    Parameters
    ----------
    embeddings : array, shape (n_utterances, embedding_dimension)
        Embeddings in chronological order.

    speaker_roles : array-like, optional
        One standardized speaker role per utterance:
        'expert' or 'learner'.

    recurrence_min_lag : int
        Smallest utterance separation treated as nonlocal.

    recurrence_max_lag : int
        Largest separation used for medium-lag recurrence.

    n_permutations : int
        Number of fixed-endpoint shuffled-order baselines.

    random_state : int
        Reproducibility seed.

    Returns
    -------
    scalar_metrics : dict
        One-row-per-conversation summaries.

    dynamic_series : dict
        Arrays retained for later trajectory plots.
    """
    X = np.asarray(
        embeddings,
        dtype=np.float64,
    )

    if X.ndim != 2:
        raise ValueError(
            "embeddings must have shape "
            "(n_utterances, embedding_dimension)"
        )

    if len(X) < 4:
        raise ValueError(
            "At least four utterances are required "
            "for Stage 1 dynamics."
        )

    if not np.isfinite(X).all():
        raise ValueError(
            "Embeddings contain NaN or infinite values."
        )

    norms = np.linalg.norm(
        X,
        axis=1,
        keepdims=True,
    )

    if np.any(norms == 0):
        raise ValueError(
            "At least one embedding has zero norm."
        )

    X = X / norms

    n, d = X.shape

    ordered = _calculate_order_metrics(X)

    # ------------------------------------------------------------
    # 1. Static all-pair baseline
    # ------------------------------------------------------------

    similarity_matrix = np.clip(
        X @ X.T,
        -1.0,
        1.0,
    )

    upper_triangle = np.triu_indices(
        n,
        k=1,
    )

    all_pair_similarities = (
        similarity_matrix[upper_triangle]
    )

    all_pair_distances = (
        1.0 - all_pair_similarities
    )

    all_pair_distance_mean = float(
        np.mean(all_pair_distances)
    )

    all_pair_similarity_mean = float(
        np.mean(all_pair_similarities)
    )

    step_distance_mean = float(
        np.mean(
            ordered["step_cosine_distances"]
        )
    )

    # Negative = adjacent utterances are closer than arbitrary pairs.
    step_distance_excess = float(
        step_distance_mean
        - all_pair_distance_mean
    )

    # ------------------------------------------------------------
    # 2. Medium-lag recurrence
    # ------------------------------------------------------------

    largest_lag = min(
        recurrence_max_lag,
        n - 1,
    )

    recurrence_by_lag = {}

    if largest_lag >= recurrence_min_lag:
        for lag in range(
            recurrence_min_lag,
            largest_lag + 1,
        ):
            lag_similarities = np.sum(
                X[:-lag] * X[lag:],
                axis=1,
            )

            recurrence_by_lag[lag] = float(
                np.mean(
                    np.clip(
                        lag_similarities,
                        -1.0,
                        1.0,
                    )
                )
            )

        # Each lag receives equal weight.
        medium_lag_similarity_mean = float(
            np.mean(
                list(recurrence_by_lag.values())
            )
        )

        medium_lag_excess_similarity = float(
            medium_lag_similarity_mean
            - all_pair_similarity_mean
        )

    else:
        medium_lag_similarity_mean = np.nan
        medium_lag_excess_similarity = np.nan

    # Exploratory: closest eligible prior utterance.
    # This is more sensitive to conversation length.
    best_prior_similarities = []
    best_prior_lags = []

    for current_index in range(
        recurrence_min_lag,
        n,
    ):
        last_eligible_index = (
            current_index
            - recurrence_min_lag
        )

        prior_similarities = (
            X[:last_eligible_index + 1]
            @ X[current_index]
        )

        best_index = int(
            np.argmax(prior_similarities)
        )

        best_prior_similarities.append(
            prior_similarities[best_index]
        )

        best_prior_lags.append(
            current_index - best_index
        )

    # ------------------------------------------------------------
    # 3. Speaker-transition movement
    # ------------------------------------------------------------

    transition_results = {
        "speaker_switch_rate": np.nan,

        "expert_to_learner_step_mean": np.nan,
        "learner_to_expert_step_mean": np.nan,
        "expert_to_expert_step_mean": np.nan,
        "learner_to_learner_step_mean": np.nan,

        "n_expert_to_learner_steps": 0,
        "n_learner_to_expert_steps": 0,
        "n_expert_to_expert_steps": 0,
        "n_learner_to_learner_steps": 0,

        "learner_to_expert_minus_expert_to_learner_step":
            np.nan,
    }

    if speaker_roles is not None:
        roles = np.asarray(
            speaker_roles,
            dtype=object,
        )

        if len(roles) != n:
            raise ValueError(
                "speaker_roles must have one value "
                "per utterance."
            )

        transition_values = {
            "expert_to_learner": [],
            "learner_to_expert": [],
            "expert_to_expert": [],
            "learner_to_learner": [],
        }

        known_transitions = 0
        speaker_switches = 0

        for index, distance in enumerate(
            ordered["step_cosine_distances"]
        ):
            source = roles[index]
            target = roles[index + 1]

            if (
                source not in {"expert", "learner"}
                or target not in {"expert", "learner"}
            ):
                continue

            known_transitions += 1

            if source != target:
                speaker_switches += 1

            transition_name = (
                f"{source}_to_{target}"
            )

            transition_values[
                transition_name
            ].append(distance)

        if known_transitions:
            transition_results[
                "speaker_switch_rate"
            ] = (
                speaker_switches
                / known_transitions
            )

        for transition_name, values in (
            transition_values.items()
        ):
            transition_results[
                f"{transition_name}_step_mean"
            ] = _mean_or_nan(values)

            transition_results[
                f"n_{transition_name}_steps"
            ] = len(values)

        learner_to_expert = transition_results[
            "learner_to_expert_step_mean"
        ]

        expert_to_learner = transition_results[
            "expert_to_learner_step_mean"
        ]

        if (
            np.isfinite(learner_to_expert)
            and np.isfinite(expert_to_learner)
        ):
            transition_results[
                "learner_to_expert_minus_"
                "expert_to_learner_step"
            ] = (
                learner_to_expert
                - expert_to_learner
            )

    # ------------------------------------------------------------
    # 4. Fixed-endpoint shuffle baselines
    # ------------------------------------------------------------

    shuffle_metric_names = [
        "step_distance_slope",
        "trajectory_efficiency_chord",
        "directional_persistence_mean",
        "distance_from_start_auc",
    ]

    shuffle_values = {
        name: []
        for name in shuffle_metric_names
    }

    rng = np.random.default_rng(
        random_state
    )

    if n_permutations > 0 and n > 3:
        interior_indices = np.arange(
            1,
            n - 1,
        )

        for _ in range(n_permutations):
            shuffled_indices = np.concatenate([
                [0],
                rng.permutation(
                    interior_indices
                ),
                [n - 1],
            ])

            shuffled = _calculate_order_metrics(
                X[shuffled_indices]
            )

            for name in shuffle_metric_names:
                shuffle_values[name].append(
                    shuffled[name]
                )

    shuffle_results = {}

    for name in shuffle_metric_names:
        null_values = np.asarray(
            shuffle_values[name],
            dtype=float,
        )

        null_values = null_values[
            np.isfinite(null_values)
        ]

        observed_value = ordered[name]

        if (
            len(null_values)
            and np.isfinite(observed_value)
        ):
            null_mean = float(
                np.mean(null_values)
            )

            null_sd = (
                float(np.std(null_values, ddof=1))
                if len(null_values) > 1
                else np.nan
            )

            shuffle_results[
                f"{name}_shuffle_mean"
            ] = null_mean

            shuffle_results[
                f"{name}_excess_vs_shuffle"
            ] = (
                observed_value - null_mean
            )

            if (
                np.isfinite(null_sd)
                and null_sd > 1e-15
            ):
                shuffle_results[
                    f"{name}_shuffle_z"
                ] = (
                    observed_value - null_mean
                ) / null_sd
            else:
                shuffle_results[
                    f"{name}_shuffle_z"
                ] = np.nan
        else:
            shuffle_results[
                f"{name}_shuffle_mean"
            ] = np.nan

            shuffle_results[
                f"{name}_excess_vs_shuffle"
            ] = np.nan

            shuffle_results[
                f"{name}_shuffle_z"
            ] = np.nan

    # ------------------------------------------------------------
    # 5. Collect scalar outputs
    # ------------------------------------------------------------

    scalar_metrics = {
        "n_utterances": int(n),
        "n_steps": int(n - 1),
        "embedding_dimension": int(d),

        # Consecutive movement
        "step_cosine_distance_mean":
            step_distance_mean,

        "step_cosine_distance_sd":
            _sample_sd(
                ordered["step_cosine_distances"]
            ),

        "step_cosine_distance_median":
            float(
                np.median(
                    ordered[
                        "step_cosine_distances"
                    ]
                )
            ),

        "step_cosine_distance_q90":
            float(
                np.quantile(
                    ordered[
                        "step_cosine_distances"
                    ],
                    0.90,
                )
            ),

        # Dynamic movement relative to global geometry
        "all_pair_distance_mean_check":
            all_pair_distance_mean,

        "step_distance_excess_over_all_pairs":
            step_distance_excess,

        # Development over time
        "step_distance_slope":
            ordered["step_distance_slope"],

        "step_distance_early_mean":
            ordered["step_distance_early_mean"],

        "step_distance_late_mean":
            ordered["step_distance_late_mean"],

        "step_distance_late_minus_early":
            ordered[
                "step_distance_late_minus_early"
            ],

        # Path structure
        "total_path_length_chord":
            ordered["total_path_length_chord"],

        "net_displacement_chord":
            ordered["net_displacement_chord"],

        "net_displacement_cosine":
            ordered["net_displacement_cosine"],

        "trajectory_efficiency_chord":
            ordered[
                "trajectory_efficiency_chord"
            ],

        # Direction and volatility
        "directional_persistence_mean":
            ordered[
                "directional_persistence_mean"
            ],

        "directional_reversal_rate":
            ordered[
                "directional_reversal_rate"
            ],

        "mean_absolute_step_change":
            ordered[
                "mean_absolute_step_change"
            ],

        # Recurrence
        "medium_lag_similarity_mean":
            medium_lag_similarity_mean,

        "medium_lag_excess_similarity":
            medium_lag_excess_similarity,

        "best_prior_recurrence_similarity_mean":
            _mean_or_nan(
                best_prior_similarities
            ),

        "best_prior_recurrence_lag_mean":
            _mean_or_nan(
                best_prior_lags
            ),

        # Development away from the beginning
        "distance_from_start_slope":
            ordered[
                "distance_from_start_slope"
            ],

        "distance_from_start_auc":
            ordered[
                "distance_from_start_auc"
            ],

        "distance_from_start_max":
            ordered[
                "distance_from_start_max"
            ],

        "distance_from_start_max_time":
            ordered[
                "distance_from_start_max_time"
            ],

        # Null-model bookkeeping
        "n_order_permutations":
            int(n_permutations),
    }

    scalar_metrics.update(
        transition_results
    )

    scalar_metrics.update(
        shuffle_results
    )

    dynamic_series = {
        "step_times":
            ordered["step_times"],

        "step_cosine_distances":
            ordered["step_cosine_distances"],

        "step_chord_distances":
            ordered["step_chord_distances"],

        "directional_persistence":
            ordered[
                "directional_persistence_values"
            ],

        "utterance_times":
            ordered["utterance_times"],

        "distance_from_start":
            ordered["distance_from_start"],

        "recurrence_by_lag":
            recurrence_by_lag,
    }

    # Mathematical check for chord-distance efficiency.
    if np.isfinite(
        scalar_metrics[
            "trajectory_efficiency_chord"
        ]
    ):
        assert (
            -1e-10
            <= scalar_metrics[
                "trajectory_efficiency_chord"
            ]
            <= 1 + 1e-10
        )

    return scalar_metrics, dynamic_series

apply calculations to each conversation

In [ ]:
# Find the speaker column used in the existing table.
speaker_candidates = [
    "speaker_role",
    "speaker",
    "speaker_label",
]

speaker_column = next(
    (
        column
        for column in speaker_candidates
        if column in geometry_utterances.columns
    ),
    None,
)

print("Speaker column:", speaker_column)

if speaker_column is None:
    print(
        "No recognized speaker column found. "
        "General dynamics will still be calculated, "
        "but speaker-transition metrics will be missing."
    )

In [ ]:
dynamic_records = []
conversation_dynamic_series = {}

group_columns = [
    "dataset",
    "conversation_id",
]

for group_number, (group_key, group) in enumerate(
    geometry_utterances.groupby(
        group_columns,
        sort=False,
        observed=True,
    )
):
    dataset, conversation_id = group_key

    # embedding_idx follows the original transcript order.
    group = (
        group
        .sort_values(
            "embedding_idx",
            kind="stable",
        )
        .copy()
    )

    embedding_indices = group[
        "embedding_idx"
    ].to_numpy(dtype=int)

    conversation_embeddings = (
        E_utterances[embedding_indices]
    )

    if speaker_column is not None:
        speaker_roles = (
            group[speaker_column]
            .map(standardize_speaker_role)
            .to_numpy()
        )
    else:
        speaker_roles = None

    scalar_metrics, dynamic_series = (
        calculate_conversation_dynamics(
            conversation_embeddings,
            speaker_roles=speaker_roles,

            # Exclude the preceding three utterances.
            recurrence_min_lag=4,

            # Examine returns 4–10 utterances later.
            recurrence_max_lag=10,

            n_permutations=200,

            # Different but reproducible seed per conversation.
            random_state=2026 + group_number,
        )
    )

    row = {
        "dataset": dataset,
        "conversation_id": conversation_id,
    }

    for column in [
        "video_id",
        "file_number",
        "level",
        "level_label",
    ]:
        if column in group.columns:
            row[column] = group[
                column
            ].iloc[0]

    if "n_words" in group.columns:
        row["n_words"] = int(
            group["n_words"].sum()
        )
    elif "text" in group.columns:
        row["n_words"] = int(
            group["text"]
            .astype(str)
            .str.split()
            .str.len()
            .sum()
        )

    if "turn_id" in group.columns:
        row["n_turns"] = int(
            group["turn_id"].nunique()
        )

    row.update(scalar_metrics)
    dynamic_records.append(row)

    conversation_dynamic_series[
        (dataset, conversation_id)
    ] = dynamic_series


conversation_dynamics = pd.DataFrame(
    dynamic_records
)

sort_columns = [
    column
    for column in [
        "level",
        "dataset",
        "conversation_id",
    ]
    if column in conversation_dynamics.columns
]

conversation_dynamics = (
    conversation_dynamics
    .sort_values(sort_columns)
    .reset_index(drop=True)
)

print(
    "Conversations analyzed:",
    len(conversation_dynamics),
)

print(
    "Dynamics table shape:",
    conversation_dynamics.shape,
)

display(conversation_dynamics)

In [ ]:
if speaker_column is not None:
    speaker_mapping_check = (
        geometry_utterances[
            [speaker_column]
        ]
        .drop_duplicates()
        .assign(
            standardized_role=lambda frame:
                frame[speaker_column].map(
                    standardize_speaker_role
                )
        )
    )

    display(speaker_mapping_check)

## visualizations for trajectory metrics

matched topic trajectories for each summary metric

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert",
]

level_x = {
    level: index
    for index, level in enumerate(level_order)
}


def plot_dynamic_metric_trajectories(
    df,
    metric,
    ylabel=None,
    zero_reference=False,
    ax=None,
):
    plot_df = df.dropna(subset=[metric]).copy()

    plot_df["_level_x"] = (
        plot_df["level_label"]
        .astype(str)
        .str.lower()
        .map(level_x)
    )

    plot_df["_video_group"] = (
        plot_df["dataset"].astype(str)
        + "::"
        + plot_df["video_id"].astype(str)
    )

    plot_df = plot_df.dropna(
        subset=["_level_x"]
    )

    if ax is None:
        _, ax = plt.subplots(
            figsize=(8, 5)
        )

    # One matched trajectory per video
    for _, video_df in plot_df.groupby(
        "_video_group"
    ):
        video_df = video_df.sort_values(
            "_level_x"
        )

        ax.plot(
            video_df["_level_x"],
            video_df[metric],
            color="gray",
            alpha=0.25,
            linewidth=1,
            marker="o",
            markersize=3,
        )

    # Mean across videos
    level_means = (
        plot_df.groupby("_level_x")[metric]
        .mean()
        .reindex(range(5))
    )

    ax.plot(
        range(5),
        level_means,
        color="black",
        linewidth=3,
        marker="o",
        markersize=7,
    )

    if zero_reference:
        ax.axhline(
            0,
            color="red",
            linestyle="--",
            linewidth=1,
        )

    ax.set_xticks(range(5))
    ax.set_xticklabels(
        level_order,
        rotation=20,
    )

    ax.set_xlabel(
        "Partner expertise → smaller expertise gap"
    )

    ax.set_ylabel(ylabel or metric)
    ax.set_title(ylabel or metric)

    return ax

In [ ]:
plot_dynamic_metric_trajectories(
    conversation_dynamics,
    metric="step_distance_excess_over_all_pairs",
    ylabel="Adjacent-distance excess",
    zero_reference=True,
)

plt.tight_layout()
plt.show()

In [ ]:
dynamic_metric_labels = {
    "step_distance_excess_over_all_pairs":
        "Sequential coherence",

    "step_distance_slope_excess_vs_shuffle":
        "Change in movement over time",

    "trajectory_efficiency_chord_excess_vs_shuffle":
        "Trajectory efficiency vs. shuffle",

    "directional_persistence_mean_excess_vs_shuffle":
        "Directional persistence vs. shuffle",

    "mean_absolute_step_change":
        "Movement volatility",

    "medium_lag_excess_similarity":
        "Medium-lag recurrence",

    "distance_from_start_auc_excess_vs_shuffle":
        "Sustained departure vs. shuffle",

    "learner_to_expert_minus_expert_to_learner_step":
        "Response-transition asymmetry",
}

In [ ]:
fig, axes = plt.subplots(
    4,
    2,
    figsize=(14, 18),
)

axes = axes.flatten()

for ax, (metric, label) in zip(
    axes,
    dynamic_metric_labels.items(),
):
    plot_dynamic_metric_trajectories(
        conversation_dynamics,
        metric=metric,
        ylabel=label,
        zero_reference=(
            "excess" in metric
            or "minus" in metric
        ),
        ax=ax,
    )

plt.tight_layout()
plt.show()

In [ ]:
def plot_observed_vs_shuffle(
    df,
    observed_metric,
    shuffle_metric,
    label=None,
):
    plot_df = df.dropna(
        subset=[
            observed_metric,
            shuffle_metric,
        ]
    )

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    sns.scatterplot(
        data=plot_df,
        x=shuffle_metric,
        y=observed_metric,
        hue="level_label",
        hue_order=level_order,
        s=55,
        alpha=0.8,
        ax=ax,
    )

    minimum = min(
        plot_df[observed_metric].min(),
        plot_df[shuffle_metric].min(),
    )

    maximum = max(
        plot_df[observed_metric].max(),
        plot_df[shuffle_metric].max(),
    )

    ax.plot(
        [minimum, maximum],
        [minimum, maximum],
        color="black",
        linestyle="--",
    )

    ax.set_xlabel("Mean shuffled-order value")
    ax.set_ylabel("Observed-order value")
    ax.set_title(label or observed_metric)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_observed_vs_shuffle(
    conversation_dynamics,
    observed_metric=
        "trajectory_efficiency_chord",
    shuffle_metric=
        "trajectory_efficiency_chord_shuffle_mean",
    label="Observed versus shuffled trajectory efficiency",
)

interpretation: above diagonal-> observed convo has higher efficiency than shuffled baseline
colors separating differently around diagonal -> strength of temporal organization vaires with gap

In [ ]:
plot_observed_vs_shuffle(
    conversation_dynamics,
    "directional_persistence_mean",
    "directional_persistence_mean_shuffle_mean",
    "Observed versus shuffled directional persistence",
)

plot_observed_vs_shuffle(
    conversation_dynamics,
    "distance_from_start_auc",
    "distance_from_start_auc_shuffle_mean",
    "Observed versus shuffled distance-from-start AUC",
)

### visualizting the actual temporal series

In [ ]:
#binned time series table
def build_binned_dynamic_series(
    conversation_dynamics,
    conversation_dynamic_series,
    time_key,
    value_key,
    n_bins=10,
):
    records = []

    for _, row in (
        conversation_dynamics.iterrows()
    ):
        key = (
            row["dataset"],
            row["conversation_id"],
        )

        if key not in conversation_dynamic_series:
            continue

        series = conversation_dynamic_series[key]

        times = np.asarray(
            series[time_key],
            dtype=float,
        )

        values = np.asarray(
            series[value_key],
            dtype=float,
        )

        valid = (
            np.isfinite(times)
            & np.isfinite(values)
        )

        times = times[valid]
        values = values[valid]

        bin_number = np.minimum(
            np.floor(times * n_bins).astype(int),
            n_bins - 1,
        )

        temporary = pd.DataFrame({
            "time_bin": bin_number,
            "value": values,
        })

        # One value per conversation per time bin
        temporary = (
            temporary
            .groupby("time_bin", as_index=False)
            ["value"]
            .mean()
        )

        temporary["normalized_time"] = (
            temporary["time_bin"] + 0.5
        ) / n_bins

        temporary["dataset"] = row["dataset"]
        temporary["conversation_id"] = (
            row["conversation_id"]
        )
        temporary["video_id"] = row["video_id"]
        temporary["level_label"] = (
            str(row["level_label"]).lower()
        )

        records.append(temporary)

    return pd.concat(
        records,
        ignore_index=True,
    )

In [ ]:
#plot sematic movement over tim e
step_time_df = build_binned_dynamic_series(
    conversation_dynamics,
    conversation_dynamic_series,
    time_key="step_times",
    value_key="step_cosine_distances",
    n_bins=10,
)

plt.figure(figsize=(9, 6))

sns.lineplot(
    data=step_time_df,
    x="normalized_time",
    y="value",
    hue="level_label",
    hue_order=level_order,
    estimator="mean",
    errorbar=("se", 1),
    linewidth=2,
)

plt.xlabel("Normalized conversation time")
plt.ylabel("Mean consecutive cosine distance")
plt.title("Semantic movement over the conversation")
plt.tight_layout()
plt.show()

In [ ]:
#distance from beginning
start_distance_df = (
    build_binned_dynamic_series(
        conversation_dynamics,
        conversation_dynamic_series,
        time_key="utterance_times",
        value_key="distance_from_start",
        n_bins=10,
    )
)

plt.figure(figsize=(9, 6))

sns.lineplot(
    data=start_distance_df,
    x="normalized_time",
    y="value",
    hue="level_label",
    hue_order=level_order,
    estimator="mean",
    errorbar=("se", 1),
    linewidth=2,
)

plt.xlabel("Normalized conversation time")
plt.ylabel("Cosine distance from first utterance")
plt.title("Semantic departure from the conversation opening")
plt.tight_layout()
plt.show()

In [ ]:
#when conversation RETURN to related semantic material
recurrence_records = []

for _, row in conversation_dynamics.iterrows():
    key = (
        row["dataset"],
        row["conversation_id"],
    )

    recurrence_by_lag = (
        conversation_dynamic_series[key]
        ["recurrence_by_lag"]
    )

    for lag, similarity in (
        recurrence_by_lag.items()
    ):
        recurrence_records.append({
            "dataset": row["dataset"],
            "conversation_id":
                row["conversation_id"],
            "video_id": row["video_id"],
            "level_label":
                str(row["level_label"]).lower(),
            "lag": lag,
            "similarity": similarity,
        })

recurrence_lag_df = pd.DataFrame(
    recurrence_records
)

In [ ]:
plt.figure(figsize=(9, 6))

sns.lineplot(
    data=recurrence_lag_df,
    x="lag",
    y="similarity",
    hue="level_label",
    hue_order=level_order,
    estimator="mean",
    errorbar=("se", 1),
    marker="o",
)

plt.xlabel("Utterance lag")
plt.ylabel("Mean cosine similarity")
plt.title("Semantic similarity across utterance lags")
plt.tight_layout()
plt.show()

interpretation: slow decline -> semantic material remains active over longer spans
lower curve -> faster semantic departure

In [ ]:
#speaker transition plot
transition_long = conversation_dynamics.melt(
    id_vars=[
        "dataset",
        "video_id",
        "conversation_id",
        "level_label",
    ],
    value_vars=[
        "expert_to_learner_step_mean",
        "learner_to_expert_step_mean",
    ],
    var_name="transition",
    value_name="mean_step_distance",
)

transition_long["transition"] = (
    transition_long["transition"].map({
        "expert_to_learner_step_mean":
            "Expert → learner",

        "learner_to_expert_step_mean":
            "Learner → expert",
    })
)

In [ ]:
plt.figure(figsize=(9, 6))

sns.pointplot(
    data=transition_long,
    x="level_label",
    y="mean_step_distance",
    hue="transition",
    order=level_order,
    errorbar=("ci", 95),
    dodge=0.25,
    markers=["o", "s"],
)

plt.xlabel(
    "Partner expertise → smaller expertise gap"
)
plt.ylabel("Mean response-transition distance")
plt.title("Semantic movement by response direction")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

interpretation: learner to expert higher -> expert response introduce more semantic movement. if lines converge as expertise increases, it would suggest that response direction asymmetry shrinks with gap. diverging lines would suggest more asymmetry

## change focus to time windows

previous part looks at OVERALL CONVERSATION'S TRAJECTORY. now look at HOW local geometry changes from beginning to end

divide each convo into 5 windows, each containing 20% of the conversation.(important to note that this can cause issues since utterance count/amount of text within each window will differ)

In [ ]:
import numpy as np
import pandas as pd

n_windows = 5

window_count_records = []

for group_key, group in geometry_utterances.groupby(
    ["dataset", "conversation_id"],
    sort=False,
    observed=True,
):
    dataset, conversation_id = group_key

    group = (
        group
        .sort_values(
            "embedding_idx",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    window_positions = np.array_split(
        np.arange(len(group)),
        n_windows,
    )

    for window_index, positions in enumerate(
        window_positions,
        start=1,
    ):
        row = {
            "dataset": dataset,
            "conversation_id": conversation_id,
            "window_index": window_index,
            "n_window_utterances": len(positions),
        }

        for column in [
            "video_id",
            "level",
            "level_label",
        ]:
            if column in group.columns:
                row[column] = group[column].iloc[0]

        window_count_records.append(row)


window_counts = pd.DataFrame(
    window_count_records
)

display(
    window_counts[
        "n_window_utterances"
    ].describe()
)

display(
    window_counts.sort_values(
        "n_window_utterances"
    ).head(20)
)

In [ ]:
print("Number of windows:", len(window_counts))

print(
    window_counts.groupby(
        ["dataset", "conversation_id"]
    ).size().value_counts()
)

In [ ]:
target_m = 20

minimum_window_size = int(
    window_counts[
        "n_window_utterances"
    ].min()
)

m_per_window = min(
    target_m,
    minimum_window_size,
)

print("Smallest window:", minimum_window_size)
print("Utterances sampled per window:", m_per_window)

In [ ]:
conversation_sizes = (
    geometry_utterances
    .groupby(
        ["dataset", "conversation_id"],
        observed=True,
        sort=False,
    )
    .agg(
        video_id=("video_id", "first"),
        level=("level", "first"),
        level_label=("level_label", "first"),
        n_utterances=("embedding_idx", "size"),
    )
    .reset_index()
)

display(conversation_sizes["n_utterances"].describe())
display(conversation_sizes.head())

In [ ]:
candidate_windows = [5, 4, 3]
candidate_m = [8, 10, 12, 15, 20]

design_records = []

for candidate_n_windows in candidate_windows:
    for candidate_sample_size in candidate_m:
        required_utterances = (
            candidate_n_windows
            * candidate_sample_size
        )

        temporary = conversation_sizes.copy()

        temporary["eligible"] = (
            temporary["n_utterances"]
            >= required_utterances
        )

        video_summary = (
            temporary
            .groupby(
                ["dataset", "video_id"],
                observed=True,
            )
            .agg(
                n_levels=(
                    "level_label",
                    "nunique",
                ),
                all_levels_eligible=(
                    "eligible",
                    "all",
                ),
            )
            .reset_index()
        )

        fully_matched_videos = (
            video_summary[
                (
                    video_summary["n_levels"] == 5
                )
                &
                (
                    video_summary[
                        "all_levels_eligible"
                    ]
                )
            ]
        )

        design_records.append({
            "n_windows":
                candidate_n_windows,

            "m_per_window":
                candidate_sample_size,

            "required_utterances":
                required_utterances,

            "eligible_conversations":
                int(
                    temporary[
                        "eligible"
                    ].sum()
                ),

            "eligible_conversations_percent":
                100 * temporary[
                    "eligible"
                ].mean(),

            "fully_matched_videos":
                len(fully_matched_videos),

            "matched_conversations":
                5 * len(
                    fully_matched_videos
                ),
        })


design_comparison = pd.DataFrame(
    design_records
)

display(
    design_comparison.sort_values(
        [
            "n_windows",
            "m_per_window",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

In [ ]:
window_metrics = [
    "pairwise_distance_mean",
    "pairwise_distance_sd",
    "radial_distance_q90",
    "nearest_neighbor_distance_mean",
    "knn_distance_median",
    "spectral_entropy_normalized",
    "participation_ratio",
    "pc1_variance_share",
]

In [ ]:
n_windows = 3
m_per_window = 10
n_resamples = 200

In [ ]:
required_stage2_utterances = n_windows * m_per_window

conversation_sizes["eligible_stage2"] = (
    conversation_sizes["n_utterances"]
    >= required_stage2_utterances
)

print("Required utterances:", required_stage2_utterances)
print(
    "Eligible conversations:",
    int(conversation_sizes["eligible_stage2"].sum()),
    "of",
    len(conversation_sizes),
)

In [ ]:
eligibility_by_level = (
    conversation_sizes
    .groupby(
        "level_label",
        observed=True,
    )
    .agg(
        total_conversations=(
            "conversation_id",
            "count",
        ),
        eligible_conversations=(
            "eligible_stage2",
            "sum",
        ),
    )
)

eligibility_by_level[
    "eligible_percent"
] = (
    100
    * eligibility_by_level[
        "eligible_conversations"
    ]
    / eligibility_by_level[
        "total_conversations"
    ]
)

display(eligibility_by_level)

In [ ]:
video_eligibility = (
    conversation_sizes
    .groupby(
        ["dataset", "video_id"],
        observed=True,
    )
    .agg(
        n_levels=(
            "level_label",
            "nunique",
        ),
        all_eligible=(
            "eligible_stage2",
            "all",
        ),
    )
    .reset_index()
)

retained_videos = video_eligibility[
    (
        video_eligibility["n_levels"] == 5
    )
    &
    (
        video_eligibility["all_eligible"]
    )
][
    ["dataset", "video_id"]
].copy()

print(
    "Matched videos retained:",
    len(retained_videos),
)

In [ ]:
stage2_utterances_matched = (
    geometry_utterances
    .merge(
        retained_videos,
        on=[
            "dataset",
            "video_id",
        ],
        how="inner",
    )
    .copy()
)

print(
    "Matched conversations:",
    stage2_utterances_matched[
        ["dataset", "conversation_id"]
    ].drop_duplicates().shape[0]
)

In [ ]:
eligible_conversation_keys = (
    conversation_sizes[
        conversation_sizes[
            "eligible_stage2"
        ]
    ][
        ["dataset", "conversation_id"]
    ]
)

stage2_utterances_unbalanced = (
    geometry_utterances
    .merge(
        eligible_conversation_keys,
        on=[
            "dataset",
            "conversation_id",
        ],
        how="inner",
    )
    .copy()
)

print(
    "Eligible conversations:",
    stage2_utterances_unbalanced[
        ["dataset", "conversation_id"]
    ].drop_duplicates().shape[0]
)

In [ ]:
grouped_conversations = (
    stage2_utterances_matched
    .groupby(
        ["dataset", "conversation_id"],
        sort=False,
        observed=True,
    )
)
n_windows = 3
m_per_window = 10
n_resamples = 200

In [ ]:
window_geometry_records = []
window_metric_draws = {}

grouped_conversations = (
    stage2_utterances_matched
    .groupby(
        ["dataset", "conversation_id"],
        sort=False,
        observed=True,
    )
)

for group_number, (group_key, group) in enumerate(
    grouped_conversations
):
    dataset, conversation_id = group_key

    # Preserve chronological order
    group = (
        group
        .sort_values(
            "embedding_idx",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    embedding_indices = group[
        "embedding_idx"
    ].to_numpy(dtype=int)

    conversation_embeddings = (
        E_utterances[embedding_indices]
    )

    # Divide into beginning, middle, and end
    window_positions_list = np.array_split(
        np.arange(len(group)),
        n_windows,
    )

    for zero_based_window, window_positions in enumerate(
        window_positions_list
    ):
        window_index = zero_based_window + 1

        assert len(window_positions) >= m_per_window

        # Reproducible independent RNG per window
        seed_sequence = np.random.SeedSequence([
            2026,
            group_number,
            window_index,
        ])

        rng = np.random.default_rng(
            seed_sequence
        )

        # If exactly 10 utterances exist, there is only
        # one possible sample.
        repetitions_used = (
            1
            if len(window_positions) == m_per_window
            else n_resamples
        )

        draw_records = []

        for repetition in range(repetitions_used):
            sampled_positions = rng.choice(
                window_positions,
                size=m_per_window,
                replace=False,
            )

            sampled_embeddings = (
                conversation_embeddings[
                    sampled_positions
                ]
            )

            scalar_metrics, _ = (
                calculate_conversation_geometry(
                    sampled_embeddings,
                    k=5,
                )
            )

            draw_records.append({
                "repetition": repetition,
                **{
                    metric:
                        scalar_metrics[metric]
                    for metric in window_metrics
                },
            })

        draws_df = pd.DataFrame(
            draw_records
        )

        row = {
            "dataset": dataset,
            "conversation_id": conversation_id,
            "window_index": window_index,

            # Actual proportional midpoint:
            # 1/6, 1/2, 5/6
            "window_midpoint": (
                zero_based_window + 0.5
            ) / n_windows,

            # Beginning=0, middle=0.5, end=1
            "time_linear": (
                zero_based_window
                / (n_windows - 1)
            ),

            # Beginning=-0.5, middle=0, end=0.5
            "time_c": (
                zero_based_window
                / (n_windows - 1)
                - 0.5
            ),

            "n_window_utterances":
                len(window_positions),

            "m_sampled":
                m_per_window,

            "n_resamples_used":
                repetitions_used,
        }

        # Retain conversation metadata
        for column in [
            "video_id",
            "file_number",
            "level",
            "level_label",
        ]:
            if column in group.columns:
                row[column] = group[
                    column
                ].iloc[0]

        # Average the repeated subsample estimates
        for metric in window_metrics:
            values = draws_df[
                metric
            ].to_numpy(dtype=float)

            row[metric] = float(
                np.nanmean(values)
            )

            # Monte Carlo/subsampling variability
            row[f"{metric}_mc_sd"] = (
                float(
                    np.nanstd(
                        values,
                        ddof=1,
                    )
                )
                if np.isfinite(values).sum() > 1
                else np.nan
            )

        window_geometry_records.append(row)

        # Retain individual draws for reliability checks
        window_metric_draws[
            (
                dataset,
                conversation_id,
                window_index,
            )
        ] = draws_df

In [ ]:
window_geometry = pd.DataFrame(
    window_geometry_records
)

window_geometry = (
    window_geometry
    .sort_values([
        "dataset",
        "conversation_id",
        "window_index",
    ])
    .reset_index(drop=True)
)

display(window_geometry)

In [ ]:
assert window_geometry[
    "pairwise_distance_mean"
].between(0, 2).all()

assert window_geometry[
    "nearest_neighbor_distance_mean"
].between(0, 2).all()

assert window_geometry[
    "spectral_entropy_normalized"
].between(0, 1).all()

assert window_geometry[
    "pc1_variance_share"
].between(0, 1).all()

assert window_geometry[
    "participation_ratio"
].between(1, 9).all()

In [ ]:
reliability_records = []

for metric in window_metrics:
    mc_column = f"{metric}_mc_sd"

    reliability_records.append({
        "metric": metric,

        "median_mc_sd":
            window_geometry[
                mc_column
            ].median(),

        "q90_mc_sd":
            window_geometry[
                mc_column
            ].quantile(0.90),

        "maximum_mc_sd":
            window_geometry[
                mc_column
            ].max(),
    })

window_reliability = pd.DataFrame(
    reliability_records
)

display(window_reliability)


In [ ]:
window_geometry

summary: evaluated window design options. since some conversations were just too short, only included videos that fit the design with three sections, ten utterances per window. 16 videos were eligible. the ten utterances in each window were sampled without replacement and the sampling was repeated 200 times(reduces dependence on one arbitrary selection of utterances). resulting estimates were averaged across repetitions, and standard deviation across repetitions was retained as a measure of subsampling variability.window-level metrics were calculated for each (mean and standard deviation of pairwise cosine distance, radial Q90 distance, mean nearest-neighbor distance, median k-nearest-neighbor distance, normalized spectral entropy, participation ratio, and PC1 variance share.)

## visualizing window level stuff

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert",
]

level_palette = dict(
    zip(
        level_order,
        sns.color_palette(
            "viridis",
            len(level_order),
        ),
    )
)

window_plot_df = window_geometry.copy()

window_plot_df["level_label"] = (
    window_plot_df["level_label"]
    .astype(str)
    .str.lower()
)

window_plot_df["window_label"] = (
    window_plot_df["window_index"].map({
        1: "Beginning",
        2: "Middle",
        3: "End",
    })
)

window_plot_df["video_group"] = (
    window_plot_df["dataset"].astype(str)
    + "::"
    + window_plot_df["video_id"].astype(str)
)

window_plot_df["conversation_group"] = (
    window_plot_df["dataset"].astype(str)
    + "::"
    + window_plot_df[
        "conversation_id"
    ].astype(str)
)

In [ ]:
#mean developmental trajectories
def plot_window_development(
    df,
    metric,
    ylabel=None,
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    sns.lineplot(
        data=df,
        x="window_index",
        y=metric,
        hue="level_label",
        hue_order=level_order,
        palette=level_palette,
        estimator="mean",
        errorbar=("ci", 95),
        marker="o",
        markersize=8,
        linewidth=2.5,
        ax=ax,
    )

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([
        "Beginning",
        "Middle",
        "End",
    ])

    ax.set_xlabel("Conversation stage")
    ax.set_ylabel(ylabel or metric)
    ax.set_title(
        f"{ylabel or metric} across the conversation"
    )

    ax.legend(
        title="Partner expertise",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_window_development(
    window_plot_df,
    metric="pc1_variance_share",
    ylabel="PC1 variance share",
)

In [ ]:
window_metric_labels = {
    "pairwise_distance_mean":
        "Mean pairwise distance",

    "pairwise_distance_sd":
        "Pairwise-distance SD",

    "radial_distance_q90":
        "Radial Q90",

    "nearest_neighbor_distance_mean":
        "Mean NN distance",

    "knn_distance_median":
        "Median kNN distance",

    "spectral_entropy_normalized":
        "Normalized spectral entropy",

    "participation_ratio":
        "Participation ratio",

    "pc1_variance_share":
        "PC1 variance share",
}

In [ ]:
fig, axes = plt.subplots(
    4,
    2,
    figsize=(14, 20),
)

axes = axes.flatten()

for ax, (metric, label) in zip(
    axes,
    window_metric_labels.items(),
):
    sns.lineplot(
        data=window_plot_df,
        x="window_index",
        y=metric,
        hue="level_label",
        hue_order=level_order,
        palette=level_palette,
        estimator="mean",
        errorbar=None,
        marker="o",
        linewidth=2,
        ax=ax,
        legend=False,
    )

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([
        "Beginning",
        "Middle",
        "End",
    ])

    ax.set_xlabel("")
    ax.set_ylabel(label)
    ax.set_title(label)

# One shared legend
handles = [
    plt.Line2D(
        [0],
        [0],
        color=level_palette[level],
        marker="o",
        linewidth=2,
        label=level,
    )
    for level in level_order
]

fig.legend(
    handles=handles,
    labels=level_order,
    title="Partner expertise",
    loc="center right",
)

plt.tight_layout(
    rect=[0, 0, 0.88, 1]
)

plt.show()

## individual video trajectories

In [ ]:
def plot_video_window_trajectories(
    df,
    metric,
    ylabel=None,
):
    fig, axes = plt.subplots(
        1,
        5,
        figsize=(18, 4),
        sharex=True,
        sharey=True,
    )

    for ax, level in zip(
        axes,
        level_order,
    ):
        level_df = df[
            df["level_label"] == level
        ]

        for _, video_df in level_df.groupby(
            "video_group"
        ):
            video_df = video_df.sort_values(
                "window_index"
            )

            ax.plot(
                video_df["window_index"],
                video_df[metric],
                color=level_palette[level],
                alpha=0.25,
                linewidth=1,
                marker="o",
                markersize=3,
            )

        level_mean = (
            level_df
            .groupby("window_index")[metric]
            .mean()
        )

        ax.plot(
            level_mean.index,
            level_mean.values,
            color="black",
            linewidth=3,
            marker="o",
            markersize=6,
        )

        ax.set_title(level)
        ax.set_xticks([1, 2, 3])
        ax.set_xticklabels([
            "Beginning",
            "Middle",
            "End",
        ], rotation=30)

    axes[0].set_ylabel(
        ylabel or metric
    )

    fig.suptitle(
        ylabel or metric,
        y=1.05,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_video_window_trajectories(
    window_plot_df,
    "spectral_entropy_normalized",
    "Normalized spectral entropy",
)

In [ ]:
#beginning to end changes
def calculate_window_changes(
    df,
    metric,
):
    change_df = (
        df.pivot(
            index=[
                "dataset",
                "video_id",
                "conversation_id",
                "level_label",
                "video_group",
            ],
            columns="window_index",
            values=metric,
        )
        .dropna(
            subset=[1, 3]
        )
        .reset_index()
    )

    change_df["beginning"] = (
        change_df[1]
    )

    change_df["middle"] = (
        change_df[2]
    )

    change_df["end"] = (
        change_df[3]
    )

    change_df["end_minus_beginning"] = (
        change_df["end"]
        - change_df["beginning"]
    )

    return change_df

In [ ]:
def plot_beginning_end_changes(
    df,
    metric,
    ylabel=None,
):
    change_df = calculate_window_changes(
        df,
        metric,
    )

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    sns.boxplot(
        data=change_df,
        x="level_label",
        y="end_minus_beginning",
        order=level_order,
        showfliers=False,
        color="lightgray",
        ax=ax,
    )

    sns.stripplot(
        data=change_df,
        x="level_label",
        y="end_minus_beginning",
        order=level_order,
        hue="level_label",
        hue_order=level_order,
        palette=level_palette,
        jitter=0.15,
        size=5,
        alpha=0.75,
        legend=False,
        ax=ax,
    )

    ax.axhline(
        0,
        color="black",
        linestyle="--",
    )

    ax.set_xlabel(
        "Partner expertise → smaller expertise gap"
    )

    ax.set_ylabel(
        f"End − beginning\n"
        f"({ylabel or metric})"
    )

    ax.set_title(
        "Beginning-to-end geometric change"
    )

    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_beginning_end_changes(
    window_plot_df,
    "participation_ratio",
    "Participation ratio",
)

In [ ]:
def plot_matched_change_trajectories(
    df,
    metric,
    ylabel=None,
):
    change_df = calculate_window_changes(
        df,
        metric,
    )

    level_x = {
        level: index
        for index, level in enumerate(
            level_order
        )
    }

    change_df["_level_x"] = (
        change_df["level_label"]
        .map(level_x)
    )

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    for _, video_df in change_df.groupby(
        "video_group"
    ):
        video_df = video_df.sort_values(
            "_level_x"
        )

        ax.plot(
            video_df["_level_x"],
            video_df[
                "end_minus_beginning"
            ],
            color="gray",
            alpha=0.3,
            linewidth=1,
            marker="o",
            markersize=3,
        )

    mean_change = (
        change_df
        .groupby("_level_x")[
            "end_minus_beginning"
        ]
        .mean()
        .reindex(range(5))
    )

    ax.plot(
        range(5),
        mean_change,
        color="black",
        linewidth=3,
        marker="o",
        markersize=7,
    )

    ax.axhline(
        0,
        color="red",
        linestyle="--",
    )

    ax.set_xticks(range(5))
    ax.set_xticklabels(
        level_order,
        rotation=20,
    )

    ax.set_xlabel(
        "Partner expertise → smaller expertise gap"
    )

    ax.set_ylabel(
        f"End − beginning\n"
        f"({ylabel or metric})"
    )

    ax.set_title(
        "Matched developmental changes"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
window_plot_df

In [ ]:
plot_matched_change_trajectories(window_plot_df, "nearest_neighbor_distance_mean")

In [ ]:
def plot_change_heatmap(
    df,
    metric,
    label=None,
):
    change_df = calculate_window_changes(
        df,
        metric,
    )

    change_matrix = (
        change_df.pivot(
            index="video_group",
            columns="level_label",
            values="end_minus_beginning",
        )
        .reindex(columns=level_order)
    )

    # Order videos by their average change
    row_order = (
        change_matrix.mean(axis=1)
        .sort_values()
        .index
    )

    change_matrix = change_matrix.loc[
        row_order
    ]

    maximum_absolute_value = np.nanmax(
        np.abs(change_matrix.to_numpy())
    )

    plt.figure(figsize=(9, 9))

    sns.heatmap(
        change_matrix,
        cmap="coolwarm",
        center=0,
        vmin=-maximum_absolute_value,
        vmax=maximum_absolute_value,
        linewidths=0.5,
        cbar_kws={
            "label":
                "End − beginning"
        },
    )

    plt.xlabel("Partner expertise")
    plt.ylabel("Video/topic")
    plt.title(
        label or metric
    )

    plt.tight_layout()
    plt.show()

# response-focus metrics

In [ ]:
stage3_source = (
    stage2_utterances_matched.copy()
)

role_map = {
    "expert": "expert",
    "partner": "partner",
    "learner": "partner",
}

stage3_source["_role"] = (
    stage3_source["speaker_role"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(role_map)
)

print(
    stage3_source["_role"]
    .value_counts(dropna=False)
)

assert stage3_source["_role"].notna().all()

collapse utterances into turns

In [ ]:
turn_records = []
turn_embedding_list = []

conversation_groups = (
    stage3_source.groupby(
        ["dataset", "conversation_id"],
        sort=False,
        observed=True,
    )
)

for group_key, group in conversation_groups:
    dataset, conversation_id = group_key

    group = (
        group
        .sort_values(
            "embedding_idx",
            kind="stable",
        )
        .reset_index(drop=True)
        .copy()
    )

    # A new turn begins whenever the speaker changes.
    group["_new_turn"] = (
        group["_role"]
        .ne(group["_role"].shift())
    )

    group["_turn_order"] = (
        group["_new_turn"]
        .cumsum()
        .astype(int)
        - 1
    )

    for turn_order, turn_group in group.groupby(
        "_turn_order",
        sort=True,
    ):
        embedding_indices = turn_group[
            "embedding_idx"
        ].to_numpy(dtype=int)

        utterance_embeddings = (
            E_utterances[
                embedding_indices
            ]
        )

        turn_centroid = (
            utterance_embeddings.mean(axis=0)
        )

        turn_resultant_length = float(
            np.linalg.norm(turn_centroid)
        )

        if turn_resultant_length == 0:
            raise ValueError(
                f"Zero-norm turn embedding in "
                f"{conversation_id}, turn {turn_order}"
            )

        turn_embedding = (
            turn_centroid
            / turn_resultant_length
        )

        turn_embedding_idx = len(
            turn_embedding_list
        )

        turn_embedding_list.append(
            turn_embedding
        )

        row = {
            "dataset": dataset,
            "conversation_id": conversation_id,
            "turn_order": int(turn_order),
            "turn_embedding_idx":
                turn_embedding_idx,

            "speaker_role":
                turn_group["_role"].iloc[0],

            "n_utterances_in_turn":
                len(turn_group),

            # Within-turn semantic concentration
            "turn_resultant_length":
                turn_resultant_length,

            "first_embedding_idx":
                int(embedding_indices.min()),

            "last_embedding_idx":
                int(embedding_indices.max()),
        }

        if "text" in turn_group.columns:
            row["turn_text"] = " ".join(
                turn_group["text"].astype(str)
            )

            row["n_words_in_turn"] = int(
                turn_group["text"]
                .astype(str)
                .str.split()
                .str.len()
                .sum()
            )

        for column in [
            "video_id",
            "file_number",
            "level",
            "level_label",
        ]:
            if column in turn_group.columns:
                row[column] = turn_group[
                    column
                ].iloc[0]

        turn_records.append(row)


turns = pd.DataFrame(
    turn_records
)

E_turns = np.vstack(
    turn_embedding_list
).astype(np.float64)

turns = (
    turns
    .sort_values([
        "dataset",
        "conversation_id",
        "turn_order",
    ])
    .reset_index(drop=True)
)

print("Turns:", len(turns))
print("Turn embeddings:", E_turns.shape)

display(turns.head())

In [ ]:
assert len(turns) == len(E_turns)

assert np.allclose(
    np.linalg.norm(E_turns, axis=1),
    1.0,
    atol=1e-10,
)

# Every turn must contain only one role.
assert turns["speaker_role"].isin(
    ["expert", "partner"]
).all()

In [ ]:
alternation_checks = []

for _, group in turns.groupby(
    ["dataset", "conversation_id"],
    observed=True,
):
    roles = group.sort_values(
        "turn_order"
    )["speaker_role"].to_numpy()

    alternation_checks.append(
        np.all(roles[1:] != roles[:-1])
    )

assert all(alternation_checks)

response transition table

calculate metrics for the transitions

In [ ]:
def normalized_recent_centroid(
    X,
    indices,
    history_k=3,
):
    indices = np.asarray(
        indices,
        dtype=int,
    )

    if len(indices) == 0:
        return None

    recent_indices = indices[
        -history_k:
    ]

    centroid = X[
        recent_indices
    ].mean(axis=0)

    centroid_norm = np.linalg.norm(
        centroid
    )

    if centroid_norm == 0:
        return None

    return centroid / centroid_norm


def cosine_similarity(first, second):
    return float(
        np.clip(
            np.dot(first, second),
            -1.0,
            1.0,
        )
    )


def cosine_distance(first, second):
    return float(
        1.0
        - cosine_similarity(
            first,
            second,
        )
    )

In [ ]:
history_k = 3
transition_records = []

for group_key, group in turns.groupby(
    ["dataset", "conversation_id"],
    sort=False,
    observed=True,
):
    dataset, conversation_id = group_key

    group = (
        group
        .sort_values("turn_order")
        .reset_index(drop=True)
    )

    embedding_indices = group[
        "turn_embedding_idx"
    ].to_numpy(dtype=int)

    X = E_turns[
        embedding_indices
    ]

    roles = group[
        "speaker_role"
    ].to_numpy()

    n_turns = len(group)
    n_transitions = n_turns - 1

    if n_transitions < 1:
        continue

    for current_index in range(
        1,
        n_turns,
    ):
        source_index = current_index - 1

        source_role = roles[
            source_index
        ]

        responder_role = roles[
            current_index
        ]

        source_embedding = X[
            source_index
        ]

        response_embedding = X[
            current_index
        ]

        response_similarity = (
            cosine_similarity(
                response_embedding,
                source_embedding,
            )
        )

        response_distance = (
            1.0 - response_similarity
        )

        # --------------------------------------------------------
        # Responder's previous turn
        # --------------------------------------------------------

        previous_own_indices = np.flatnonzero(
            roles[:current_index]
            == responder_role
        )

        if len(previous_own_indices):
            previous_own_index = int(
                previous_own_indices[-1]
            )

            self_similarity = (
                cosine_similarity(
                    response_embedding,
                    X[previous_own_index],
                )
            )

            self_update_distance = (
                1.0 - self_similarity
            )

            immediate_accommodation = (
                response_similarity
                - self_similarity
            )

        else:
            previous_own_index = None
            self_similarity = np.nan
            self_update_distance = np.nan
            immediate_accommodation = np.nan

        # --------------------------------------------------------
        # Recent speaker-history centroids
        # --------------------------------------------------------

        previous_other_indices = np.flatnonzero(
            roles[:current_index]
            != responder_role
        )

        own_history_centroid = (
            normalized_recent_centroid(
                X,
                previous_own_indices,
                history_k=history_k,
            )
        )

        other_history_centroid = (
            normalized_recent_centroid(
                X,
                previous_other_indices,
                history_k=history_k,
            )
        )

        if (
            own_history_centroid is not None
            and other_history_centroid is not None
        ):
            own_history_similarity = (
                cosine_similarity(
                    response_embedding,
                    own_history_centroid,
                )
            )

            other_history_similarity = (
                cosine_similarity(
                    response_embedding,
                    other_history_centroid,
                )
            )

            history_accommodation = (
                other_history_similarity
                - own_history_similarity
            )
        else:
            own_history_similarity = np.nan
            other_history_similarity = np.nan
            history_accommodation = np.nan

        # --------------------------------------------------------
        # Expert–partner centroid convergence
        # --------------------------------------------------------

        before_expert_indices = np.flatnonzero(
            roles[:current_index]
            == "expert"
        )

        before_partner_indices = np.flatnonzero(
            roles[:current_index]
            == "partner"
        )

        after_expert_indices = np.flatnonzero(
            roles[:current_index + 1]
            == "expert"
        )

        after_partner_indices = np.flatnonzero(
            roles[:current_index + 1]
            == "partner"
        )

        expert_before = normalized_recent_centroid(
            X,
            before_expert_indices,
            history_k=history_k,
        )

        partner_before = normalized_recent_centroid(
            X,
            before_partner_indices,
            history_k=history_k,
        )

        expert_after = normalized_recent_centroid(
            X,
            after_expert_indices,
            history_k=history_k,
        )

        partner_after = normalized_recent_centroid(
            X,
            after_partner_indices,
            history_k=history_k,
        )

        if (
            expert_before is not None
            and partner_before is not None
        ):
            role_distance_before = (
                cosine_distance(
                    expert_before,
                    partner_before,
                )
            )
        else:
            role_distance_before = np.nan

        if (
            expert_after is not None
            and partner_after is not None
        ):
            role_distance_after = (
                cosine_distance(
                    expert_after,
                    partner_after,
                )
            )
        else:
            role_distance_after = np.nan

        if (
            np.isfinite(role_distance_before)
            and np.isfinite(role_distance_after)
        ):
            convergence_contribution = (
                role_distance_before
                - role_distance_after
            )
        else:
            convergence_contribution = np.nan

        # --------------------------------------------------------
        # Normalized response time
        # --------------------------------------------------------

        if n_transitions > 1:
            time_linear = (
                current_index - 1
            ) / (n_transitions - 1)
        else:
            time_linear = 0.5

        time_c = (
            time_linear - 0.5
        )

        window_index = min(
            int(
                np.floor(
                    time_linear * 3
                )
            )
            + 1,
            3,
        )

        row = {
            "dataset": dataset,
            "conversation_id":
                conversation_id,

            "transition_order":
                current_index,

            "source_role":
                source_role,

            "responder_role":
                responder_role,

            "transition_type":
                f"{source_role}_to_"
                f"{responder_role}",

            "time_linear":
                time_linear,

            "time_c":
                time_c,

            "window_index":
                window_index,

            "response_similarity":
                response_similarity,

            "response_distance":
                response_distance,

            "self_similarity":
                self_similarity,

            "self_update_distance":
                self_update_distance,

            "immediate_accommodation_index":
                immediate_accommodation,

            "own_history_similarity":
                own_history_similarity,

            "other_history_similarity":
                other_history_similarity,

            "history_accommodation_index":
                history_accommodation,

            "rolling_role_centroid_distance_before":
                role_distance_before,

            "rolling_role_centroid_distance_after":
                role_distance_after,

            "convergence_contribution":
                convergence_contribution,
        }

        for column in [
            "video_id",
            "file_number",
            "level",
            "level_label",
        ]:
            if column in group.columns:
                row[column] = group[
                    column
                ].iloc[0]

        transition_records.append(row)


turn_transitions = pd.DataFrame(
    transition_records
)

turn_transitions = (
    turn_transitions
    .sort_values([
        "dataset",
        "conversation_id",
        "transition_order",
    ])
    .reset_index(drop=True)
)

display(turn_transitions.head())

In [ ]:
#validation
print(
    "Transitions:",
    len(turn_transitions)
)

print(
    turn_transitions[
        "transition_type"
    ].value_counts()
)

print(
    turn_transitions[
        [
            "response_distance",
            "self_update_distance",
            "immediate_accommodation_index",
            "history_accommodation_index",
            "convergence_contribution",
        ]
    ].isna().sum()
)

In [ ]:
assert set(
    turn_transitions[
        "transition_type"
    ].unique()
).issubset({
    "expert_to_partner",
    "partner_to_expert",
})

In [ ]:
assert turn_transitions[
    "response_distance"
].between(0, 2).all()

In [ ]:
response_window_geometry = (
    turn_transitions
    .groupby(
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level",
            "level_label",
            "window_index",
            "responder_role",
        ],
        observed=True,
    )
    .agg(
        n_responses=(
            "response_distance",
            "size",
        ),

        response_distance_mean=(
            "response_distance",
            "mean",
        ),

        response_distance_sd=(
            "response_distance",
            "std",
        ),

        n_self_updates=(
            "self_update_distance",
            "count",
        ),

        self_update_distance_mean=(
            "self_update_distance",
            "mean",
        ),

        immediate_accommodation_mean=(
            "immediate_accommodation_index",
            "mean",
        ),

        n_history_accommodation=(
            "history_accommodation_index",
            "count",
        ),

        history_accommodation_mean=(
            "history_accommodation_index",
            "mean",
        ),

        n_convergence_contributions=(
            "convergence_contribution",
            "count",
        ),

        convergence_contribution_mean=(
            "convergence_contribution",
            "mean",
        ),
    )
    .reset_index()
)

In [ ]:
window_time_map = {
    1: -0.5,
    2: 0.0,
    3: 0.5,
}

gap_map = {
    "expert": 0,
    "graduate": 1,
    "undergraduate": 2,
    "teenager": 3,
    "child": 4,
}

response_window_geometry[
    "time_c"
] = (
    response_window_geometry[
        "window_index"
    ].map(window_time_map)
)

response_window_geometry[
    "level_label"
] = (
    response_window_geometry[
        "level_label"
    ]
    .astype(str)
    .str.lower()
)

response_window_geometry[
    "gap_score"
] = (
    response_window_geometry[
        "level_label"
    ].map(gap_map)
)

response_window_geometry[
    "gap_c"
] = (
    response_window_geometry[
        "gap_score"
    ]
    - 2
)

response_window_geometry[
    "responder_expert"
] = (
    response_window_geometry[
        "responder_role"
    ]
    == "expert"
).astype(int)

response_window_geometry[
    "video_group"
] = (
    response_window_geometry[
        "dataset"
    ].astype(str)
    + "::"
    + response_window_geometry[
        "video_id"
    ].astype(str)
)

response_window_geometry[
    "conversation_group"
] = (
    response_window_geometry[
        "dataset"
    ].astype(str)
    + "::"
    + response_window_geometry[
        "conversation_id"
    ].astype(str)
)

response_window_geometry[
    "window_group"
] = (
    response_window_geometry[
        "conversation_group"
    ]
    + "::"
    + response_window_geometry[
        "window_index"
    ].astype(str)
)

display(response_window_geometry.head())

In [ ]:
display(
    response_window_geometry[
        "n_responses"
    ].describe()
)

display(
    response_window_geometry[
        "n_responses"
    ].value_counts().sort_index()
)

In [ ]:
response_cells_primary = (
    response_window_geometry[
        response_window_geometry[
            "n_responses"
        ] >= 2
    ]
    .copy()
)

## model response distance

![img](img.png)

In [ ]:
import statsmodels.formula.api as smf


def prepare_stage3_outcome(
    df,
    outcome,
    count_column=None,
    minimum_count=2,
):
    model_df = df.copy()

    if count_column is not None:
        model_df = model_df[
            model_df[count_column]
            >= minimum_count
        ]

    required = [
        outcome,
        "gap_c",
        "time_c",
        "responder_expert",
        "video_group",
        "conversation_group",
        "window_group",
    ]

    model_df = (
        model_df[required]
        .dropna()
        .copy()
    )

    outcome_sd = model_df[
        outcome
    ].std(ddof=1)

    model_df["outcome_z"] = (
        model_df[outcome]
        - model_df[outcome].mean()
    ) / outcome_sd

    return model_df

In [ ]:
response_model_df = (
    prepare_stage3_outcome(
        response_window_geometry,
        outcome="response_distance_mean",
        count_column="n_responses",
        minimum_count=2,
    )
)

response_model = smf.mixedlm(
    (
        "outcome_z ~ "
        "gap_c * time_c * responder_expert"
    ),
    data=response_model_df,
    groups="video_group",
    re_formula="1",
    vc_formula={
        "conversation_intercept":
            "0 + C(conversation_group)",

        "window_intercept":
            "0 + C(window_group)",
    },
).fit(
    reml=True,
    method="lbfgs",
)

print(response_model.summary())

## model accommodation


In [ ]:
accommodation_model_df = (
    prepare_stage3_outcome(
        response_window_geometry,
        outcome=
            "history_accommodation_mean",
        count_column=
            "n_history_accommodation",
        minimum_count=2,
    )
)

accommodation_model = smf.mixedlm(
    (
        "outcome_z ~ "
        "gap_c * time_c * responder_expert"
    ),
    data=accommodation_model_df,
    groups="video_group",
    re_formula="1",
    vc_formula={
        "conversation_intercept":
            "0 + C(conversation_group)",

        "window_intercept":
            "0 + C(window_group)",
    },
).fit(
    reml=True,
    method="lbfgs",
)

print(accommodation_model.summary())

In [ ]:
#convergence contribution
convergence_model_df = (
    prepare_stage3_outcome(
        response_window_geometry,
        outcome=
            "convergence_contribution_mean",
        count_column=
            "n_convergence_contributions",
        minimum_count=2,
    )
)

convergence_model = smf.mixedlm(
    (
        "outcome_z ~ "
        "gap_c * time_c * responder_expert"
    ),
    data=convergence_model_df,
    groups="video_group",
    re_formula="1",
    vc_formula={
        "conversation_intercept":
            "0 + C(conversation_group)",

        "window_intercept":
            "0 + C(window_group)",
    },
).fit(
    reml=True,
    method="lbfgs",
)

print(convergence_model.summary())

In [ ]:
#OVERALL DYADIC ALIGNMENT. overall expert-partner distance within each window
alignment_window_geometry = (
    turn_transitions
    .groupby(
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level",
            "level_label",
            "window_index",
        ],
        observed=True,
    )
    .agg(
        n_alignment_observations=(
            "rolling_role_centroid_distance_after",
            "count",
        ),

        rolling_role_centroid_distance_mean=(
            "rolling_role_centroid_distance_after",
            "mean",
        ),
    )
    .reset_index()
)

alignment_window_geometry[
    "time_c"
] = (
    alignment_window_geometry[
        "window_index"
    ].map(window_time_map)
)

alignment_window_geometry[
    "level_label"
] = (
    alignment_window_geometry[
        "level_label"
    ]
    .astype(str)
    .str.lower()
)

alignment_window_geometry[
    "gap_score"
] = (
    alignment_window_geometry[
        "level_label"
    ].map(gap_map)
)

alignment_window_geometry[
    "gap_c"
] = (
    alignment_window_geometry[
        "gap_score"
    ]
    - 2
)

alignment_window_geometry[
    "video_group"
] = (
    alignment_window_geometry[
        "dataset"
    ].astype(str)
    + "::"
    + alignment_window_geometry[
        "video_id"
    ].astype(str)
)

alignment_window_geometry[
    "conversation_group"
] = (
    alignment_window_geometry[
        "dataset"
    ].astype(str)
    + "::"
    + alignment_window_geometry[
        "conversation_id"
    ].astype(str)
)

In [ ]:
alignment_model_df = (
    alignment_window_geometry[
        alignment_window_geometry[
            "n_alignment_observations"
        ] >= 2
    ]
    .dropna(
        subset=[
            "rolling_role_centroid_distance_mean",
            "gap_c",
            "time_c",
        ]
    )
    .copy()
)

alignment_outcome = (
    "rolling_role_centroid_distance_mean"
)

alignment_model_df[
    "outcome_z"
] = (
    alignment_model_df[
        alignment_outcome
    ]
    - alignment_model_df[
        alignment_outcome
    ].mean()
) / alignment_model_df[
    alignment_outcome
].std(ddof=1)

In [ ]:
alignment_model = smf.mixedlm(
    "outcome_z ~ gap_c * time_c",
    data=alignment_model_df,
    groups="video_group",
    re_formula="1",
    vc_formula={
        "conversation_intercept":
            "0 + C(conversation_group)"
    },
).fit(
    reml=True,
    method="lbfgs",
)

print(alignment_model.summary())

## refocus on adaptation to "novelty"

In [ ]:
adaptation_records = []
history_k = 3

for group_key, group in turns.groupby(
    ["dataset", "conversation_id"],
    sort=False,
    observed=True,
):
    dataset, conversation_id = group_key

    group = (
        group
        .sort_values("turn_order")
        .reset_index(drop=True)
    )

    embedding_indices = group[
        "turn_embedding_idx"
    ].to_numpy(dtype=int)

    X = E_turns[embedding_indices]

    roles = group[
        "speaker_role"
    ].to_numpy()

    n_turns = len(group)
    n_transitions = n_turns - 1

    for current_index in range(
        1,
        n_turns,
    ):
        source_index = current_index - 1

        # We specifically want:
        # expert introduction → partner response
        if not (
            roles[source_index] == "expert"
            and roles[current_index] == "partner"
        ):
            continue

        expert_input = X[source_index]
        partner_response = X[current_index]

        previous_partner_indices = (
            np.flatnonzero(
                roles[:current_index]
                == "partner"
            )
        )

        # First partner response has no prior
        # articulated partner state.
        if len(previous_partner_indices):
            previous_partner_index = int(
                previous_partner_indices[-1]
            )

            previous_partner = X[
                previous_partner_index
            ]

            recent_partner_indices = (
                previous_partner_indices[
                    -history_k:
                ]
            )

            partner_history_centroid = (
                normalized_recent_centroid(
                    X,
                    recent_partner_indices,
                    history_k=history_k,
                )
            )

            # ------------------------------------------
            # Expert novelty relative to partner history
            # ------------------------------------------

            expert_novelty_to_centroid = (
                cosine_distance(
                    expert_input,
                    partner_history_centroid,
                )
            )

            similarities_to_recent_partner = (
                X[recent_partner_indices]
                @ expert_input
            )

            expert_novelty_to_nearest = float(
                1.0
                - np.max(
                    similarities_to_recent_partner
                )
            )

            # ------------------------------------------
            # Before-versus-after partner uptake
            # ------------------------------------------

            distance_before = cosine_distance(
                previous_partner,
                expert_input,
            )

            distance_after = cosine_distance(
                partner_response,
                expert_input,
            )

            uptake_closure = (
                distance_before
                - distance_after
            )

            if distance_before > 1e-12:
                relative_uptake_closure = (
                    uptake_closure
                    / distance_before
                )
            else:
                relative_uptake_closure = np.nan

            # ------------------------------------------
            # Direction of partner movement
            # ------------------------------------------

            partner_update_vector = (
                partner_response
                - previous_partner
            )

            expert_target_vector = (
                expert_input
                - previous_partner
            )

            update_norm = np.linalg.norm(
                partner_update_vector
            )

            target_norm = np.linalg.norm(
                expert_target_vector
            )

            if (
                update_norm > 1e-12
                and target_norm > 1e-12
            ):
                directional_adaptation = float(
                    np.clip(
                        np.dot(
                            partner_update_vector,
                            expert_target_vector,
                        )
                        / (
                            update_norm
                            * target_norm
                        ),
                        -1.0,
                        1.0,
                    )
                )
            else:
                directional_adaptation = np.nan

        else:
            previous_partner_index = None
            expert_novelty_to_centroid = np.nan
            expert_novelty_to_nearest = np.nan
            distance_before = np.nan
            distance_after = cosine_distance(
                partner_response,
                expert_input,
            )
            uptake_closure = np.nan
            relative_uptake_closure = np.nan
            directional_adaptation = np.nan

        # ----------------------------------------------
        # Forward alignment with the next expert turn
        # ----------------------------------------------

        later_expert_indices = (
            np.flatnonzero(
                roles[current_index + 1:]
                == "expert"
            )
            + current_index
            + 1
        )

        if len(later_expert_indices):
            next_expert_index = int(
                later_expert_indices[0]
            )

            next_expert = X[
                next_expert_index
            ]

            current_expert_similarity = (
                cosine_similarity(
                    partner_response,
                    expert_input,
                )
            )

            next_expert_similarity = (
                cosine_similarity(
                    partner_response,
                    next_expert,
                )
            )

            forward_alignment_excess = (
                next_expert_similarity
                - current_expert_similarity
            )
        else:
            next_expert_similarity = np.nan
            forward_alignment_excess = np.nan

        # ----------------------------------------------
        # Retention in the partner's next turn
        # ----------------------------------------------

        later_partner_indices = (
            np.flatnonzero(
                roles[current_index + 1:]
                == "partner"
            )
            + current_index
            + 1
        )

        if (
            len(later_partner_indices)
            and np.isfinite(distance_before)
        ):
            next_partner_index = int(
                later_partner_indices[0]
            )

            retained_distance = cosine_distance(
                X[next_partner_index],
                expert_input,
            )

            retained_uptake_closure = (
                distance_before
                - retained_distance
            )
        else:
            retained_distance = np.nan
            retained_uptake_closure = np.nan

        if n_transitions > 1:
            time_linear = (
                current_index - 1
            ) / (n_transitions - 1)
        else:
            time_linear = 0.5

        window_index = min(
            int(
                np.floor(
                    time_linear * 3
                )
            ) + 1,
            3,
        )

        row = {
            "dataset": dataset,
            "conversation_id":
                conversation_id,

            "expert_turn_order":
                source_index,

            "partner_response_turn_order":
                current_index,

            "time_linear":
                time_linear,

            "time_c":
                time_linear - 0.5,

            "window_index":
                window_index,

            "expert_novelty_to_centroid":
                expert_novelty_to_centroid,

            "expert_novelty_to_nearest":
                expert_novelty_to_nearest,

            "partner_distance_before":
                distance_before,

            "partner_distance_after":
                distance_after,

            "uptake_closure":
                uptake_closure,

            "relative_uptake_closure":
                relative_uptake_closure,

            "directional_adaptation":
                directional_adaptation,

            "next_expert_similarity":
                next_expert_similarity,

            "forward_alignment_excess":
                forward_alignment_excess,

            "retained_distance":
                retained_distance,

            "retained_uptake_closure":
                retained_uptake_closure,
        }

        for column in [
            "video_id",
            "file_number",
            "level",
            "level_label",
        ]:
            if column in group.columns:
                row[column] = group[
                    column
                ].iloc[0]

        adaptation_records.append(row)


partner_adaptation_events = pd.DataFrame(
    adaptation_records
)

partner_adaptation_events[
    "level_label"
] = (
    partner_adaptation_events[
        "level_label"
    ]
    .astype(str)
    .str.lower()
)

partner_adaptation_events[
    "window_label"
] = (
    partner_adaptation_events[
        "window_index"
    ].map({
        1: "Beginning",
        2: "Middle",
        3: "End",
    })
)

display(partner_adaptation_events.head())

In [ ]:
adaptation_valid = (
    partner_adaptation_events
    .dropna(
        subset=[
            "partner_distance_before",
            "partner_distance_after",
        ]
    )
)

g = sns.relplot(
    data=adaptation_valid,
    x="partner_distance_before",
    y="partner_distance_after",
    hue="level_label",
    hue_order=level_order,
    col="window_label",
    col_order=[
        "Beginning",
        "Middle",
        "End",
    ],
    palette=level_palette,
    alpha=0.45,
    s=35,
    height=4,
    aspect=1,
)

for ax in g.axes.flat:
    lower = min(
        ax.get_xlim()[0],
        ax.get_ylim()[0],
    )

    upper = max(
        ax.get_xlim()[1],
        ax.get_ylim()[1],
    )

    ax.plot(
        [lower, upper],
        [lower, upper],
        color="black",
        linestyle="--",
    )

g.set_axis_labels(
    "Distance before expert introduction",
    "Distance after partner response",
)

g.fig.suptitle(
    "Partner movement toward expert-introduced information",
    y=1.05,
)

plt.show()

In [ ]:
novelty_uptake = (
    partner_adaptation_events
    .dropna(
        subset=[
            "expert_novelty_to_centroid",
            "uptake_closure",
        ]
    )
)

g = sns.lmplot(
    data=novelty_uptake,
    x="expert_novelty_to_centroid",
    y="uptake_closure",
    hue="level_label",
    hue_order=level_order,
    col="window_label",
    col_order=[
        "Beginning",
        "Middle",
        "End",
    ],
    palette=level_palette,
    scatter_kws={
        "alpha": 0.25,
        "s": 25,
    },
    line_kws={
        "linewidth": 2,
    },
    ci=None,
    height=4,
    aspect=1,
)

for ax in g.axes.flat:
    ax.axhline(
        0,
        color="black",
        linestyle="--",
    )

g.set_axis_labels(
    "Novelty relative to partner’s prior horizon",
    "Partner uptake closure",
)

g.fig.suptitle(
    "Adaptation to expert-introduced novelty",
    y=1.05,
)

plt.show()

In [ ]:
novelty_uptake = (
    partner_adaptation_events
    .dropna(
        subset=[
            "expert_novelty_to_centroid",
            "uptake_closure",
        ]
    )
)

g = sns.lmplot(
    data=novelty_uptake,
    x="expert_novelty_to_centroid",
    y="uptake_closure",
    hue="level_label",
    hue_order=level_order,
    col="window_label",
    col_order=[
        "Beginning",
        "Middle",
        "End",
    ],
    palette=level_palette,
    scatter_kws={
        "alpha": 0.25,
        "s": 25,
    },
    line_kws={
        "linewidth": 2,
    },
    ci=None,
    height=4,
    aspect=1,
)

for ax in g.axes.flat:
    ax.axhline(
        0,
        color="black",
        linestyle="--",
    )

g.set_axis_labels(
    "Novelty relative to partner’s prior horizon",
    "Partner uptake closure",
)

g.fig.suptitle(
    "Adaptation to expert-introduced novelty",
    y=1.05,
)

plt.show()

In [ ]:
adaptation_window = (
    partner_adaptation_events
    .groupby(
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level_label",
            "window_index",
            "window_label",
        ],
        observed=True,
    )
    .agg(
        n_adaptation_events=(
            "uptake_closure",
            "count",
        ),

        expert_novelty_mean=(
            "expert_novelty_to_centroid",
            "mean",
        ),

        uptake_closure_mean=(
            "uptake_closure",
            "mean",
        ),

        directional_adaptation_mean=(
            "directional_adaptation",
            "mean",
        ),

        retained_uptake_mean=(
            "retained_uptake_closure",
            "mean",
        ),

        forward_alignment_mean=(
            "forward_alignment_excess",
            "mean",
        ),
    )
    .reset_index()
)

In [ ]:
anticipation_df = (
    partner_adaptation_events
    .dropna(
        subset=[
            "partner_distance_after",
            "next_expert_similarity",
        ]
    )
    .copy()
)

anticipation_df[
    "current_expert_similarity"
] = (
    1
    - anticipation_df[
        "partner_distance_after"
    ]
)

g = sns.relplot(
    data=anticipation_df,
    x="current_expert_similarity",
    y="next_expert_similarity",
    hue="level_label",
    hue_order=level_order,
    col="window_label",
    col_order=[
        "Beginning",
        "Middle",
        "End",
    ],
    palette=level_palette,
    alpha=0.4,
    height=4,
)

for ax in g.axes.flat:
    lower = min(
        ax.get_xlim()[0],
        ax.get_ylim()[0],
    )

    upper = max(
        ax.get_xlim()[1],
        ax.get_ylim()[1],
    )

    ax.plot(
        [lower, upper],
        [lower, upper],
        color="black",
        linestyle="--",
    )

g.set_axis_labels(
    "Alignment with current expert turn",
    "Alignment with next expert turn",
)

g.fig.suptitle(
    "Forward alignment of partner responses",
    y=1.05,
)

plt.show()

## adaptation smoothness

In [ ]:
smoothness_records = []

turn_lookup = (
    turns.set_index([
        "dataset",
        "conversation_id",
        "turn_order",
    ])
)

for event_index, event in (
    partner_adaptation_events.iterrows()
):
    dataset = event["dataset"]
    conversation_id = event[
        "conversation_id"
    ]

    expert_turn_order = int(
        event["expert_turn_order"]
    )

    response_turn_order = int(
        event[
            "partner_response_turn_order"
        ]
    )

    conversation_turns = (
        turns[
            (
                turns["dataset"]
                == dataset
            )
            &
            (
                turns["conversation_id"]
                == conversation_id
            )
        ]
        .sort_values("turn_order")
    )

    previous_partner_turns = (
        conversation_turns[
            (
                conversation_turns[
                    "speaker_role"
                ] == "partner"
            )
            &
            (
                conversation_turns[
                    "turn_order"
                ] < response_turn_order
            )
        ]
    )

    # The first partner response has no
    # previous partner position.
    if len(previous_partner_turns) == 0:
        smoothness_records.append({
            "event_index": event_index,
            "n_response_utterances": np.nan,
            "response_path_net_uptake": np.nan,
            "response_path_total_adjustment": np.nan,
            "signed_adaptation_fluency": np.nan,
            "monotonic_progress_fraction": np.nan,
            "radial_detour": np.nan,
        })

        continue

    previous_partner_turn = (
        previous_partner_turns.iloc[-1]
    )

    expert_turn = turn_lookup.loc[
        (
            dataset,
            conversation_id,
            expert_turn_order,
        )
    ]

    response_turn = turn_lookup.loc[
        (
            dataset,
            conversation_id,
            response_turn_order,
        )
    ]

    previous_partner_embedding = (
        E_turns[
            int(
                previous_partner_turn[
                    "turn_embedding_idx"
                ]
            )
        ]
    )

    expert_embedding = (
        E_turns[
            int(
                expert_turn[
                    "turn_embedding_idx"
                ]
            )
        ]
    )

    # Recover the individual utterances
    # contained in the response turn.
    response_first_idx = int(
        response_turn[
            "first_embedding_idx"
        ]
    )

    response_last_idx = int(
        response_turn[
            "last_embedding_idx"
        ]
    )

    response_rows = (
        stage3_source[
            (
                stage3_source["dataset"]
                == dataset
            )
            &
            (
                stage3_source[
                    "conversation_id"
                ] == conversation_id
            )
            &
            (
                stage3_source[
                    "embedding_idx"
                ].between(
                    response_first_idx,
                    response_last_idx,
                )
            )
        ]
        .sort_values("embedding_idx")
    )

    response_embedding_indices = (
        response_rows[
            "embedding_idx"
        ].to_numpy(dtype=int)
    )

    response_utterance_embeddings = (
        E_utterances[
            response_embedding_indices
        ]
    )

    n_response_utterances = len(
        response_utterance_embeddings
    )

    # Start at the partner's previous turn,
    # then follow each response utterance.
    response_path = np.vstack([
        previous_partner_embedding,
        response_utterance_embeddings,
    ])

    response_path = (
        response_path
        / np.linalg.norm(
            response_path,
            axis=1,
            keepdims=True,
        )
    )

    distances_to_expert = np.clip(
        1.0
        - response_path
        @ expert_embedding,
        0.0,
        2.0,
    )

    # Positive progress = moved closer.
    progress_steps = (
        distances_to_expert[:-1]
        - distances_to_expert[1:]
    )

    net_uptake = float(
        distances_to_expert[0]
        - distances_to_expert[-1]
    )

    total_adjustment = float(
        np.sum(
            np.abs(progress_steps)
        )
    )

    if total_adjustment > 1e-12:
        signed_fluency = float(
            net_uptake
            / total_adjustment
        )
    else:
        signed_fluency = np.nan

    monotonic_progress_fraction = float(
        np.mean(
            progress_steps > 0
        )
    )

    radial_detour = float(
        total_adjustment
        - abs(net_uptake)
    )

    # A single utterance can show uptake,
    # but not meaningful internal smoothness.
    if n_response_utterances < 2:
        signed_fluency_for_analysis = np.nan
        monotonic_fraction_for_analysis = np.nan
        radial_detour_for_analysis = np.nan
    else:
        signed_fluency_for_analysis = (
            signed_fluency
        )

        monotonic_fraction_for_analysis = (
            monotonic_progress_fraction
        )

        radial_detour_for_analysis = (
            radial_detour
        )

    smoothness_records.append({
        "event_index": event_index,

        "n_response_utterances":
            n_response_utterances,

        "response_path_net_uptake":
            net_uptake,

        "response_path_total_adjustment":
            total_adjustment,

        "signed_adaptation_fluency":
            signed_fluency_for_analysis,

        "monotonic_progress_fraction":
            monotonic_fraction_for_analysis,

        "radial_detour":
            radial_detour_for_analysis,

        # Optional: retain for individual
        # trajectory inspection.
        "distance_path_to_expert":
            distances_to_expert,
    })

In [ ]:
smoothness_df = pd.DataFrame(
    smoothness_records
).set_index("event_index")

partner_adaptation_events = (
    partner_adaptation_events.join(
        smoothness_df
    )
)

In [ ]:
display(
    partner_adaptation_events[
        [
            "n_response_utterances",
            "uptake_closure",
            "signed_adaptation_fluency",
            "monotonic_progress_fraction",
            "radial_detour",
        ]
    ].describe()
)

assert (
    partner_adaptation_events[
        "signed_adaptation_fluency"
    ]
    .dropna()
    .between(-1, 1)
    .all()
)

In [ ]:
coverage_by_level = (
    partner_adaptation_events
    .groupby(
        "level_label",
        observed=True,
    )
    .agg(
        total_events=(
            "conversation_id",
            "size",
        ),

        multi_utterance_responses=(
            "signed_adaptation_fluency",
            "count",
        ),
    )
)

coverage_by_level[
    "coverage_percent"
] = (
    100
    * coverage_by_level[
        "multi_utterance_responses"
    ]
    / coverage_by_level[
        "total_events"
    ]
)

display(coverage_by_level)

In [ ]:
fluency_plot_df = (
    partner_adaptation_events
    .dropna(
        subset=[
            "uptake_closure",
            "signed_adaptation_fluency",
            "expert_novelty_to_centroid",
        ]
    )
)

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=fluency_plot_df,
    x="uptake_closure",
    y="signed_adaptation_fluency",
    hue="level_label",
    hue_order=level_order,
    palette=level_palette,
    size="expert_novelty_to_centroid",
    sizes=(20, 140),
    alpha=0.45,
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
)

plt.axhline(
    0,
    color="black",
    linestyle="--",
)

plt.xlabel(
    "Partner uptake of expert information"
)

plt.ylabel(
    "Signed adaptation fluency"
)

plt.title(
    "Success and smoothness of adaptation to novelty"
)

plt.tight_layout()
plt.show()

## efficient movement

In [ ]:
import numpy as np
import pandas as pd

from scipy.spatial.distance import pdist
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
CONVERSATION_COL = "conversation_id"

VIDEO_COL = (
    "video_id"
    if "video_id" in turns.columns
    else "dataset"
)

LEVEL_COL = "level_label"
SPEAKER_COL = "speaker_role"

possible_order_columns = [
    "turn_index",
    "turn_idx",
    "turn_number",
    "turn_id",
    "first_embedding_idx",
    "start_embedding_idx",
]

ORDER_COL = next(
    (
        col for col in possible_order_columns
        if col in turns.columns
    ),
    None,
)

print("Video column:", VIDEO_COL)
print("Turn-order column:", ORDER_COL)

In [ ]:
turns_refine = turns.copy().reset_index(drop=True)

X_turns = np.asarray(
    E_turns,
    dtype=float,
).copy()

assert len(turns_refine) == len(X_turns), (
    f"{len(turns_refine)} turn rows but "
    f"{len(X_turns)} embeddings"
)

norms = np.linalg.norm(
    X_turns,
    axis=1,
    keepdims=True,
)

if np.any(norms < 1e-12):
    raise ValueError(
        "At least one turn embedding has approximately zero norm."
    )

X_turns = X_turns / norms

# Preserve the mapping to the embedding matrix
turns_refine["_embedding_row"] = np.arange(
    len(turns_refine)
)

In [ ]:
turns_refine["_role"] = (
    turns_refine[SPEAKER_COL]
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "learner": "partner",
        "novice": "partner",
    })
)

print(
    turns_refine["_role"]
    .value_counts(dropna=False)
)

In [ ]:
turns_refine["video_uid"] = (
    turns_refine[VIDEO_COL].astype(str)
)

turns_refine["conversation_uid"] = (
    turns_refine["video_uid"]
    + "::"
    + turns_refine[CONVERSATION_COL].astype(str)
)

In [ ]:
if ORDER_COL is None:
    # Assumes turns are already in chronological order
    turns_refine["_turn_order"] = (
        turns_refine
        .groupby(
            "conversation_uid",
            sort=False,
        )
        .cumcount()
    )
else:
    turns_refine["_turn_order"] = (
        turns_refine[ORDER_COL]
    )

turns_refine = (
    turns_refine
    .sort_values(
        [
            "video_uid",
            "conversation_uid",
            "_turn_order",
        ]
    )
    .reset_index(drop=True)
)

turns_refine["_conversation_position"] = (
    turns_refine
    .groupby("conversation_uid")
    .cumcount()
)

turns_refine["_conversation_n_turns"] = (
    turns_refine
    .groupby("conversation_uid")[
        "_conversation_position"
    ]
    .transform("size")
)

denominator = (
    turns_refine["_conversation_n_turns"] - 1
)

turns_refine["_conversation_time"] = np.where(
    denominator > 0,
    (
        turns_refine["_conversation_position"]
        / denominator
    ),
    0.5,
)

In [ ]:
EPSILON = 1e-12


def vector_cosine(a, b):
    """Cosine between two displacement vectors."""

    denominator = (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

    if denominator < EPSILON:
        return np.nan

    value = np.dot(a, b) / denominator

    return float(
        np.clip(value, -1.0, 1.0)
    )


def unit_vector(x):
    norm = np.linalg.norm(x)

    if norm < EPSILON:
        return None

    return x / norm


def assign_stage(time_value):
    if time_value < 1 / 3:
        return "Beginning"

    if time_value < 2 / 3:
        return "Middle"

    return "End"

In [ ]:
def calculate_local_refinement(points):
    """
    Calculate trajectory-refinement metrics for an ordered
    sequence of unit-normalized embeddings.
    """

    points = np.asarray(
        points,
        dtype=float,
    )

    if len(points) < 3:
        raise ValueError(
            "At least three points are required."
        )

    # -----------------------------------------
    # 1. Step vectors and distances
    # -----------------------------------------

    step_vectors = np.diff(
        points,
        axis=0,
    )

    step_distances = np.linalg.norm(
        step_vectors,
        axis=1,
    )

    path_length = step_distances.sum()

    direct_displacement = np.linalg.norm(
        points[-1] - points[0]
    )

    if path_length > EPSILON:
        route_efficiency = (
            direct_displacement
            / path_length
        )

        route_efficiency = float(
            np.clip(
                route_efficiency,
                0.0,
                1.0,
            )
        )
    else:
        route_efficiency = np.nan

    # -----------------------------------------
    # 2. Directional persistence
    # -----------------------------------------

    directional_cosines = [
        vector_cosine(
            step_vectors[i],
            step_vectors[i + 1],
        )
        for i in range(
            len(step_vectors) - 1
        )
    ]

    directional_cosines = np.asarray(
        directional_cosines,
        dtype=float,
    )

    directional_persistence = (
        np.nanmean(directional_cosines)
        if np.any(
            np.isfinite(directional_cosines)
        )
        else np.nan
    )

    turning_cost = (
        (1 - directional_persistence) / 2
        if np.isfinite(
            directional_persistence
        )
        else np.nan
    )

    # -----------------------------------------
    # 3. Backtracking relative to endpoint
    # -----------------------------------------

    endpoint = points[-1]

    distance_to_endpoint = np.linalg.norm(
        points - endpoint,
        axis=1,
    )

    progress = (
        distance_to_endpoint[:-1]
        - distance_to_endpoint[1:]
    )

    backward_movement = np.maximum(
        -progress,
        0,
    ).sum()

    if path_length > EPSILON:
        backtracking_fraction = (
            backward_movement
            / path_length
        )
    else:
        backtracking_fraction = np.nan

    monotonic_progress_fraction = np.mean(
        progress >= -1e-10
    )

    excess_path_fraction = (
        1 - route_efficiency
        if np.isfinite(route_efficiency)
        else np.nan
    )

    # -----------------------------------------
    # 4. Future-horizon alignment
    # -----------------------------------------

    future_alignments = []

    # At each position, exclude the immediate next
    # point from the future centroid.
    for i in range(len(points) - 2):
        current = points[i]
        next_point = points[i + 1]

        future_centroid = points[
            i + 2:
        ].mean(axis=0)

        future_direction = unit_vector(
            future_centroid
        )

        if future_direction is None:
            continue

        current_movement = (
            next_point - current
        )

        direction_to_future = (
            future_direction - current
        )

        alignment = vector_cosine(
            current_movement,
            direction_to_future,
        )

        future_alignments.append(
            alignment
        )

    future_alignment = (
        np.nanmean(future_alignments)
        if len(future_alignments) > 0
        else np.nan
    )

    # -----------------------------------------
    # 5. Local semantic coverage
    # -----------------------------------------

    local_centroid = points.mean(axis=0)

    centroid_dispersion = np.mean(
        np.linalg.norm(
            points - local_centroid,
            axis=1,
        )
    )

    resultant_length = np.linalg.norm(
        local_centroid
    )

    pairwise_coverage = (
        pdist(
            points,
            metric="euclidean",
        ).mean()
    )

    # -----------------------------------------
    # 6. Pace and acceleration
    # -----------------------------------------

    mean_step_distance = (
        step_distances.mean()
    )

    step_distance_sd = (
        step_distances.std(ddof=0)
    )

    step_distance_cv = (
        step_distance_sd
        / mean_step_distance
        if mean_step_distance > EPSILON
        else np.nan
    )

    acceleration_vectors = np.diff(
        step_vectors,
        axis=0,
    )

    mean_acceleration = np.mean(
        np.linalg.norm(
            acceleration_vectors,
            axis=1,
        )
    )

    normalized_acceleration = (
        mean_acceleration
        / mean_step_distance
        if mean_step_distance > EPSILON
        else np.nan
    )

    coverage_per_path = (
        pairwise_coverage
        / path_length
        if path_length > EPSILON
        else np.nan
    )

    return {
        "path_length": path_length,
        "direct_displacement": direct_displacement,
        "route_efficiency": route_efficiency,
        "excess_path_fraction": excess_path_fraction,

        "directional_persistence": directional_persistence,
        "turning_cost": turning_cost,

        "backward_movement": backward_movement,
        "backtracking_fraction": backtracking_fraction,
        "monotonic_progress_fraction": (
            monotonic_progress_fraction
        ),

        "future_alignment": future_alignment,

        "pairwise_coverage": pairwise_coverage,
        "centroid_dispersion": centroid_dispersion,
        "resultant_length": resultant_length,
        "coverage_per_path": coverage_per_path,

        "mean_step_distance": mean_step_distance,
        "step_distance_sd": step_distance_sd,
        "step_distance_cv": step_distance_cv,
        "normalized_acceleration": (
            normalized_acceleration
        ),
    }

In [ ]:
trajectory_roles = {
    "joint": None,
    "expert": "expert",
    "partner": "partner",
}

HORIZONS = [3, 5]

local_results = []

for conversation_uid, conversation in (
    turns_refine
    .groupby(
        "conversation_uid",
        sort=False,
    )
):
    conversation = (
        conversation
        .sort_values("_turn_order")
        .reset_index(drop=True)
    )

    for trajectory_type, role in (
        trajectory_roles.items()
    ):
        if role is None:
            trajectory = conversation.copy()
        else:
            trajectory = (
                conversation[
                    conversation["_role"] == role
                ]
                .copy()
                .reset_index(drop=True)
            )

        for horizon in HORIZONS:
            required_points = horizon + 1

            if len(trajectory) < required_points:
                continue

            for start in range(
                len(trajectory) - horizon
            ):
                stop = start + required_points

                local_window = trajectory.iloc[
                    start:stop
                ]

                embedding_rows = (
                    local_window[
                        "_embedding_row"
                    ]
                    .to_numpy(dtype=int)
                )

                points = X_turns[
                    embedding_rows
                ]

                metrics = (
                    calculate_local_refinement(
                        points
                    )
                )

                midpoint_time = (
                    local_window[
                        "_conversation_time"
                    ]
                    .mean()
                )

                result = {
                    "video_uid": (
                        local_window[
                            "video_uid"
                        ].iloc[0]
                    ),
                    "conversation_uid": (
                        conversation_uid
                    ),
                    "conversation_id": (
                        local_window[
                            CONVERSATION_COL
                        ].iloc[0]
                    ),
                    "level_label": (
                        local_window[
                            LEVEL_COL
                        ].iloc[0]
                    ),
                    "trajectory_type": (
                        trajectory_type
                    ),
                    "horizon": horizon,
                    "local_start": start,
                    "local_end": (
                        start + horizon
                    ),
                    "midpoint_time": (
                        midpoint_time
                    ),
                    "stage": assign_stage(
                        midpoint_time
                    ),
                    "n_trajectory_turns": (
                        len(trajectory)
                    ),
                }

                result.update(metrics)
                local_results.append(result)

trajectory_refinement_local = pd.DataFrame(
    local_results
)

print(
    "Local trajectory windows:",
    len(trajectory_refinement_local),
)

trajectory_refinement_local.head()

In [ ]:
trajectory_refinement_local[
    [
        "route_efficiency",
        "directional_persistence",
        "backtracking_fraction",
        "future_alignment",
    ]
].describe()

In [ ]:
assert (
    trajectory_refinement_local[
        "route_efficiency"
    ]
    .dropna()
    .between(0, 1)
    .all()
)

In [ ]:
coverage_summary = (
    trajectory_refinement_local
    .groupby(
        [
            "trajectory_type",
            "horizon",
        ]
    )
    .agg(
        local_windows=(
            "route_efficiency",
            "size",
        ),
        conversations=(
            "conversation_uid",
            "nunique",
        ),
        videos=(
            "video_uid",
            "nunique",
        ),
    )
    .reset_index()
)

coverage_summary

In [ ]:
stage_coverage = (
    trajectory_refinement_local
    .groupby(
        [
            "trajectory_type",
            "horizon",
            "stage",
        ],
        observed=True,
    )
    .agg(
        local_windows=(
            "route_efficiency",
            "size",
        ),
        conversations=(
            "conversation_uid",
            "nunique",
        ),
    )
    .reset_index()
)

stage_coverage

In [ ]:
metric_columns = [
    "path_length",
    "direct_displacement",
    "route_efficiency",
    "excess_path_fraction",
    "directional_persistence",
    "turning_cost",
    "backward_movement",
    "backtracking_fraction",
    "monotonic_progress_fraction",
    "future_alignment",
    "pairwise_coverage",
    "centroid_dispersion",
    "resultant_length",
    "coverage_per_path",
    "mean_step_distance",
    "step_distance_sd",
    "step_distance_cv",
    "normalized_acceleration",
]

group_columns = [
    "video_uid",
    "conversation_uid",
    "conversation_id",
    "level_label",
    "trajectory_type",
    "horizon",
    "stage",
]

refinement_stage_metrics = (
    trajectory_refinement_local
    .groupby(
        group_columns,
        observed=True,
    )[metric_columns]
    .mean()
    .reset_index()
)

window_counts = (
    trajectory_refinement_local
    .groupby(
        group_columns,
        observed=True,
    )
    .size()
    .rename("n_local_windows")
    .reset_index()
)

refinement_stage_metrics = (
    refinement_stage_metrics
    .merge(
        window_counts,
        on=group_columns,
        how="left",
        validate="one_to_one",
    )
)

print(
    refinement_stage_metrics.shape
)

refinement_stage_metrics.head()

In [ ]:
primary_refinement = (
    refinement_stage_metrics[
        (
            refinement_stage_metrics[
                "trajectory_type"
            ] == "joint"
        )
        &
        (
            refinement_stage_metrics[
                "horizon"
            ] == 3
        )
    ]
    .copy()
)

In [ ]:
complete_conversations = (
    primary_refinement
    .groupby("conversation_uid")[
        "stage"
    ]
    .nunique()
)

complete_conversations = (
    complete_conversations[
        complete_conversations == 3
    ]
    .index
)

primary_refinement = (
    primary_refinement[
        primary_refinement[
            "conversation_uid"
        ].isin(complete_conversations)
    ]
    .copy()
)

print(
    "Complete conversations:",
    primary_refinement[
        "conversation_uid"
    ].nunique(),
)

print(
    "Rows:",
    len(primary_refinement),
)

In [ ]:
gap_mapping = {
    "expert": 0,
    "graduate": 1,
    "grad": 1,
    "graduate student": 1,
    "undergraduate": 2,
    "college": 2,
    "college student": 2,
    "teenager": 3,
    "teen": 3,
    "child": 4,
}

primary_refinement["level_key"] = (
    primary_refinement[
        "level_label"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
)

primary_refinement["gap_score"] = (
    primary_refinement[
        "level_key"
    ]
    .map(gap_mapping)
)

if primary_refinement[
    "gap_score"
].isna().any():
    print(
        "Unmapped labels:",
        primary_refinement.loc[
            primary_refinement[
                "gap_score"
            ].isna(),
            "level_label",
        ].unique()
    )

In [ ]:
time_mapping = {
    "Beginning": -0.5,
    "Middle": 0.0,
    "End": 0.5,
}

primary_refinement["time_c"] = (
    primary_refinement[
        "stage"
    ]
    .map(time_mapping)
    .astype(float)
)

primary_refinement["gap_c"] = (
    primary_refinement["gap_score"]
    - primary_refinement[
        "gap_score"
    ].mean()
)

In [ ]:
def standardize(series):
    sd = series.std(ddof=0)

    if sd < EPSILON:
        raise ValueError(
            f"{series.name} has zero variance."
        )

    return (
        series - series.mean()
    ) / sd

In [ ]:
def fit_refinement_model(
    data,
    outcome,
    adjust_for_coverage=False,
):
    required = [
        outcome,
        "gap_c",
        "time_c",
        "video_uid",
        "conversation_uid",
    ]

    if adjust_for_coverage:
        required.append(
            "pairwise_coverage"
        )

    model_data = (
        data
        .dropna(subset=required)
        .copy()
    )

    outcome_z = f"{outcome}_z"

    model_data[outcome_z] = standardize(
        model_data[outcome]
    )

    formula = (
        f"{outcome_z} ~ gap_c * time_c"
    )

    if adjust_for_coverage:
        model_data["coverage_z"] = (
            standardize(
                model_data[
                    "pairwise_coverage"
                ]
            )
        )

        formula += " + coverage_z"

    model = smf.mixedlm(
        formula=formula,
        data=model_data,
        groups=model_data[
            "video_uid"
        ],
        re_formula="1",
        vc_formula={
            "conversation": (
                "0 + C(conversation_uid)"
            )
        },
    )

    result = model.fit(
        reml=False,
        method="lbfgs",
        maxiter=2000,
    )

    return result, model_data

In [ ]:
route_model, route_data = (
    fit_refinement_model(
        primary_refinement,
        outcome="route_efficiency",
        adjust_for_coverage=False,
    )
)

print(route_model.summary())

In [ ]:
primary_outcomes = [
    "route_efficiency",
    "directional_persistence",
    "backtracking_fraction",
    "future_alignment",
    "normalized_acceleration",
]

model_results = {}
result_rows = []

for outcome in primary_outcomes:
    result, _ = fit_refinement_model(
        primary_refinement,
        outcome=outcome,
        adjust_for_coverage=True,
    )

    model_results[outcome] = result

    for term in [
        "gap_c",
        "time_c",
        "gap_c:time_c",
    ]:
        result_rows.append({
            "metric": outcome,
            "term": term,
            "beta_standardized": (
                result.params.get(
                    term,
                    np.nan,
                )
            ),
            "se": result.bse.get(
                term,
                np.nan,
            ),
            "p_value": result.pvalues.get(
                term,
                np.nan,
            ),
            "converged": result.converged,
        })

refinement_model_summary = pd.DataFrame(
    result_rows
)

In [ ]:
refinement_model_summary["p_fdr_bh"] = np.nan

for term, indices in (
    refinement_model_summary
    .groupby("term")
    .groups
    .items()
):
    p_values = (
        refinement_model_summary
        .loc[indices, "p_value"]
    )

    valid = p_values.notna()

    if valid.any():
        adjusted = multipletests(
            p_values[valid],
            method="fdr_bh",
        )[1]

        refinement_model_summary.loc[
            p_values[valid].index,
            "p_fdr_bh",
        ] = adjusted

refinement_model_summary[
    "significant_fdr_05"
] = (
    refinement_model_summary[
        "p_fdr_bh"
    ] < 0.05
)

refinement_model_summary

In [ ]:
path_cost_data = primary_refinement.copy()

path_cost_data["log_path_length"] = np.log(
    path_cost_data["path_length"]
)

path_cost_data["log_path_z"] = standardize(
    path_cost_data["log_path_length"]
)

path_cost_data["coverage_z"] = standardize(
    path_cost_data["pairwise_coverage"]
)

path_cost_model = smf.mixedlm(
    formula=(
        "log_path_z ~ "
        "coverage_z * gap_c "
        "+ time_c "
        "+ gap_c:time_c"
    ),
    data=path_cost_data,
    groups=path_cost_data["video_uid"],
    re_formula="1",
    vc_formula={
        "conversation": (
            "0 + C(conversation_uid)"
        )
    },
)

path_cost_result = path_cost_model.fit(
    reml=False,
    method="lbfgs",
    maxiter=2000,
)

print(path_cost_result.summary())

In [ ]:
stage_order = [
    "Beginning",
    "Middle",
    "End",
]

level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert",
]

plot_data = primary_refinement.copy()

plot_data["stage"] = pd.Categorical(
    plot_data["stage"],
    categories=stage_order,
    ordered=True,
)

plt.figure(figsize=(9, 5))

sns.lineplot(
    data=plot_data,
    x="stage",
    y="route_efficiency",
    hue="level_key",
    hue_order=level_order,
    estimator="mean",
    errorbar=("ci", 95),
    marker="o",
)

plt.axhline(
    0,
    color="gray",
    linewidth=1,
    alpha=0.5,
)

plt.xlabel("Conversation stage")
plt.ylabel("Local semantic route efficiency")
plt.title(
    "Development of semantic route efficiency"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")

stage_order = [
    "Beginning",
    "Middle",
    "End",
]

level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert",
]

palette = dict(zip(
    level_order,
    sns.color_palette(
        "viridis",
        n_colors=len(level_order),
    )
))

plot_data = primary_refinement.copy()

plot_data["stage"] = pd.Categorical(
    plot_data["stage"],
    categories=stage_order,
    ordered=True,
)

plot_data["level_key"] = (
    plot_data["level_label"]
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "teen": "teenager",
        "college": "undergraduate",
        "college student": "undergraduate",
        "graduate student": "graduate",
    })
)

In [ ]:
g = sns.lmplot(
    data=plot_data,
    x="pairwise_coverage",
    y="route_efficiency",
    hue="level_key",
    hue_order=level_order,
    palette=palette,
    col="stage",
    col_order=stage_order,
    height=4,
    aspect=0.85,
    ci=95,
    scatter_kws={
        "alpha": 0.45,
        "s": 35,
    },
    line_kws={
        "linewidth": 2,
    },
)

g.set_axis_labels(
    "Local semantic coverage",
    "Local route efficiency",
)

g.set_titles("{col_name}")

g.figure.suptitle(
    "Semantic breadth and trajectory refinement",
    y=1.05,
)

plt.show()

In [ ]:
g = sns.lmplot(
    data=plot_data,
    x="pairwise_coverage",
    y="path_length",
    hue="level_key",
    hue_order=level_order,
    palette=palette,
    col="stage",
    col_order=stage_order,
    height=4,
    aspect=0.85,
    ci=95,
    scatter_kws={
        "alpha": 0.4,
        "s": 30,
    },
    line_kws={
        "linewidth": 2,
    },
)

g.set_axis_labels(
    "Local semantic coverage",
    "Total path length",
)

g.set_titles("{col_name}")

g.figure.suptitle(
    "Semantic movement cost relative to coverage",
    y=1.05,
)

plt.show()

In [ ]:
refinement_directions = {
    # Higher raw values indicate greater refinement
    "route_efficiency": 1,
    "directional_persistence": 1,
    "future_alignment": 1,

    # Lower raw values indicate greater refinement
    "backtracking_fraction": -1,
    "normalized_acceleration": -1,
}

metric_labels = {
    "route_efficiency": "Route efficiency",
    "directional_persistence": "Directional persistence",
    "future_alignment": "Future-horizon alignment",
    "backtracking_fraction": "Low backtracking",
    "normalized_acceleration": "Low trajectory acceleration",
}

long_frames = []

for metric, direction in refinement_directions.items():
    temporary = plot_data[
        [
            "conversation_uid",
            "video_uid",
            "level_key",
            "stage",
            metric,
        ]
    ].dropna().copy()

    mean = temporary[metric].mean()
    sd = temporary[metric].std(ddof=0)

    temporary["refinement_z"] = (
        direction
        * (temporary[metric] - mean)
        / sd
    )

    temporary["metric"] = metric_labels[
        metric
    ]

    long_frames.append(temporary)

refinement_long = pd.concat(
    long_frames,
    ignore_index=True,
)

In [ ]:
g = sns.relplot(
    data=refinement_long,
    x="stage",
    y="refinement_z",
    hue="level_key",
    hue_order=level_order,
    palette=palette,
    col="metric",
    col_wrap=3,
    kind="line",
    estimator="mean",
    errorbar=("ci", 95),
    marker="o",
    height=3.4,
    aspect=1.1,
    facet_kws={
        "sharey": True,
    },
)

for ax in g.axes.flat:
    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1,
        alpha=0.7,
    )

g.set_axis_labels(
    "Conversation stage",
    "Refinement-oriented standardized score",
)

g.set_titles("{col_name}")

g.figure.suptitle(
    "Development of trajectory-refinement components",
    y=1.03,
)

plt.show()

In [ ]:
continuous = (
    trajectory_refinement_local[
        (
            trajectory_refinement_local[
                "trajectory_type"
            ] == "joint"
        )
        &
        (
            trajectory_refinement_local[
                "horizon"
            ] == 3
        )
    ]
    .copy()
)

continuous["level_key"] = (
    continuous["level_label"]
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "teen": "teenager",
        "college": "undergraduate",
        "college student": "undergraduate",
        "graduate student": "graduate",
    })
)

continuous["time_bin"] = pd.cut(
    continuous["midpoint_time"],
    bins=np.linspace(0, 1, 11),
    labels=False,
    include_lowest=True,
)

continuous["time_bin_midpoint"] = (
    continuous["time_bin"] + 0.5
) / 10

In [ ]:
continuous_summary = (
    continuous
    .groupby(
        [
            "conversation_uid",
            "video_uid",
            "level_key",
            "time_bin_midpoint",
        ],
        observed=True,
    )["route_efficiency"]
    .mean()
    .reset_index()
)

In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(
    data=continuous_summary,
    x="time_bin_midpoint",
    y="route_efficiency",
    hue="level_key",
    hue_order=level_order,
    palette=palette,
    estimator="mean",
    errorbar=("ci", 95),
    marker="o",
)

plt.xlabel("Normalized conversation time")
plt.ylabel("Local route efficiency")
plt.title(
    "Semantic route efficiency across conversational time"
)

plt.xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
coefficient_directions = {
    "route_efficiency": 1,
    "directional_persistence": 1,
    "future_alignment": 1,
    "backtracking_fraction": -1,
    "normalized_acceleration": -1,
}

forest = (
    refinement_model_summary[
        refinement_model_summary[
            "term"
        ] == "gap_c"
    ]
    .copy()
)

forest["direction"] = (
    forest["metric"]
    .map(coefficient_directions)
)

# Original beta describes increasing gap.
# Negating it converts it to movement toward expert.
forest["expert_advantage"] = (
    -forest["direction"]
    * forest["beta_standardized"]
)

forest["ci_lower"] = (
    forest["expert_advantage"]
    - 1.96 * forest["se"]
)

forest["ci_upper"] = (
    forest["expert_advantage"]
    + 1.96 * forest["se"]
)

forest["metric_label"] = (
    forest["metric"]
    .map(metric_labels)
)

forest = forest.sort_values(
    "expert_advantage"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

y = np.arange(len(forest))

ax.errorbar(
    forest["expert_advantage"],
    y,
    xerr=[
        (
            forest["expert_advantage"]
            - forest["ci_lower"]
        ),
        (
            forest["ci_upper"]
            - forest["expert_advantage"]
        ),
    ],
    fmt="o",
    color="black",
    ecolor="gray",
    capsize=4,
)

ax.axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1,
)

ax.set_yticks(y)
ax.set_yticklabels(
    forest["metric_label"]
)

ax.set_xlabel(
    "Standardized effect toward the expert end\n"
    "Positive = greater trajectory refinement"
)

ax.set_title(
    "Expertise differences in semantic trajectory refinement"
)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_turns_pca = pca.fit_transform(X_turns)

trajectory_plot = turns_refine.copy()

trajectory_plot["pc1"] = X_turns_pca[
    trajectory_plot["_embedding_row"],
    0,
]

trajectory_plot["pc2"] = X_turns_pca[
    trajectory_plot["_embedding_row"],
    1,
]

In [ ]:
print(
    trajectory_plot[
        "video_uid"
    ].unique()
)

CHOSEN_VIDEO = 'gravity'

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
)

for ax, level in zip(
    axes,
    ["expert", "child"],
):
    subset = trajectory_plot[
        (
            trajectory_plot[
                "video_uid"
            ] == CHOSEN_VIDEO
        )
        &
        (
            trajectory_plot[
                LEVEL_COL
            ].str.lower() == level
        )
    ].sort_values("_turn_order")

    x = subset["pc1"].to_numpy()
    y = subset["pc2"].to_numpy()

    ax.plot(
        x,
        y,
        color="gray",
        alpha=0.6,
        linewidth=1,
    )

    points = ax.scatter(
        x,
        y,
        c=subset[
            "_conversation_time"
        ],
        cmap="viridis",
        s=35,
    )

    ax.scatter(
        x[0],
        y[0],
        marker="s",
        s=100,
        color="black",
        label="Start",
    )

    ax.scatter(
        x[-1],
        y[-1],
        marker="X",
        s=100,
        color="black",
        label="End",
    )

    ax.set_title(level.title())
    ax.set_xlabel("Global PCA 1")
    ax.set_ylabel("Global PCA 2")
    ax.legend()

fig.colorbar(
    points,
    ax=axes,
    label="Conversation time",
)

fig.suptitle(
    f"Matched semantic trajectories: {CHOSEN_VIDEO}"
)

plt.show()

# multilevel modeling for each metric

model each metric using a mixed effect(multilevel) model. (https://pmc.ncbi.nlm.nih.gov/articles/PMC10171296/)


gap scores: 12->4 13->3 14->2 15->1 16->0

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

analysis_df = conversation_geometry.copy()

gap_map = {
    "expert": 0,
    "graduate": 1,
    "undergraduate": 2,
    "teenager": 3,
    "child": 4,
}

analysis_df["gap_score"] = (
    analysis_df["level_label"]
    .map(gap_map)
    .astype(float)
)

# Centering makes the intercept represent the middle expertise condition.
analysis_df["gap_c"] = (
    analysis_df["gap_score"]
    - analysis_df["gap_score"].mean()
)

# Use a composite identifier in case video_id is repeated across datasets.
analysis_df["video_group"] = (
    analysis_df["dataset"].astype(str)
    + "::"
    + analysis_df["video_id"].astype(str)
)

# Length variable for sensitivity analyses
analysis_df["log_n_utterances"] = np.log(
    analysis_df["n_utterances"]
)

analysis_df["log_n_utterances_c"] = (
    analysis_df["log_n_utterances"]
    - analysis_df["log_n_utterances"].mean()
)

analysis_df["level_label"] = pd.Categorical(
    analysis_df["level_label"],
    categories=[
        "expert",
        "graduate",
        "undergraduate",
        "teenager",
        "child",
    ],
    ordered=True,
)

In [ ]:
#standardize outcomes
def prepare_metric(df, metric):
    model_df = df[
        [
            metric,
            "gap_score",
            "gap_c",
            "level_label",
            "video_group",
            "n_utterances",
            "log_n_utterances_c",
        ]
    ].dropna().copy()

    model_df["outcome_z"] = (
        model_df[metric] - model_df[metric].mean()
    ) / model_df[metric].std(ddof=1)

    return model_df

-> betagap represents SD change in the metric per gap level

## model 0 (null model)

separates vaience into 1: between video variance 2: within video residual variance

In [ ]:
metric = "pairwise_distance_mean"
model_df = prepare_metric(analysis_df, metric)

model_0 = smf.mixedlm(
    "outcome_z ~ 1",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(model_0.summary())

In [ ]:
#intraclass correlation
between_video_variance = model_0.cov_re.iloc[0, 0]
residual_variance = model_0.scale

icc = (
    between_video_variance
    / (between_video_variance + residual_variance)
)

print("ICC:", icc)

icc: % unexplained variation that occurs between videos. 5 is good


## model 1 : linear expertisegap model

In [ ]:
model_1 = smf.mixedlm(
    "outcome_z ~ gap_c",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(model_1.summary())

In [ ]:
beta_gap = model_1.fe_params["gap_c"]
ci_gap = model_1.conf_int().loc["gap_c"]

print("Gap coefficient:", beta_gap)
print("95% CI:", tuple(ci_gap))
print("Predicted child–expert difference:", 4 * beta_gap)

## model 2: length adjusted model

In [ ]:
model_2 = smf.mixedlm(
    "outcome_z ~ gap_c + log_n_utterances_c",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(model_2.summary())

In [ ]:
#compare gap coefficients between length adjusted and not
comparison = pd.DataFrame({
    "model": [
        "Unadjusted",
        "Length-adjusted",
    ],
    "gap_beta": [
        model_1.fe_params["gap_c"],
        model_2.fe_params["gap_c"],
    ],
    "gap_p": [
        model_1.pvalues["gap_c"],
        model_2.pvalues["gap_c"],
    ],
})

display(comparison)

length probably doesnt have too big of an effect

## model 3: categorical expertise model

asks if each partner level differs from expert condition WITHOUT imposing linear progression

In [ ]:
model_3 = smf.mixedlm(
    (
        "outcome_z ~ "
        "C(level_label, Treatment(reference='expert'))"
    ),
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=True,
    method="lbfgs",
)

print(model_3.summary())

In [ ]:
from scipy.stats import chi2

model_linear_ml = smf.mixedlm(
    "outcome_z ~ gap_c",
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=False,
    method="lbfgs",
)

model_categorical_ml = smf.mixedlm(
    (
        "outcome_z ~ "
        "C(level_label, Treatment(reference='expert'))"
    ),
    data=model_df,
    groups=model_df["video_group"],
    re_formula="1",
).fit(
    reml=False,
    method="lbfgs",
)

lr_statistic = 2 * (
    model_categorical_ml.llf
    - model_linear_ml.llf
)

df_difference = (
    len(model_categorical_ml.fe_params)
    - len(model_linear_ml.fe_params)
)

lr_pvalue = chi2.sf(
    lr_statistic,
    df_difference,
)

print("Linear AIC:", model_linear_ml.aic)
print("Categorical AIC:", model_categorical_ml.aic)
print("Likelihood-ratio statistic:", lr_statistic)
print("Degrees of freedom:", df_difference)
print("p-value:", lr_pvalue)

AIC values are similar - USE LINEAR MODEL FOR PARSIMONY

# model all selected metrics

In [ ]:
# set of metrics to be modeled
metrics_to_model = [
    "pairwise_distance_mean",
    "pairwise_distance_sd",
    "radial_distance_q90",
    "nearest_neighbor_distance_mean",
    "knn_distance_median",
    "spectral_entropy_normalized",
    "participation_ratio",
    "pc1_variance_share",
]

linear random intercept model for each metric

In [ ]:
def fit_metric_model(
    df,
    metric,
    adjust_for_length=False,
    random_slope=False,
):
    model_df = prepare_metric(df, metric)

    formula = "outcome_z ~ gap_c"

    if adjust_for_length:
        formula += " + log_n_utterances_c"

    re_formula = (
        "~gap_c"
        if random_slope
        else "1"
    )

    result = smf.mixedlm(
        formula,
        data=model_df,
        groups=model_df["video_group"],
        re_formula=re_formula,
    ).fit(
        reml=True,
        method="lbfgs",
    )

    return result

In [ ]:
#run models. get gap effects
model_results = {}
summary_records = []

for metric in metrics_to_model:
    result = fit_metric_model(
        analysis_df,
        metric,
        adjust_for_length=False,
        random_slope=False,
    )

    model_results[metric] = result

    ci = result.conf_int().loc["gap_c"]

    summary_records.append({
        "metric": metric,
        "gap_beta_standardized":
            result.fe_params["gap_c"],
        "gap_se":
            result.bse["gap_c"],
        "ci_lower":
            ci.iloc[0],
        "ci_upper":
            ci.iloc[1],
        "p_value":
            result.pvalues["gap_c"],
        "converged":
            result.converged,
        "child_expert_difference_sd":
            4 * result.fe_params["gap_c"],
    })

model_summary = pd.DataFrame(summary_records)

display(model_summary)

In [ ]:
#correct for multiple testing
from statsmodels.stats.multitest import multipletests

model_summary["p_fdr_bh"] = multipletests(
    model_summary["p_value"],
    method="fdr_bh",
)[1]

model_summary["significant_fdr_05"] = (
    model_summary["p_fdr_bh"] < 0.05
)

model_summary = model_summary.sort_values(
    "p_fdr_bh"
).reset_index(drop=True)

display(model_summary)

# diagnostics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

result = model_results["pairwise_distance_mean"]

residuals = result.resid
fitted = result.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.scatterplot(
    x=fitted,
    y=residuals,
    ax=axes[0],
)

axes[0].axhline(
    0,
    color="black",
    linestyle="--",
)

axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals versus fitted")

stats.probplot(
    residuals,
    dist="norm",
    plot=axes[1],
)

axes[1].set_title("Residual Q–Q plot")

plt.tight_layout()
plt.show()

In [ ]:
model_summary

larger gap -> one dominant semantic direction(this makes sense), less even spectral variation,

## animation of embeddings

In [ ]:
TRAJECTORY_TYPE = "joint"
# "joint", "partner", or "expert"

In [ ]:
# ============================================================
# ANIMATED MATCHED SEMANTIC TRAJECTORIES — GIF ONLY
# Requires:
#   turns_refine
#   X_turns
#   LEVEL_COL
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from matplotlib.animation import (
    FuncAnimation,
    PillowWriter,
)
from matplotlib.lines import Line2D
from IPython.display import Image, display


# ============================================================
# 1. SETTINGS
# ============================================================

# View available video/topic identifiers with:
# print(turns_refine["video_uid"].unique())

CHOSEN_VIDEO = 'gravity'

# Any two matched partner conditions:
CONDITIONS = [
    "expert",
    "child",
]

# Choose:
#   "joint"   = all expert and partner turns
#   "partner" = partner turns only
#   "expert"  = explaining-expert turns only
TRAJECTORY_TYPE = "joint"

# GIF settings
N_FRAMES = 100
FPS = 12
DPI = 110

OUTPUT_GIF = (
    f"matched_{TRAJECTORY_TYPE}_"
    f"semantic_trajectories.gif"
)


# ============================================================
# 2. PROJECT ALL TURN EMBEDDINGS INTO ONE GLOBAL PCA SPACE
# ============================================================

pca = PCA(
    n_components=2,
    random_state=42,
)

X_turns_pca = pca.fit_transform(
    X_turns
)

animation_data = turns_refine.copy()

# turns_refine may have been sorted after X_turns was made.
# _embedding_row preserves the correct embedding alignment.
animation_data["pc1"] = X_turns_pca[
    animation_data["_embedding_row"],
    0,
]

animation_data["pc2"] = X_turns_pca[
    animation_data["_embedding_row"],
    1,
]

animation_data["level_key"] = (
    animation_data[LEVEL_COL]
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "teen": "teenager",
        "college": "undergraduate",
        "college student": "undergraduate",
        "graduate student": "graduate",
    })
)


# ============================================================
# 3. SELECT THE MATCHED TRAJECTORIES
# ============================================================

condition_data = {}

for condition in CONDITIONS:
    subset = animation_data[
        (
            animation_data[
                "video_uid"
            ] == CHOSEN_VIDEO
        )
        &
        (
            animation_data[
                "level_key"
            ] == condition
        )
    ].copy()

    if TRAJECTORY_TYPE != "joint":
        subset = subset[
            subset["_role"]
            == TRAJECTORY_TYPE
        ].copy()

    subset = (
        subset
        .sort_values("_turn_order")
        .reset_index(drop=True)
    )

    if len(subset) == 0:
        raise ValueError(
            f"No {TRAJECTORY_TYPE} turns found "
            f"for condition '{condition}' "
            f"in video '{CHOSEN_VIDEO}'."
        )

    condition_data[condition] = subset

    print(
        f"{condition}: "
        f"{len(subset)} "
        f"{TRAJECTORY_TYPE} turns"
    )


# ============================================================
# 4. CALCULATE SHARED AXIS LIMITS
# ============================================================

selected_points = pd.concat(
    condition_data.values(),
    ignore_index=True,
)

x_min = selected_points["pc1"].min()
x_max = selected_points["pc1"].max()

y_min = selected_points["pc2"].min()
y_max = selected_points["pc2"].max()

x_range = max(
    x_max - x_min,
    1e-6,
)

y_range = max(
    y_max - y_min,
    1e-6,
)

x_padding = 0.08 * x_range
y_padding = 0.08 * y_range

shared_xlim = (
    x_min - x_padding,
    x_max + x_padding,
)

shared_ylim = (
    y_min - y_padding,
    y_max + y_padding,
)


# ============================================================
# 5. VISUAL SETTINGS
# ============================================================

colorblind_palette = sns.color_palette(
    "colorblind"
)

role_colors = {
    "expert": colorblind_palette[0],
    "partner": colorblind_palette[1],
}

role_markers = {
    "expert": "o",
    "partner": "^",
}

role_labels = {
    "expert": "Explaining expert",
    "partner": "Partner role",
}

if TRAJECTORY_TYPE == "joint":
    visible_roles = [
        "expert",
        "partner",
    ]
else:
    visible_roles = [
        TRAJECTORY_TYPE,
    ]

frame_times = np.linspace(
    0,
    1,
    N_FRAMES,
)


# ============================================================
# 6. CREATE THE FIGURE
# ============================================================

fig, axes = plt.subplots(
    1,
    len(CONDITIONS),
    figsize=(12, 5.5),
    sharex=True,
    sharey=True,
)

if len(CONDITIONS) == 1:
    axes = [axes]

plot_artists = {}

for ax, condition in zip(
    axes,
    CONDITIONS,
):
    subset = condition_data[
        condition
    ]

    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)

    ax.set_xlabel("Global PCA 1")
    ax.set_ylabel("Global PCA 2")

    ax.set_title(
        f"{condition.title()} partner condition\n"
        f"{len(subset)} {TRAJECTORY_TYPE} turns"
    )

    # Accumulating trajectory line
    trajectory_line, = ax.plot(
        [],
        [],
        color="gray",
        linewidth=1.4,
        alpha=0.7,
        zorder=1,
    )

    # Separate point collections for each role
    role_scatters = {}

    for role in visible_roles:
        role_scatters[role] = ax.scatter(
            [],
            [],
            marker=role_markers[role],
            color=role_colors[role],
            s=42,
            alpha=0.8,
            zorder=2,
        )

    # First visible turn
    start_point = ax.scatter(
        [],
        [],
        marker="s",
        facecolors="none",
        edgecolors="black",
        linewidths=2,
        s=95,
        zorder=3,
    )

    # Current turn
    current_point = ax.scatter(
        [],
        [],
        marker="*",
        color="black",
        edgecolors="white",
        linewidths=0.7,
        s=170,
        zorder=4,
    )

    # Turn counter
    status_text = ax.text(
        0.02,
        0.98,
        "",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    plot_artists[condition] = {
        "line": trajectory_line,
        "role_scatters": role_scatters,
        "start": start_point,
        "current": current_point,
        "status": status_text,
    }


# ============================================================
# 7. LEGEND AND TITLES
# ============================================================

legend_handles = []

for role in visible_roles:
    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker=role_markers[role],
            linestyle="none",
            markerfacecolor=role_colors[role],
            markeredgecolor=role_colors[role],
            label=role_labels[role],
        )
    )

legend_handles.extend([
    Line2D(
        [0],
        [0],
        marker="*",
        linestyle="none",
        color="black",
        markersize=12,
        label="Current turn",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        linestyle="none",
        markerfacecolor="none",
        markeredgecolor="black",
        markersize=8,
        label="Start",
    ),
])

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=len(legend_handles),
    bbox_to_anchor=(0.5, 0.01),
)

progress_text = fig.text(
    0.5,
    0.94,
    "",
    ha="center",
    va="center",
    fontsize=12,
)

fig.suptitle(
    f"Matched {TRAJECTORY_TYPE} semantic trajectories: "
    f"{CHOSEN_VIDEO}",
    y=0.995,
)

plt.tight_layout(
    rect=[0, 0.09, 1, 0.91]
)


# ============================================================
# 8. ANIMATION UPDATE FUNCTION
# ============================================================

def empty_offsets():
    return np.empty(
        (0, 2)
    )


def update(frame_number):
    progress = frame_times[
        frame_number
    ]

    progress_text.set_text(
        f"Conversation progress: "
        f"{progress:.0%}"
    )

    updated_artists = [
        progress_text
    ]

    for condition in CONDITIONS:
        subset = condition_data[
            condition
        ]

        # Reveal points according to normalized
        # conversation time, not turn number.
        visible = subset[
            subset["_conversation_time"]
            <= progress + 1e-12
        ]

        condition_artists = (
            plot_artists[condition]
        )

        if len(visible) == 0:
            condition_artists[
                "line"
            ].set_data([], [])

            condition_artists[
                "start"
            ].set_offsets(
                empty_offsets()
            )

            condition_artists[
                "current"
            ].set_offsets(
                empty_offsets()
            )

            for scatter in (
                condition_artists[
                    "role_scatters"
                ].values()
            ):
                scatter.set_offsets(
                    empty_offsets()
                )

            condition_artists[
                "status"
            ].set_text(
                f"0 / {len(subset)} turns"
            )

            continue

        x = visible[
            "pc1"
        ].to_numpy()

        y = visible[
            "pc2"
        ].to_numpy()

        # Accumulated trajectory
        condition_artists[
            "line"
        ].set_data(x, y)

        # Start point
        condition_artists[
            "start"
        ].set_offsets(
            [[x[0], y[0]]]
        )

        # Most recently revealed point
        condition_artists[
            "current"
        ].set_offsets(
            [[x[-1], y[-1]]]
        )

        # Reveal speaker-specific markers
        for role, scatter in (
            condition_artists[
                "role_scatters"
            ].items()
        ):
            role_points = visible[
                visible["_role"] == role
            ]

            if len(role_points) == 0:
                scatter.set_offsets(
                    empty_offsets()
                )
            else:
                scatter.set_offsets(
                    role_points[
                        ["pc1", "pc2"]
                    ].to_numpy()
                )

        condition_artists[
            "status"
        ].set_text(
            f"{len(visible)} / "
            f"{len(subset)} turns"
        )

        updated_artists.extend([
            condition_artists["line"],
            condition_artists["start"],
            condition_artists["current"],
            condition_artists["status"],
            *condition_artists[
                "role_scatters"
            ].values(),
        ])

    return updated_artists


# ============================================================
# 9. CREATE, SAVE, AND DISPLAY THE GIF
# ============================================================

matched_anim = FuncAnimation(
    fig,
    update,
    frames=N_FRAMES,
    interval=1000 / FPS,
    blit=False,
    repeat=False,
)

print("Rendering GIF...")

matched_anim.save(
    OUTPUT_GIF,
    writer=PillowWriter(
        fps=FPS,
    ),
    dpi=DPI,
)

plt.close(fig)

print(
    f"Saved: {OUTPUT_GIF}"
)

display(
    Image(
        filename=OUTPUT_GIF,
    )
)

In [ ]:
from pathlib import Path
import json
import numpy as np

EXPORT_DIR = Path(
    "analysis_exports"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [ ]:
turn_export = turns_refine.copy()

if LEVEL_COL != "level_label":
    turn_export = turn_export.rename(
        columns={
            LEVEL_COL: "level_label"
        }
    )

required_turn_columns = [
    "video_uid",
    "conversation_uid",
    "conversation_id",
    "level_label",
    "_role",
    "_turn_order",
    "_conversation_time",
    "_embedding_row",
]

missing_columns = [
    column
    for column in required_turn_columns
    if column not in turn_export.columns
]

if missing_columns:
    raise ValueError(
        f"Missing turn columns: {missing_columns}"
    )

turn_export = turn_export[
    required_turn_columns
].copy()

turn_export.to_csv(
    EXPORT_DIR / "turns_dynamic.csv",
    index=False,
)

In [ ]:
np.save(
    EXPORT_DIR / "turn_embeddings.npy",
    X_turns,
)

In [ ]:
if "conversation_geometry" in globals():
    conversation_geometry.to_csv(
        EXPORT_DIR / "conversation_geometry.csv",
        index=False,
    )

if "trajectory_refinement_local" in globals():
    trajectory_refinement_local.to_csv(
        EXPORT_DIR
        / "trajectory_refinement_local.csv",
        index=False,
    )

if "refinement_stage_metrics" in globals():
    refinement_stage_metrics.to_csv(
        EXPORT_DIR
        / "refinement_stage_metrics.csv",
        index=False,
    )

In [ ]:
manifest = {
    "n_turns": int(len(turn_export)),
    "embedding_rows": int(X_turns.shape[0]),
    "embedding_dimensions": int(X_turns.shape[1]),
    "distance": "euclidean_chord_on_unit_embeddings",
    "turn_unit": "collapsed consecutive same-speaker utterances",
    "roles": sorted(
        turn_export["_role"]
        .dropna()
        .unique()
        .tolist()
    ),
}

with open(
    EXPORT_DIR / "dynamic_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )

print(
    "Exported to:",
    EXPORT_DIR.resolve(),
)

# Speaker–speaker centroid geometry (corrected)

This section measures the cosine distance between the explaining expert's mean utterance embedding and the conversation partner's mean utterance embedding in each conversation.

The earlier version returned only `NaN` because the notebook stores `speaker` as `"A"`/`"B"`, while the function searched for `"Speaker A"`/`"Speaker B"`. This corrected version uses the canonical `speaker_role` values already created by the notebook: `"expert"` and `"partner"`.

Run the notebook through the construction of `utterances`, `E_utterances`, and `conversation_geometry` before running this section.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import cosine

required_objects = [
    "utterances",
    "E_utterances",
    "conversation_geometry",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the earlier notebook cells first. Missing: "
        + ", ".join(missing_objects)
    )

embedding_matrix = np.asarray(E_utterances)

print("utterances shape:", utterances.shape)
print("embedding matrix shape:", embedding_matrix.shape)
print("speaker values:")
print(utterances["speaker"].value_counts(dropna=False))
print("speaker_role values:")
print(utterances["speaker_role"].value_counts(dropna=False))

In [ ]:
# Validate the explicit mapping between table rows and embedding rows.
embedding_indices = (
    utterances["embedding_idx"]
    .dropna()
    .astype(int)
    .to_numpy()
)

if len(embedding_indices) == 0:
    raise ValueError("utterances contains no embedding_idx values.")

if embedding_indices.min() < 0:
    raise ValueError("embedding_idx contains a negative value.")

if embedding_indices.max() >= embedding_matrix.shape[0]:
    raise ValueError(
        f"Maximum embedding_idx is {embedding_indices.max()}, "
        f"but E_utterances has {embedding_matrix.shape[0]} rows."
    )

if utterances["embedding_idx"].duplicated().any():
    raise ValueError("embedding_idx must be unique in utterances.")

if not np.isfinite(embedding_matrix).all():
    raise ValueError("E_utterances contains non-finite values.")

print("Embedding alignment checks passed.")

In [ ]:
# Use the same primary subset as the earlier conversation geometry.
# This handles both Boolean and string representations safely.
include_primary = (
    utterances["include_geometry_primary"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

geometry_utterances = utterances.loc[include_primary].copy()

# Normalize the already-canonical roles defensively.
geometry_utterances["_centroid_role"] = (
    geometry_utterances["speaker_role"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "learner": "partner",
        "speaker a": "expert",
        "a": "expert",
        "speaker b": "partner",
        "b": "partner",
    })
)

unexpected_roles = sorted(
    set(geometry_utterances["_centroid_role"].dropna())
    - {"expert", "partner"}
)

if unexpected_roles:
    raise ValueError(
        "Unrecognized speaker roles: "
        + repr(unexpected_roles)
    )

role_counts = (
    geometry_utterances
    .groupby(
        ["dataset", "conversation_id", "_centroid_role"],
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
)

missing_role_conversations = role_counts[
    (role_counts.get("expert", 0) == 0)
    | (role_counts.get("partner", 0) == 0)
]

display(role_counts.head())

if not missing_role_conversations.empty:
    display(missing_role_conversations)
    raise ValueError(
        "At least one primary-geometry conversation lacks an expert "
        "or partner. See the displayed table."
    )

print(
    f"Validated {len(role_counts)} conversations; "
    f"all contain both roles."
)

In [ ]:
def calculate_speaker_centroid_distance(
    conversation_utterances,
    embeddings,
    role_column="_centroid_role",
    embedding_index_column="embedding_idx",
):
    """Return expert–partner centroid distances for one conversation."""
    expert_indices = (
        conversation_utterances.loc[
            conversation_utterances[role_column].eq("expert"),
            embedding_index_column,
        ]
        .dropna()
        .astype(int)
        .to_numpy()
    )

    partner_indices = (
        conversation_utterances.loc[
            conversation_utterances[role_column].eq("partner"),
            embedding_index_column,
        ]
        .dropna()
        .astype(int)
        .to_numpy()
    )

    if len(expert_indices) == 0 or len(partner_indices) == 0:
        raise ValueError(
            "Each conversation must contain both expert and partner utterances."
        )

    expert_centroid = embeddings[expert_indices].mean(axis=0)
    partner_centroid = embeddings[partner_indices].mean(axis=0)

    return {
        "expert_n_utterances": int(len(expert_indices)),
        "partner_n_utterances": int(len(partner_indices)),
        "speaker_centroid_cosine_distance": float(
            cosine(expert_centroid, partner_centroid)
        ),
        "speaker_centroid_euclidean_distance": float(
            np.linalg.norm(expert_centroid - partner_centroid)
        ),
    }


# Test one conversation before applying the function to all conversations.
first_key, first_group = next(iter(
    geometry_utterances.groupby(
        ["dataset", "conversation_id"],
        observed=True,
        sort=False,
    )
))

print("Test conversation:", first_key)
print(
    calculate_speaker_centroid_distance(
        first_group,
        embedding_matrix,
    )
)

In [ ]:
speaker_centroid_records = []

for (dataset, conversation_id), group in (
    geometry_utterances.groupby(
        ["dataset", "conversation_id"],
        observed=True,
        sort=False,
    )
):
    result = calculate_speaker_centroid_distance(
        conversation_utterances=group,
        embeddings=embedding_matrix,
    )

    speaker_centroid_records.append({
        "dataset": dataset,
        "conversation_id": conversation_id,
        **result,
    })

speaker_centroid_geometry = pd.DataFrame(speaker_centroid_records)

if speaker_centroid_geometry.empty:
    raise ValueError("No conversation-level centroid results were created.")

distance_columns = [
    "speaker_centroid_cosine_distance",
    "speaker_centroid_euclidean_distance",
]

if speaker_centroid_geometry[distance_columns].isna().any().any():
    raise ValueError("Centroid calculation unexpectedly produced NaN.")

display(speaker_centroid_geometry.head())
display(speaker_centroid_geometry[distance_columns].describe())

In [ ]:
# Merge by dataset and conversation_id to avoid accidental cross-dataset matches.
centroid_output_columns = [
    "expert_n_utterances",
    "partner_n_utterances",
    "speaker_centroid_cosine_distance",
    "speaker_centroid_euclidean_distance",
]

conversation_geometry = conversation_geometry.drop(
    columns=[
        column for column in centroid_output_columns
        if column in conversation_geometry.columns
    ],
    errors="ignore",
)

conversation_geometry = conversation_geometry.merge(
    speaker_centroid_geometry,
    on=["dataset", "conversation_id"],
    how="left",
    validate="one_to_one",
)

missing_after_merge = conversation_geometry[
    "speaker_centroid_cosine_distance"
].isna()

print(
    f"Merged centroid metrics into {len(conversation_geometry)} conversations."
)
print("Missing after merge:", int(missing_after_merge.sum()))

if missing_after_merge.any():
    display(
        conversation_geometry.loc[
            missing_after_merge,
            ["dataset", "conversation_id", "level_label"],
        ]
    )
    raise ValueError(
        "Some conversation_geometry rows did not match centroid results."
    )

display(
    conversation_geometry[
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level_label",
            "expert_n_utterances",
            "partner_n_utterances",
            "speaker_centroid_cosine_distance",
        ]
    ].head()
)

## Visualizations

Higher cosine distance means the two speakers' average semantic positions are farther apart. The matched-lines plot is especially important because it shows whether the pattern repeats within topics rather than being driven only by differences among topics.

In [ ]:
level_lookup = (
    conversation_geometry[["level", "level_label"]]
    .dropna()
    .drop_duplicates()
    .sort_values("level")
)

level_order = level_lookup["level_label"].tolist()
level_values = level_lookup["level"].tolist()

plot_data = conversation_geometry.dropna(
    subset=["level_label", "speaker_centroid_cosine_distance"]
).copy()

if plot_data.empty:
    raise ValueError("No finite centroid distances are available to plot.")

palette = dict(zip(
    level_order,
    sns.color_palette("coolwarm", n_colors=len(level_order)),
))

display(
    plot_data.groupby("level_label", observed=True)[
        "speaker_centroid_cosine_distance"
    ].agg(["count", "mean", "std", "median"])
)

In [ ]:
# Distribution by expertise level, using Matplotlib for the box layer.
rng = np.random.default_rng(2026)
groups = [
    plot_data.loc[
        plot_data["level_label"].eq(level),
        "speaker_centroid_cosine_distance",
    ].to_numpy()
    for level in level_order
]

fig, ax = plt.subplots(figsize=(9, 6))

box = ax.boxplot(
    groups,
    labels=level_order,
    showfliers=False,
    patch_artist=True,
)

for patch, level in zip(box["boxes"], level_order):
    patch.set_facecolor(palette[level])
    patch.set_alpha(0.55)

for position, (level, values) in enumerate(
    zip(level_order, groups),
    start=1,
):
    x = position + rng.uniform(-0.12, 0.12, size=len(values))
    ax.scatter(x, values, color="black", alpha=0.45, s=22, zorder=3)

ax.set_xlabel("Conversation partner expertise")
ax.set_ylabel("Expert–partner centroid cosine distance")
ax.set_title("Semantic distance between speaker centroids")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Matched within-topic trajectories.
fig, ax = plt.subplots(figsize=(9, 6))

for (_, video_id), group in plot_data.groupby(
    ["dataset", "video_id"],
    observed=True,
):
    group = group.sort_values("level")
    ax.plot(
        group["level"],
        group["speaker_centroid_cosine_distance"],
        color="gray",
        alpha=0.3,
        linewidth=1,
    )

level_summary = (
    plot_data.groupby(["level", "level_label"], observed=True)
    ["speaker_centroid_cosine_distance"]
    .agg(["mean", "sem"])
    .reset_index()
    .sort_values("level")
)

ax.errorbar(
    level_summary["level"],
    level_summary["mean"],
    yerr=1.96 * level_summary["sem"],
    color="black",
    marker="o",
    linewidth=2.5,
    capsize=4,
    label="Mean ± 95% normal CI",
)

ax.set_xticks(level_values)
ax.set_xticklabels(level_order, rotation=20)
ax.set_xlabel("Conversation partner expertise")
ax.set_ylabel("Expert–partner centroid cosine distance")
ax.set_title("Within-topic change in speaker-centroid separation")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Topic-by-level heatmap.
centroid_heatmap = (
    plot_data.pivot_table(
        index=["dataset", "video_id"],
        columns="level_label",
        values="speaker_centroid_cosine_distance",
        aggfunc="first",
    )
    .reindex(columns=level_order)
)

fig, ax = plt.subplots(
    figsize=(9, max(5, 0.35 * len(centroid_heatmap)))
)

sns.heatmap(
    centroid_heatmap,
    cmap="viridis",
    linewidths=0.3,
    cbar_kws={"label": "Centroid cosine distance"},
    ax=ax,
)

ax.set_xlabel("Conversation partner expertise")
ax.set_ylabel("Topic")
ax.set_title("Speaker-centroid separation by topic and expertise")
plt.tight_layout()
plt.show()

In [ ]:
# Save reusable conversation-level results.
from pathlib import Path

OUTPUT_DIR = Path("analysis_exports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

speaker_centroid_geometry.to_csv(
    OUTPUT_DIR / "speaker_centroid_geometry.csv",
    index=False,
)

conversation_geometry.to_csv(
    OUTPUT_DIR / "conversation_geometry_with_speaker_centroids.csv",
    index=False,
)

print("Saved centroid results to:", OUTPUT_DIR.resolve())

# FEP-informed semantic state-space analysis

This section implements a deliberately low-parameter linear-Gaussian state-space model of the utterance-embedding trajectory. It estimates utterance-level predictive surprisal/minimized variational free energy, posterior-to-prior KL complexity, expected inaccuracy, and update magnitude.

The first implementation is exploratory and is fitted to the pooled corpus. It is appropriate for validating the model and inspecting its dynamics. Final confirmatory claims should use fold-specific PCA and held-out conversation or topic scoring, which is listed as the next methodological step at the end of this section.

By default, every utterance is retained because short acknowledgements and questions are dynamically meaningful. Set `FEP_INCLUDE_PRIMARY_ONLY = True` to reproduce the five-word geometry subset instead.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


FEP_N_COMPONENTS = 12
FEP_INCLUDE_PRIMARY_ONLY = False
FEP_INITIAL_VARIANCE = 1.0
FEP_MIN_VARIANCE = 1e-8
FEP_RANDOM_STATE = 2026

required_fep_objects = ["utterances", "E_utterances"]
missing_fep_objects = [
    name for name in required_fep_objects
    if name not in globals()
]

if missing_fep_objects:
    raise RuntimeError(
        "Run the earlier embedding cells first. Missing: "
        + ", ".join(missing_fep_objects)
    )

print("FEP analysis uses primary subset only:", FEP_INCLUDE_PRIMARY_ONLY)

In [ ]:
# Build a canonical chronological utterance table.
fep_utterances = utterances.copy()

if FEP_INCLUDE_PRIMARY_ONLY:
    fep_include_mask = (
        fep_utterances["include_geometry_primary"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )
    fep_utterances = fep_utterances.loc[fep_include_mask].copy()

fep_utterances["_fep_role"] = (
    fep_utterances["speaker_role"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "learner": "partner",
        "speaker a": "expert",
        "a": "expert",
        "speaker b": "partner",
        "b": "partner",
    })
)

unexpected_roles = sorted(
    set(fep_utterances["_fep_role"].dropna())
    - {"expert", "partner"}
)
if unexpected_roles:
    raise ValueError(f"Unexpected roles: {unexpected_roles}")

fep_utterances = (
    fep_utterances
    .sort_values(
        ["dataset", "conversation_id", "embedding_idx"],
        kind="stable",
    )
    .reset_index(drop=True)
)

embedding_indices = fep_utterances["embedding_idx"].astype(int).to_numpy()
embedding_matrix = np.asarray(E_utterances)

if embedding_indices.max() >= embedding_matrix.shape[0]:
    raise ValueError("embedding_idx does not align with E_utterances.")

X_fep_raw = embedding_matrix[embedding_indices]

print("FEP utterances:", len(fep_utterances))
print("Raw embedding shape:", X_fep_raw.shape)
print(fep_utterances["_fep_role"].value_counts())

In [ ]:
# Reduce and standardize the observation space.
# Standardization makes the fitted scalar process/noise variances comparable
# across retained PCA dimensions.
n_components = min(
    FEP_N_COMPONENTS,
    X_fep_raw.shape[0] - 1,
    X_fep_raw.shape[1],
)

fep_pca = PCA(
    n_components=n_components,
    random_state=FEP_RANDOM_STATE,
)
X_fep_pca = fep_pca.fit_transform(X_fep_raw)

fep_scaler = StandardScaler()
X_fep = fep_scaler.fit_transform(X_fep_pca)

fep_utterances["_fep_row"] = np.arange(len(fep_utterances))

print("Reduced observation shape:", X_fep.shape)
print(
    "Cumulative explained variance:",
    float(fep_pca.explained_variance_ratio_.sum()),
)
print(
    "Maximum standardized mean magnitude:",
    float(np.abs(X_fep.mean(axis=0)).max()),
)

In [ ]:
# Estimate a low-parameter role bias in the standardized PCA space.
# The dynamic latent state represents shared semantic context; this bias allows
# experts and partners to express that state with different average offsets.
fep_role_bias = {
    role: X_fep[
        fep_utterances["_fep_role"].eq(role).to_numpy()
    ].mean(axis=0)
    for role in ["expert", "partner"]
}


def build_fep_sequences(table, observations):
    sequences = []

    for (dataset, conversation_id), group in table.groupby(
        ["dataset", "conversation_id"],
        observed=True,
        sort=False,
    ):
        group = group.sort_values("embedding_idx", kind="stable").copy()
        rows = group["_fep_row"].to_numpy(dtype=int)

        if len(rows) < 2:
            continue

        sequences.append({
            "dataset": dataset,
            "conversation_id": conversation_id,
            "frame": group.reset_index(drop=True),
            "X": observations[rows],
            "roles": group["_fep_role"].to_numpy(dtype=object),
        })

    return sequences


fep_sequences = build_fep_sequences(fep_utterances, X_fep)

print("Sequences:", len(fep_sequences))
print(
    "Sequence-length range:",
    min(len(sequence["X"]) for sequence in fep_sequences),
    "to",
    max(len(sequence["X"]) for sequence in fep_sequences),
)

## Exact diagonal Kalman filter

After PCA standardization, the latent state follows a random walk:

$$z_t=z_{t-1}+\epsilon_t,\qquad \epsilon_t\sim\mathcal N(0,qI).$$

The utterance observation is:

$$x_t=z_t+a_{r_t}+\eta_t,\qquad \eta_t\sim\mathcal N(0,r_{r_t}I).$$

Only three positive parameters are estimated: process variance $q$, expert observation variance $r_E$, and partner observation variance $r_P$. The filter is exact under these assumptions, so minimized free energy and predictive surprisal should match numerically.

In [ ]:
def filter_fep_sequence(
    X,
    roles,
    role_bias,
    q,
    r_expert,
    r_partner,
    initial_variance=1.0,
    return_records=True,
):
    """Filter one standardized PCA utterance sequence."""
    X = np.asarray(X, dtype=float)
    roles = np.asarray(roles, dtype=object)
    n, d = X.shape

    if n < 2:
        return [] if return_records else 0.0

    q = max(float(q), FEP_MIN_VARIANCE)
    r_expert = max(float(r_expert), FEP_MIN_VARIANCE)
    r_partner = max(float(r_partner), FEP_MIN_VARIANCE)

    # Initialize the latent state from the first observation and do not score
    # that observation. This prevents a topic-agnostic initial prior from
    # dominating the beginning of every conversation.
    m = X[0] - role_bias[roles[0]]
    p = np.full(d, float(initial_variance))

    total_nll = 0.0
    records = []

    for position in range(1, n):
        role = roles[position]
        r = r_expert if role == "expert" else r_partner

        # Predict
        m_prior = m.copy()
        p_prior = p + q

        predicted_x = m_prior + role_bias[role]
        innovation = X[position] - predicted_x
        predictive_variance = p_prior + r

        mahalanobis = float(
            np.sum(innovation ** 2 / predictive_variance)
        )
        log_predictive_volume = float(
            np.sum(np.log(predictive_variance))
        )
        predictive_surprisal = 0.5 * (
            d * np.log(2.0 * np.pi)
            + log_predictive_volume
            + mahalanobis
        )

        # Exact posterior update
        gain = p_prior / predictive_variance
        m_post = m_prior + gain * innovation
        p_post = np.maximum(
            (1.0 - gain) * p_prior,
            FEP_MIN_VARIANCE,
        )

        # KL[q(z_t | x_<=t) || p(z_t | x_<t)]
        complexity_kl = 0.5 * float(np.sum(
            p_post / p_prior
            + (m_post - m_prior) ** 2 / p_prior
            - 1.0
            + np.log(p_prior / p_post)
        ))

        # E_q[-log p(x_t | z_t)]
        posterior_residual = (
            X[position]
            - (m_post + role_bias[role])
        )
        inaccuracy = 0.5 * float(np.sum(
            np.log(2.0 * np.pi * r)
            + (posterior_residual ** 2 + p_post) / r
        ))

        free_energy = complexity_kl + inaccuracy
        update_norm = float(np.linalg.norm(m_post - m_prior))

        total_nll += predictive_surprisal

        if return_records:
            records.append({
                "utterance_position": position,
                "predictive_surprisal": predictive_surprisal,
                "free_energy": free_energy,
                "complexity_kl": complexity_kl,
                "inaccuracy": inaccuracy,
                "mahalanobis_prediction_error": mahalanobis,
                "log_predictive_volume": log_predictive_volume,
                "latent_update_norm": update_norm,
                "prior_variance_mean": float(np.mean(p_prior)),
                "posterior_variance_mean": float(np.mean(p_post)),
            })

        m = m_post
        p = p_post

    return records if return_records else float(total_nll)


def unpack_fep_parameters(log_parameters):
    q, r_expert, r_partner = np.exp(np.asarray(log_parameters))
    return {
        "q": float(q),
        "r_expert": float(r_expert),
        "r_partner": float(r_partner),
    }


def pooled_fep_objective(log_parameters, sequences, role_bias):
    parameters = unpack_fep_parameters(log_parameters)
    total = 0.0

    for sequence in sequences:
        total += filter_fep_sequence(
            sequence["X"],
            sequence["roles"],
            role_bias,
            **parameters,
            initial_variance=FEP_INITIAL_VARIANCE,
            return_records=False,
        )

    return float(total)


def fit_fep_model(sequences, role_bias):
    initial_log_parameters = np.log([0.05, 0.20, 0.20])

    result = minimize(
        pooled_fep_objective,
        x0=initial_log_parameters,
        args=(sequences, role_bias),
        method="L-BFGS-B",
        bounds=[(-12.0, 3.0)] * 3,
        options={"maxiter": 300},
    )

    if not result.success:
        raise RuntimeError(
            "FEP parameter optimization failed: " + result.message
        )

    parameters = unpack_fep_parameters(result.x)
    parameters["total_negative_log_likelihood"] = float(result.fun)
    parameters["optimizer_iterations"] = int(result.nit)
    return parameters


fep_parameters = fit_fep_model(fep_sequences, fep_role_bias)
fep_parameters

In [ ]:
# Filter all conversations and attach utterance metadata.
fep_event_records = []

metadata_columns = [
    "dataset",
    "video_id",
    "conversation_id",
    "file_number",
    "level",
    "level_label",
    "utterance_id",
    "utterance_number",
    "turn_id",
    "sequence",
    "speaker",
    "speaker_role",
    "text",
    "n_words",
    "embedding_idx",
]

for sequence in fep_sequences:
    sequence_records = filter_fep_sequence(
        sequence["X"],
        sequence["roles"],
        fep_role_bias,
        q=fep_parameters["q"],
        r_expert=fep_parameters["r_expert"],
        r_partner=fep_parameters["r_partner"],
        initial_variance=FEP_INITIAL_VARIANCE,
        return_records=True,
    )

    frame = sequence["frame"]
    n_sequence = len(frame)

    for record in sequence_records:
        position = record["utterance_position"]
        source_row = frame.iloc[position]

        for column in metadata_columns:
            if column in frame.columns:
                record[column] = source_row[column]

        record["fep_role"] = source_row["_fep_role"]
        record["previous_role"] = frame.iloc[position - 1]["_fep_role"]
        record["normalized_time"] = position / max(n_sequence - 1, 1)
        fep_event_records.append(record)

fep_events = pd.DataFrame(fep_event_records)

identity_error = np.abs(
    fep_events["free_energy"]
    - fep_events["predictive_surprisal"]
)

print("FEP-scored utterances:", len(fep_events))
print("Maximum |F - surprisal|:", float(identity_error.max()))
print("Nonfinite values:", int((~np.isfinite(
    fep_events[
        [
            "predictive_surprisal",
            "free_energy",
            "complexity_kl",
            "inaccuracy",
        ]
    ].to_numpy()
)).sum()))

assert identity_error.max() < 1e-7
assert np.isfinite(
    fep_events[
        [
            "predictive_surprisal",
            "free_energy",
            "complexity_kl",
            "inaccuracy",
        ]
    ].to_numpy()
).all()

display(fep_events.head())
display(
    fep_events[
        [
            "predictive_surprisal",
            "complexity_kl",
            "inaccuracy",
            "latent_update_norm",
        ]
    ].describe()
)

## Conversation-level summaries

These summaries are descriptive. Because free energy equals predictive surprisal under exact filtering, the primary complementary outcomes are mean surprisal, KL complexity, inaccuracy, and update magnitude.

In [ ]:
def linear_slope(time, values):
    time = np.asarray(time, dtype=float)
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(time) & np.isfinite(values)
    time = time[valid]
    values = values[valid]

    if len(values) < 2 or np.ptp(time) <= 0:
        return np.nan

    return float(np.polyfit(time, values, deg=1)[0])


summary_records = []

for group_key, group in fep_events.groupby(
    ["dataset", "conversation_id"],
    observed=True,
    sort=False,
):
    dataset, conversation_id = group_key
    row = {
        "dataset": dataset,
        "conversation_id": conversation_id,
        "n_scored_utterances": len(group),
        "fep_surprisal_mean": group["predictive_surprisal"].mean(),
        "fep_surprisal_sd": group["predictive_surprisal"].std(ddof=1),
        "fep_surprisal_max": group["predictive_surprisal"].max(),
        "fep_surprisal_slope": linear_slope(
            group["normalized_time"],
            group["predictive_surprisal"],
        ),
        "fep_complexity_mean": group["complexity_kl"].mean(),
        "fep_complexity_sd": group["complexity_kl"].std(ddof=1),
        "fep_inaccuracy_mean": group["inaccuracy"].mean(),
        "fep_update_norm_mean": group["latent_update_norm"].mean(),
    }

    for column in ["video_id", "file_number", "level", "level_label"]:
        if column in group.columns:
            row[column] = group[column].iloc[0]

    summary_records.append(row)

fep_conversation_summary = pd.DataFrame(summary_records)

display(fep_conversation_summary.head())
display(
    fep_conversation_summary.groupby(
        "level_label",
        observed=True,
    )[
        [
            "fep_surprisal_mean",
            "fep_complexity_mean",
            "fep_inaccuracy_mean",
            "fep_update_norm_mean",
        ]
    ].mean()
)

In [ ]:
# Distribution of conversation-level FEP components by expertise.
level_lookup = (
    fep_conversation_summary[["level", "level_label"]]
    .drop_duplicates()
    .sort_values("level")
)
fep_level_order = level_lookup["level_label"].astype(str).tolist()
fep_palette = dict(zip(
    fep_level_order,
    sns.color_palette("coolwarm", n_colors=len(fep_level_order)),
))

plot_metrics = {
    "fep_surprisal_mean": "Mean predictive surprisal / F",
    "fep_complexity_mean": "Mean KL belief update",
    "fep_inaccuracy_mean": "Mean posterior inaccuracy",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
rng = np.random.default_rng(FEP_RANDOM_STATE)

for ax, (metric, title) in zip(axes, plot_metrics.items()):
    for position, level_label in enumerate(fep_level_order, start=1):
        values = fep_conversation_summary.loc[
            fep_conversation_summary["level_label"].astype(str).eq(level_label),
            metric,
        ].dropna().to_numpy()

        x = position + rng.uniform(-0.12, 0.12, size=len(values))
        ax.scatter(x, values, color=fep_palette[level_label], alpha=0.45, s=24)

        if len(values):
            mean = values.mean()
            sem = values.std(ddof=1) / np.sqrt(len(values)) if len(values) > 1 else 0
            ax.errorbar(
                position,
                mean,
                yerr=1.96 * sem,
                color="black",
                marker="D",
                capsize=4,
                linewidth=1.5,
                zorder=4,
            )

    ax.set_xticks(range(1, len(fep_level_order) + 1))
    ax.set_xticklabels(fep_level_order, rotation=30)
    ax.set_title(title)
    ax.set_xlabel("Partner expertise")
    ax.grid(axis="y", alpha=0.15)

plt.tight_layout()
plt.show()

In [ ]:
# Equal-conversation-weight trajectories over normalized time.
N_FEP_TIME_BINS = 20

fep_trajectory_data = fep_events.copy()
fep_trajectory_data["time_bin"] = pd.cut(
    fep_trajectory_data["normalized_time"],
    bins=np.linspace(0, 1, N_FEP_TIME_BINS + 1),
    labels=False,
    include_lowest=True,
)
fep_trajectory_data["time_bin_midpoint"] = (
    fep_trajectory_data["time_bin"] + 0.5
) / N_FEP_TIME_BINS

fep_conversation_bins = (
    fep_trajectory_data
    .groupby(
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level",
            "level_label",
            "time_bin",
            "time_bin_midpoint",
        ],
        observed=True,
        as_index=False,
    )[
        ["predictive_surprisal", "complexity_kl", "inaccuracy"]
    ]
    .mean()
)

fep_level_trajectory = (
    fep_conversation_bins
    .groupby(
        ["level", "level_label", "time_bin_midpoint"],
        observed=True,
    )["predictive_surprisal"]
    .agg(mean="mean", sem="sem", count="count")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))

for level_label in fep_level_order:
    subset = (
        fep_level_trajectory[
            fep_level_trajectory["level_label"].astype(str).eq(level_label)
        ]
        .sort_values("time_bin_midpoint")
    )

    color = fep_palette[level_label]
    lower = subset["mean"] - 1.96 * subset["sem"]
    upper = subset["mean"] + 1.96 * subset["sem"]

    ax.plot(
        subset["time_bin_midpoint"],
        subset["mean"],
        color=color,
        linewidth=2.2,
        label=level_label.title(),
    )
    ax.fill_between(
        subset["time_bin_midpoint"],
        lower,
        upper,
        color=color,
        alpha=0.12,
    )

ax.set_xlabel("Normalized conversation time")
ax.set_ylabel("Predictive surprisal / minimized free energy")
ax.set_title("Semantic predictive free energy over conversation time")
ax.set_xlim(0, 1)
ax.legend(title="Partner expertise", frameon=False)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()

## Expert perturbation and recovery

For each scored expert utterance with enough surrounding observations, this section estimates a pre-event baseline, perturbation amplitude, raw recovery, baseline-adjusted recovery fraction, and excess-free-energy area. Recovery fractions are retained only when the event rises above its pre-event baseline.

In [ ]:
FEP_RECOVERY_K = 3
fep_recovery_records = []

for group_key, group in fep_events.groupby(
    ["dataset", "conversation_id"],
    observed=True,
    sort=False,
):
    dataset, conversation_id = group_key
    group = group.sort_values("utterance_position").reset_index(drop=True)
    values = group["free_energy"].to_numpy(dtype=float)

    for index in range(FEP_RECOVERY_K, len(group) - FEP_RECOVERY_K):
        if group.loc[index, "fep_role"] != "expert":
            continue

        baseline = float(np.mean(values[index - FEP_RECOVERY_K:index]))
        event_value = float(values[index])
        post_mean = float(np.mean(values[index + 1:index + 1 + FEP_RECOVERY_K]))
        amplitude = event_value - baseline
        raw_recovery = event_value - post_mean
        residual_above_baseline = post_mean - baseline

        recovery_fraction = (
            raw_recovery / amplitude
            if amplitude > 1e-8
            else np.nan
        )

        post_values = values[index:index + FEP_RECOVERY_K + 1]
        excess_auc = float(np.maximum(post_values - baseline, 0.0).sum())

        row = {
            "dataset": dataset,
            "conversation_id": conversation_id,
            "utterance_position": group.loc[index, "utterance_position"],
            "embedding_idx": group.loc[index, "embedding_idx"],
            "pre_event_baseline": baseline,
            "event_free_energy": event_value,
            "perturbation_amplitude": amplitude,
            "post_event_mean": post_mean,
            "raw_recovery": raw_recovery,
            "residual_above_baseline": residual_above_baseline,
            "recovery_fraction": recovery_fraction,
            "excess_free_energy_auc": excess_auc,
        }

        for column in ["video_id", "level", "level_label", "turn_id", "text"]:
            if column in group.columns:
                row[column] = group.loc[index, column]

        fep_recovery_records.append(row)

fep_recovery_events = pd.DataFrame(fep_recovery_records)

display(fep_recovery_events.head())
display(
    fep_recovery_events[
        [
            "perturbation_amplitude",
            "raw_recovery",
            "recovery_fraction",
            "excess_free_energy_auc",
        ]
    ].describe()
)

## Optional link to the existing partner-uptake analysis

If `partner_adaptation_events` exists from Stage 3, the following cells aggregate utterance-level FEP quantities to turns and attach the expert turn's predictive quantities to the subsequent partner-uptake event. The quadratic model is exploratory; inspect the plotted support and estimated vertex before interpreting it as an optimal-grip relationship.

In [ ]:
fep_turn_metrics = (
    fep_events
    .groupby(
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level",
            "level_label",
            "turn_id",
            "fep_role",
        ],
        observed=True,
    )
    .agg(
        fep_turn_n_scored_utterances=("predictive_surprisal", "size"),
        fep_turn_surprisal_mean=("predictive_surprisal", "mean"),
        fep_turn_surprisal_max=("predictive_surprisal", "max"),
        fep_turn_complexity_mean=("complexity_kl", "mean"),
        fep_turn_complexity_sum=("complexity_kl", "sum"),
        fep_turn_inaccuracy_mean=("inaccuracy", "mean"),
    )
    .reset_index()
)

if "partner_adaptation_events" in globals():
    expert_fep_turns = (
        fep_turn_metrics[
            fep_turn_metrics["fep_role"].eq("expert")
        ]
        .copy()
    )

    fep_uptake_events = partner_adaptation_events.merge(
        expert_fep_turns,
        left_on=["dataset", "conversation_id", "expert_turn_order"],
        right_on=["dataset", "conversation_id", "turn_id"],
        how="left",
        validate="many_to_one",
        suffixes=("", "_fep"),
    )

    print(
        "Uptake events with expert-turn FEP:",
        int(fep_uptake_events["fep_turn_surprisal_mean"].notna().sum()),
        "of",
        len(fep_uptake_events),
    )
    display(fep_uptake_events.head())
else:
    fep_uptake_events = None
    print("partner_adaptation_events is not present; uptake merge skipped.")

In [ ]:
# Quadratic optimal-grip model and visualization.
if fep_uptake_events is not None:
    import statsmodels.formula.api as smf

    gap_score_map = {
        "expert": 0,
        "graduate": 1,
        "undergraduate": 2,
        "teenager": 3,
        "child": 4,
    }

    optimal_grip_data = (
        fep_uptake_events
        .dropna(
            subset=[
                "uptake_closure",
                "fep_turn_surprisal_mean",
                "level_label",
            ]
        )
        .copy()
    )

    optimal_grip_data["gap_score"] = (
        optimal_grip_data["level_label"]
        .astype(str)
        .str.lower()
        .map(gap_score_map)
    )
    optimal_grip_data["gap_c"] = optimal_grip_data["gap_score"] - 2.0

    surprise_mean = optimal_grip_data["fep_turn_surprisal_mean"].mean()
    surprise_sd = optimal_grip_data["fep_turn_surprisal_mean"].std(ddof=1)
    optimal_grip_data["surprisal_z"] = (
        optimal_grip_data["fep_turn_surprisal_mean"] - surprise_mean
    ) / surprise_sd
    optimal_grip_data["surprisal_z_sq"] = optimal_grip_data["surprisal_z"] ** 2
    optimal_grip_data["video_group"] = (
        optimal_grip_data["dataset"].astype(str)
        + "::"
        + optimal_grip_data["video_id"].astype(str)
    )
    optimal_grip_data["conversation_group"] = (
        optimal_grip_data["dataset"].astype(str)
        + "::"
        + optimal_grip_data["conversation_id"].astype(str)
    )

    optimal_grip_formula = (
        "uptake_closure ~ "
        "surprisal_z + surprisal_z_sq + gap_c + "
        "surprisal_z:gap_c + surprisal_z_sq:gap_c"
    )

    try:
        optimal_grip_model = smf.mixedlm(
            optimal_grip_formula,
            data=optimal_grip_data,
            groups=optimal_grip_data["video_group"],
            re_formula="1",
            vc_formula={
                "conversation": "0 + C(conversation_group)"
            },
        ).fit(
            reml=False,
            method="lbfgs",
        )
        optimal_grip_model_type = "mixed-effects model"
    except Exception as mixed_model_error:
        print(
            "Mixed model did not converge; using conversation-clustered OLS."
        )
        print("Mixed-model error:", mixed_model_error)
        optimal_grip_model = smf.ols(
            optimal_grip_formula,
            data=optimal_grip_data,
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": optimal_grip_data["conversation_group"]
            },
        )
        optimal_grip_model_type = "conversation-clustered OLS"

    print("Model used:", optimal_grip_model_type)
    print(optimal_grip_model.summary())

    beta_linear = optimal_grip_model.params.get("surprisal_z", np.nan)
    beta_quadratic = optimal_grip_model.params.get("surprisal_z_sq", np.nan)
    vertex_z = (
        -beta_linear / (2.0 * beta_quadratic)
        if np.isfinite(beta_quadratic) and beta_quadratic < 0
        else np.nan
    )
    print("Expertise-centered quadratic vertex (z units):", vertex_z)

    grid = np.linspace(
        optimal_grip_data["surprisal_z"].quantile(0.02),
        optimal_grip_data["surprisal_z"].quantile(0.98),
        200,
    )
    prediction_frame = pd.DataFrame({
        "surprisal_z": grid,
        "surprisal_z_sq": grid ** 2,
        "gap_c": 0.0,
    })
    predicted_uptake = optimal_grip_model.predict(prediction_frame)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(
        optimal_grip_data["surprisal_z"],
        optimal_grip_data["uptake_closure"],
        alpha=0.25,
        s=24,
        color="gray",
    )
    ax.plot(grid, predicted_uptake, color="black", linewidth=2.5)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    if np.isfinite(vertex_z):
        ax.axvline(vertex_z, color="gray", linestyle=":", linewidth=1.5)
    ax.set_xlabel("Expert-turn predictive surprisal (standardized)")
    ax.set_ylabel("Partner uptake closure")
    ax.set_title("Exploratory optimal-grip relationship")
    ax.grid(alpha=0.15)
    plt.tight_layout()
    plt.show()

In [ ]:
# Export reusable outputs.
from pathlib import Path

FEP_OUTPUT_DIR = Path("analysis_exports")
FEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fep_events.to_csv(
    FEP_OUTPUT_DIR / "fep_utterance_events.csv",
    index=False,
)
fep_conversation_summary.to_csv(
    FEP_OUTPUT_DIR / "fep_conversation_summary.csv",
    index=False,
)
fep_turn_metrics.to_csv(
    FEP_OUTPUT_DIR / "fep_turn_metrics.csv",
    index=False,
)
fep_recovery_events.to_csv(
    FEP_OUTPUT_DIR / "fep_recovery_events.csv",
    index=False,
)

print("Saved FEP outputs to:", FEP_OUTPUT_DIR.resolve())

## Before confirmatory inference

This pooled implementation is the correct first mechanism check, but its scores are in-sample. Before treating expertise differences as confirmatory evidence:

1. Fit PCA, scaling, role biases, and noise parameters only on training folds.
2. Score held-out conversations.
3. Repeat with leave-one-topic-out folds.
4. Compare against a static role/topic Gaussian baseline with no temporal state.
5. Repeat with 8, 12, and 20 PCA dimensions.
6. Inspect whether diagonal Gaussian residuals are adequate; use shrinkage or Student-t observations if needed.

The next implementation step should therefore be fold-safe model comparison, after the pooled results and diagnostics have been inspected.